# Soportes: modelo por planilla Excel

Notebook nuevo (no modifica el ejemplo `agente_estructural.ipynb`). El modelo se define en una planilla, como en RAM Element: no hay que dibujar.

## Flujo

1. **Bloque 6**: carga la plantilla `plantilla_soportes.xlsx` (incrustada) y muestra sus hojas. Descárguela desde la pestaña Output.
2. Llénela en Excel:
   - **NODOS**: ID, X, Y, Z (m) y APOYO (`empotrado`, `rotulado` o vacío = libre).
   - **BARRAS**: ID, NODO_I, NODO_J, ELEMENTO, PERFIL (lista Gerdau), EJE_FUERTE_X/Y/Z (vector libre del peralte; vacío = por defecto), CONEXION_I y CONEXION_J (`empotrado` o `rotulado` en cada extremo).
   - **VISTA_VIVA**: planta y alzado que cambian al editar NODOS o BARRAS. Puede elegir el nivel Z (C3) y la cota Y (C4).
   - **CONEXIONES**, **LISTAS**, **CALCULO**: no editar (CALCULO sostiene los gráficos).
3. Súbala a Kaggle con **Add data**. El **bloque 7** la lee y valida.
4. **Bloque 8**: dibuja planta, alzado y 3D (secciones y ejes fuertes) en el notebook y regenera la planilla como `planilla_con_vista.xlsx`, con las vistas en vivo de VISTA_VIVA. Vuelva a ejecutar este bloque después de cambiar la planilla.

**Lo que se actualiza solo en Excel:** la vista en planta y alzado (hoja VISTA_VIVA).
**Lo que requiere ejecutar el bloque 8:** la vista 3D con secciones y ejes fuertes (Excel no dibuja en 3D por sí solo).

**Reglas:** coordenadas en metros; un nodo no puede repetir coordenadas; una barra no puede unir un nodo consigo mismo; el eje fuerte, si se da, no puede ser paralelo a la barra. Basta una componente: las vacias valen 0.

## Índice de bloques

1. Motor de dibujo (dibujo_directo.py)
2. Datos Gerdau rev04 (gerdau_datos.py)
3. Base de perfiles y lista ordenada (perfiles_gerdau.py)
4. Orientación y secciones (orientacion_gerdau.py)
5. Uso (sin código)
6. Cargar plantilla Excel
7. Leer, validar y armar el modelo
8. Dibujo de revisión y planilla regenerada


## Bloque 1. Motor de dibujo (dibujo_directo.py)

In [ ]:
%%writefile dibujo_directo.py
"""
MODULO DE DIBUJO DIRECTO (reemplaza la lectura de DXF).

Permite construir el modelo estructural dentro del notebook: nodos (puntos x, y, z en m),
barras (polilineas entre nodos), apoyos y ejes fuertes. Produce EXACTAMENTE el mismo
diccionario que entrega lector_validador.leer_dxf(), para que el motor de analisis
(motor_analisis.construir_y_resolver) y el validador funcionen sin cambios.

Formato de salida (igual al DXF):
  {'NODOS': [(x,y,z) en m], 'BARRAS': [{'n1','n2','elemento','perfil','capa','eje_fuerte'}],
   'NODOS_APOYO': [indices], 'ruta': str}

Unidades: metros (la conversion a cm la hace el motor).
"""
import json
import numpy as np

TOL_NODO = 1e-3                 # m: nodos a menos de esta distancia son el mismo nodo
ELEMENTOS = ['COLUMNA', 'VIGA', 'ARRIOSTRE', 'DIAGONAL']
TIPOS_APOYO = ['empotrado', 'rotulado']   # nodo: empotrado = 6 GDL fijos; rotulado = 3 traslaciones fijas, giros libres
TIPOS_CONEXION = ['empotrado', 'rotulado']  # extremo de barra: empotrado = momento transmitido; rotulado = momento liberado


class DibujoModelo:
    def __init__(self, tol=TOL_NODO, ruta='dibujo_directo'):
        self.nodos = []         # lista de tuplas (x, y, z)
        self.barras = []        # lista de dicts con el mismo formato que el DXF
        self.apoyos = {}        # {indice_nodo: tipo}
        self.tol = tol
        self.ruta = ruta

    # ---------- nodos ----------
    def nodo(self, x, y, z):
        """Agrega un nodo (o devuelve el existente a menos de la tolerancia). Devuelve su indice."""
        p = (round(float(x), 4), round(float(y), 4), round(float(z), 4))
        for i, q in enumerate(self.nodos):
            if all(abs(p[k] - q[k]) < self.tol for k in range(3)):
                return i
        self.nodos.append(p)
        return len(self.nodos) - 1

    # ---------- barras ----------
    def _existe(self, n1, n2):
        return any({b['n1'], b['n2']} == {n1, n2} for b in self.barras)

    def barra(self, n1, n2, elemento, perfil, conexion_i='empotrado', conexion_j='empotrado', material=None):
        """Agrega una barra entre dos nodos (indices). Si ya existe, no la duplica. Devuelve su indice o None.
        conexion_i / conexion_j: 'empotrado' o 'rotulado' en cada extremo de la barra."""
        if n1 == n2:
            raise ValueError(f'barra degenerada en el nodo {n1}')
        if self._existe(n1, n2):
            return None
        elemento = str(elemento).upper()
        if elemento not in ELEMENTOS:
            raise ValueError(f'elemento {elemento!r} no reconocido; usar {ELEMENTOS}')
        for nombre, c in [('conexion_i', conexion_i), ('conexion_j', conexion_j)]:
            if c not in TIPOS_CONEXION:
                raise ValueError(f'{nombre} {c!r} no reconocida; usar {TIPOS_CONEXION}')
        self.barras.append({'n1': n1, 'n2': n2, 'elemento': elemento, 'perfil': perfil,
                            'capa': f'{elemento}-{perfil}', 'eje_fuerte': None,
                            'conexion_i': conexion_i, 'conexion_j': conexion_j, 'material': material})
        return len(self.barras) - 1

    def polilinea(self, puntos, elemento, perfil):
        """Une una secuencia de puntos con barras consecutivas (estilo 'dibujar con lineas').
        puntos: lista de indices de nodo o de coordenadas (x, y, z). Devuelve indices de barras creadas."""
        ids = [p if isinstance(p, (int, np.integer)) else self.nodo(*p) for p in puntos]
        creadas = []
        for a, b in zip(ids[:-1], ids[1:]):
            r = self.barra(a, b, elemento, perfil)
            if r is not None:
                creadas.append(r)
        return creadas

    def eje_fuerte(self, indice_barra, direccion):
        """Fija la direccion del eje fuerte de una barra (vector en coordenadas globales)."""
        d = np.array(direccion, float)
        n = np.linalg.norm(d)
        if n < 1e-9:
            raise ValueError('direccion nula')
        self.barras[indice_barra]['eje_fuerte'] = tuple(float(v) for v in d / n)

    # ---------- apoyos ----------
    def apoyo(self, nodo_idx, tipo='empotrado'):
        if tipo not in TIPOS_APOYO:
            raise ValueError(f'tipo de apoyo {tipo!r} no soportado por el motor (usar {TIPOS_APOYO})')
        self.apoyos[int(nodo_idx)] = tipo

    # ---------- generadores ----------
    def marco_grilla(self, xs, ys, zs, perfil_columna, perfil_viga_x, perfil_viga_y, eje_fuerte_columna=None):
        """Marco ortogonal a partir de ejes: nodos en todas las intersecciones (x, y, z).
        Columnas en cada (x, y) entre niveles consecutivos; vigas X y vigas Y en cada nivel superior.
        eje_fuerte_columna: direccion del eje fuerte para las columnas (opcional, ej. (1,0,0))."""
        xs, ys, zs = sorted(xs), sorted(ys), sorted(zs)
        idx = {}
        for x in xs:
            for y in ys:
                for z in zs:
                    idx[(x, y, z)] = self.nodo(x, y, z)
        for x in xs:
            for y in ys:
                for za, zb in zip(zs[:-1], zs[1:]):
                    i = self.barra(idx[(x, y, za)], idx[(x, y, zb)], 'COLUMNA', perfil_columna)
                    if i is not None and eje_fuerte_columna is not None:
                        self.eje_fuerte(i, eje_fuerte_columna)
        for y in ys:
            for z in zs[1:]:
                for xa, xb in zip(xs[:-1], xs[1:]):
                    self.barra(idx[(xa, y, z)], idx[(xb, y, z)], 'VIGA', perfil_viga_x)
        for x in xs:
            for z in zs[1:]:
                for ya, yb in zip(ys[:-1], ys[1:]):
                    self.barra(idx[(x, ya, z)], idx[(x, yb, z)], 'VIGA', perfil_viga_y)
        return idx

    # ---------- salida y persistencia ----------
    def a_modelo(self):
        """Diccionario con el formato de leer_dxf(): listo para motor_analisis y validar_modelo."""
        return {'NODOS': list(self.nodos), 'BARRAS': [dict(b) for b in self.barras],
                'NODOS_APOYO': sorted(self.apoyos.keys()), 'TIPO_APOYO': dict(self.apoyos), 'ruta': self.ruta}

    def guardar_json(self, ruta):
        datos = {'nodos': self.nodos, 'barras': self.barras,
                 'apoyos': {str(k): v for k, v in self.apoyos.items()}}
        with open(ruta, 'w', encoding='utf-8') as f:
            json.dump(datos, f, ensure_ascii=False, indent=1)

    @classmethod
    def cargar_json(cls, ruta):
        with open(ruta, encoding='utf-8') as f:
            datos = json.load(f)
        m = cls()
        m.nodos = [tuple(p) for p in datos['nodos']]
        m.barras = datos['barras']
        m.apoyos = {int(k): v for k, v in datos['apoyos'].items()}
        return m


def comparar_modelos(m1, m2, tol=1e-3):
    """Compara dos modelos (formato leer_dxf) sin depender del orden de nodos ni barras.
    Devuelve (ok, lista_de_diferencias)."""
    def claves_nodos(m):
        return {tuple(round(c, 3) for c in p) for p in m['NODOS']}

    def claves_barras(m):
        out = []
        for b in m['BARRAS']:
            p1 = tuple(round(c, 3) for c in m['NODOS'][b['n1']])
            p2 = tuple(round(c, 3) for c in m['NODOS'][b['n2']])
            ext = tuple(sorted([p1, p2]))
            ef = None if b['eje_fuerte'] is None else tuple(round(c, 3) for c in b['eje_fuerte'])
            out.append((ext, b['elemento'], b['perfil'], ef))
        return sorted(out, key=str)

    def claves_apoyos(m):
        return sorted(tuple(round(c, 3) for c in m['NODOS'][i]) for i in m['NODOS_APOYO'])

    dif = []
    if claves_nodos(m1) != claves_nodos(m2):
        dif.append(f"nodos distintos: {len(claves_nodos(m1))} vs {len(claves_nodos(m2))}")
    b1, b2 = claves_barras(m1), claves_barras(m2)
    if b1 != b2:
        s1, s2 = set(map(str, b1)), set(map(str, b2))
        dif.append(f"barras distintas: {len(b1)} vs {len(b2)} | solo en 1: {len(s1 - s2)} | solo en 2: {len(s2 - s1)}")
    if claves_apoyos(m1) != claves_apoyos(m2):
        dif.append("apoyos distintos")
    return (len(dif) == 0), dif


## Bloque 2. Datos Gerdau rev04 (gerdau_datos.py)

Los 247 perfiles en un solo texto. No hay que editar este bloque.

In [ ]:
%%writefile gerdau_datos.py
_DATOS_TEXTO = "[{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 8.5\",\"estado\":\"Revisado\",\"masa_kg_m\":12.7,\"d_ala_diametro_cm\":14.8,\"bf_b_cm\":10,\"tw_cm\":0.43,\"tf_t_cm\":0.49000000000000005,\"area_cm2\":16.3,\"ix_cm4\":620,\"iy_cm4\":82,\"sx_cm3\":83.78378378378378,\"sy_cm3\":16.4,\"rx_cm\":6.2,\"ry_cm\":2.2,\"j_cm4\":1.4,\"cw_cm6\":4222.3},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 9\",\"estado\":\"Revisado\",\"masa_kg_m\":13.6,\"d_ala_diametro_cm\":15,\"bf_b_cm\":10,\"tw_cm\":0.43,\"tf_t_cm\":0.55,\"area_cm2\":17.3,\"ix_cm4\":683,\"iy_cm4\":91,\"sx_cm3\":91.06666666666666,\"sy_cm3\":18.2,\"rx_cm\":6.3,\"ry_cm\":2.3,\"j_cm4\":1.7,\"cw_cm6\":4750.26},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 12\",\"estado\":\"Con restricciones\",\"masa_kg_m\":18,\"d_ala_diametro_cm\":15.3,\"bf_b_cm\":10.2,\"tw_cm\":0.58,\"tf_t_cm\":0.71,\"area_cm2\":22.9,\"ix_cm4\":920,\"iy_cm4\":124,\"sx_cm3\":120.26143790849673,\"sy_cm3\":24.31372549019608,\"rx_cm\":6.3,\"ry_cm\":2.3,\"j_cm4\":3.7,\"cw_cm6\":6598.91},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 16\",\"estado\":\"Revisado\",\"masa_kg_m\":24,\"d_ala_diametro_cm\":16,\"bf_b_cm\":10.2,\"tw_cm\":0.6599999999999999,\"tf_t_cm\":1.03,\"area_cm2\":30.6,\"ix_cm4\":1336,\"iy_cm4\":184,\"sx_cm3\":167,\"sy_cm3\":36.07843137254902,\"rx_cm\":6.6,\"ry_cm\":2.5,\"j_cm4\":9.2,\"cw_cm6\":10308.64},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 15\",\"estado\":\"Revisado\",\"masa_kg_m\":22.4,\"d_ala_diametro_cm\":15.2,\"bf_b_cm\":15.2,\"tw_cm\":0.58,\"tf_t_cm\":0.6599999999999999,\"area_cm2\":28.6,\"ix_cm4\":1211,\"iy_cm4\":388,\"sx_cm3\":159.3421052631579,\"sy_cm3\":51.05263157894737,\"rx_cm\":6.5,\"ry_cm\":3.7,\"j_cm4\":4.2,\"cw_cm6\":20506.93},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 20\",\"estado\":\"Revisado\",\"masa_kg_m\":29.7,\"d_ala_diametro_cm\":15.7,\"bf_b_cm\":15.3,\"tw_cm\":0.6599999999999999,\"tf_t_cm\":0.93,\"area_cm2\":37.9,\"ix_cm4\":1723,\"iy_cm4\":554,\"sx_cm3\":219.4904458598726,\"sy_cm3\":72.41830065359477,\"rx_cm\":6.8,\"ry_cm\":3.8,\"j_cm4\":9.9,\"cw_cm6\":30214.18},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 6 \u00d7 25\",\"estado\":\"Revisado\",\"masa_kg_m\":37.2,\"d_ala_diametro_cm\":16.2,\"bf_b_cm\":15.4,\"tw_cm\":0.8099999999999999,\"tf_t_cm\":1.16,\"area_cm2\":47.4,\"ix_cm4\":2223,\"iy_cm4\":712,\"sx_cm3\":274.44444444444446,\"sy_cm3\":92.46753246753246,\"rx_cm\":6.9,\"ry_cm\":3.9,\"j_cm4\":19.1,\"cw_cm6\":40263.88},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 10\",\"estado\":\"Revisado\",\"masa_kg_m\":15,\"d_ala_diametro_cm\":20,\"bf_b_cm\":10,\"tw_cm\":0.43,\"tf_t_cm\":0.52,\"area_cm2\":19.1,\"ix_cm4\":1282,\"iy_cm4\":87,\"sx_cm3\":128.2,\"sy_cm3\":17.4,\"rx_cm\":8.2,\"ry_cm\":2.1,\"j_cm4\":1.7,\"cw_cm6\":8253.48},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 13\",\"estado\":\"Revisado\",\"masa_kg_m\":19.4,\"d_ala_diametro_cm\":20.3,\"bf_b_cm\":10.2,\"tw_cm\":0.58,\"tf_t_cm\":0.65,\"area_cm2\":24.8,\"ix_cm4\":1648,\"iy_cm4\":114,\"sx_cm3\":162.36453201970443,\"sy_cm3\":22.35294117647059,\"rx_cm\":8.2,\"ry_cm\":2.1,\"j_cm4\":3.7,\"cw_cm6\":11004.49},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 15\",\"estado\":\"Revisado\",\"masa_kg_m\":22.5,\"d_ala_diametro_cm\":20.6,\"bf_b_cm\":10.2,\"tw_cm\":0.62,\"tf_t_cm\":0.8,\"area_cm2\":28.6,\"ix_cm4\":1998,\"iy_cm4\":142,\"sx_cm3\":193.98058252427182,\"sy_cm3\":27.84313725490196,\"rx_cm\":8.4,\"ry_cm\":2.2,\"j_cm4\":5.8,\"cw_cm6\":13917.42},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 18\",\"estado\":\"Revisado\",\"masa_kg_m\":26.6,\"d_ala_diametro_cm\":20.7,\"bf_b_cm\":13.3,\"tw_cm\":0.58,\"tf_t_cm\":0.8400000000000001,\"area_cm2\":33.9,\"ix_cm4\":2576,\"iy_cm4\":332,\"sx_cm3\":248.88888888888889,\"sy_cm3\":49.92481203007519,\"rx_cm\":8.7,\"ry_cm\":3.1,\"j_cm4\":7.1,\"cw_cm6\":32736.83},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 21\",\"estado\":\"Revisado\",\"masa_kg_m\":31.2,\"d_ala_diametro_cm\":21,\"bf_b_cm\":13.4,\"tw_cm\":0.64,\"tf_t_cm\":1.02,\"area_cm2\":39.7,\"ix_cm4\":3134,\"iy_cm4\":407,\"sx_cm3\":298.4761904761905,\"sy_cm3\":60.74626865671642,\"rx_cm\":8.9,\"ry_cm\":3.2,\"j_cm4\":11.7,\"cw_cm6\":40618.64},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 24\",\"estado\":\"Revisado\",\"masa_kg_m\":35.9,\"d_ala_diametro_cm\":20.1,\"bf_b_cm\":16.5,\"tw_cm\":0.62,\"tf_t_cm\":1.02,\"area_cm2\":45.7,\"ix_cm4\":3446,\"iy_cm4\":762,\"sx_cm3\":342.88557213930346,\"sy_cm3\":92.36363636363636,\"rx_cm\":8.7,\"ry_cm\":4.1,\"j_cm4\":14.6,\"cw_cm6\":69350.84},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 28\",\"estado\":\"Revisado\",\"masa_kg_m\":41.8,\"d_ala_diametro_cm\":20.5,\"bf_b_cm\":16.6,\"tw_cm\":0.72,\"tf_t_cm\":1.1800000000000002,\"area_cm2\":53.2,\"ix_cm4\":4079,\"iy_cm4\":903,\"sx_cm3\":397.9512195121951,\"sy_cm3\":108.79518072289156,\"rx_cm\":8.8,\"ry_cm\":4.1,\"j_cm4\":22.5,\"cw_cm6\":84263.99},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 31\",\"estado\":\"Revisado\",\"masa_kg_m\":46.2,\"d_ala_diametro_cm\":20.3,\"bf_b_cm\":20.3,\"tw_cm\":0.72,\"tf_t_cm\":1.1,\"area_cm2\":58.9,\"ix_cm4\":4579,\"iy_cm4\":1544,\"sx_cm3\":451.13300492610836,\"sy_cm3\":152.11822660098522,\"rx_cm\":8.8,\"ry_cm\":5.1,\"j_cm4\":22.5,\"cw_cm6\":142295.04},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 35\",\"estado\":\"Revisado\",\"masa_kg_m\":52.2,\"d_ala_diametro_cm\":20.6,\"bf_b_cm\":20.4,\"tw_cm\":0.79,\"tf_t_cm\":1.26,\"area_cm2\":66.5,\"ix_cm4\":5286,\"iy_cm4\":1773,\"sx_cm3\":513.2038834951456,\"sy_cm3\":173.82352941176472,\"rx_cm\":8.9,\"ry_cm\":5.2,\"j_cm4\":32,\"cw_cm6\":165791.28},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 40\",\"estado\":\"Revisado\",\"masa_kg_m\":59.3,\"d_ala_diametro_cm\":21,\"bf_b_cm\":20.5,\"tw_cm\":0.9099999999999999,\"tf_t_cm\":1.42,\"area_cm2\":75.5,\"ix_cm4\":6077,\"iy_cm4\":2044,\"sx_cm3\":578.7619047619048,\"sy_cm3\":199.41463414634146,\"rx_cm\":9,\"ry_cm\":5.2,\"j_cm4\":46.6,\"cw_cm6\":195905.34},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 48\",\"estado\":\"Revisado\",\"masa_kg_m\":71.4,\"d_ala_diametro_cm\":21.6,\"bf_b_cm\":20.6,\"tw_cm\":1.02,\"tf_t_cm\":1.7399999999999998,\"area_cm2\":91,\"ix_cm4\":7659,\"iy_cm4\":2535,\"sx_cm3\":709.1666666666666,\"sy_cm3\":246.1165048543689,\"rx_cm\":9.2,\"ry_cm\":5.3,\"j_cm4\":81.6,\"cw_cm6\":249963.42},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 58\",\"estado\":\"Revisado\",\"masa_kg_m\":86.6,\"d_ala_diametro_cm\":22.2,\"bf_b_cm\":20.9,\"tw_cm\":1.3,\"tf_t_cm\":2.06,\"area_cm2\":110.3,\"ix_cm4\":9490,\"iy_cm4\":3126,\"sx_cm3\":854.9549549549549,\"sy_cm3\":299.1387559808613,\"rx_cm\":9.3,\"ry_cm\":5.3,\"j_cm4\":139,\"cw_cm6\":316991.72},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 8 \u00d7 67\",\"estado\":\"Revisado\",\"masa_kg_m\":99.8,\"d_ala_diametro_cm\":22.9,\"bf_b_cm\":21,\"tw_cm\":1.45,\"tf_t_cm\":2.37,\"area_cm2\":127.1,\"ix_cm4\":11321,\"iy_cm4\":3688,\"sx_cm3\":988.7336244541485,\"sy_cm3\":351.23809523809524,\"rx_cm\":9.4,\"ry_cm\":5.4,\"j_cm4\":211,\"cw_cm6\":388605.4},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 12\",\"estado\":\"Con restricciones\",\"masa_kg_m\":17.9,\"d_ala_diametro_cm\":25.1,\"bf_b_cm\":10.1,\"tw_cm\":0.48,\"tf_t_cm\":0.53,\"area_cm2\":22.8,\"ix_cm4\":2239,\"iy_cm4\":91,\"sx_cm3\":178.40637450199202,\"sy_cm3\":18.01980198019802,\"rx_cm\":9.9,\"ry_cm\":2,\"j_cm4\":null,\"cw_cm6\":13733.83},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 15\",\"estado\":\"Con restricciones\",\"masa_kg_m\":22.3,\"d_ala_diametro_cm\":25.4,\"bf_b_cm\":10.2,\"tw_cm\":0.58,\"tf_t_cm\":0.6900000000000001,\"area_cm2\":28.5,\"ix_cm4\":null,\"iy_cm4\":120,\"sx_cm3\":null,\"sy_cm3\":23.529411764705884,\"rx_cm\":null,\"ry_cm\":2.1,\"j_cm4\":4.2,\"cw_cm6\":18317.52},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 17\",\"estado\":\"Revisado\",\"masa_kg_m\":25.3,\"d_ala_diametro_cm\":25.7,\"bf_b_cm\":10.2,\"tw_cm\":0.61,\"tf_t_cm\":0.8400000000000001,\"area_cm2\":32.2,\"ix_cm4\":3409,\"iy_cm4\":148,\"sx_cm3\":265.2918287937743,\"sy_cm3\":29.01960784313726,\"rx_cm\":10.3,\"ry_cm\":2.1,\"j_cm4\":6.7,\"cw_cm6\":22866.73},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 19\",\"estado\":\"Revisado\",\"masa_kg_m\":28.5,\"d_ala_diametro_cm\":26,\"bf_b_cm\":10.2,\"tw_cm\":0.64,\"tf_t_cm\":1,\"area_cm2\":36.3,\"ix_cm4\":4008,\"iy_cm4\":176,\"sx_cm3\":308.3076923076923,\"sy_cm3\":34.50980392156863,\"rx_cm\":10.5,\"ry_cm\":2.2,\"j_cm4\":9.6,\"cw_cm6\":27500},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 22\",\"estado\":\"Revisado\",\"masa_kg_m\":32.9,\"d_ala_diametro_cm\":25.8,\"bf_b_cm\":14.6,\"tw_cm\":0.61,\"tf_t_cm\":0.9099999999999999,\"area_cm2\":41.9,\"ix_cm4\":4912,\"iy_cm4\":475,\"sx_cm3\":380.7751937984496,\"sy_cm3\":65.06849315068493,\"rx_cm\":10.8,\"ry_cm\":3.4,\"j_cm4\":10,\"cw_cm6\":73567.06},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 26\",\"estado\":\"Revisado\",\"masa_kg_m\":38.5,\"d_ala_diametro_cm\":26.2,\"bf_b_cm\":14.7,\"tw_cm\":0.6599999999999999,\"tf_t_cm\":1.1199999999999999,\"area_cm2\":49.1,\"ix_cm4\":5994,\"iy_cm4\":587,\"sx_cm3\":457.55725190839695,\"sy_cm3\":79.8639455782313,\"rx_cm\":11,\"ry_cm\":3.5,\"j_cm4\":16.6,\"cw_cm6\":92306.69},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 30\",\"estado\":\"Revisado\",\"masa_kg_m\":44.8,\"d_ala_diametro_cm\":26.6,\"bf_b_cm\":14.8,\"tw_cm\":0.76,\"tf_t_cm\":1.3,\"area_cm2\":57,\"ix_cm4\":7076,\"iy_cm4\":695,\"sx_cm3\":532.0300751879699,\"sy_cm3\":93.91891891891892,\"rx_cm\":11.1,\"ry_cm\":3.5,\"j_cm4\":25.8,\"cw_cm6\":111215.64},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 33\",\"estado\":\"Revisado\",\"masa_kg_m\":49.2,\"d_ala_diametro_cm\":24.7,\"bf_b_cm\":20.2,\"tw_cm\":0.74,\"tf_t_cm\":1.1,\"area_cm2\":62.6,\"ix_cm4\":7076,\"iy_cm4\":1523,\"sx_cm3\":572.9554655870445,\"sy_cm3\":150.7920792079208,\"rx_cm\":10.6,\"ry_cm\":4.9,\"j_cm4\":24.1,\"cw_cm6\":212062.52},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 39\",\"estado\":\"Revisado\",\"masa_kg_m\":58.2,\"d_ala_diametro_cm\":25.2,\"bf_b_cm\":20.3,\"tw_cm\":0.8,\"tf_t_cm\":1.35,\"area_cm2\":74.2,\"ix_cm4\":8699,\"iy_cm4\":1873,\"sx_cm3\":690.3968253968254,\"sy_cm3\":184.53201970443348,\"rx_cm\":10.8,\"ry_cm\":5,\"j_cm4\":40.8,\"cw_cm6\":266351.14},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 45\",\"estado\":\"Revisado\",\"masa_kg_m\":67.4,\"d_ala_diametro_cm\":25.7,\"bf_b_cm\":20.2,\"tw_cm\":0.89,\"tf_t_cm\":1.6,\"area_cm2\":85.8,\"ix_cm4\":10323,\"iy_cm4\":2223,\"sx_cm3\":803.3463035019455,\"sy_cm3\":220.0990099009901,\"rx_cm\":11,\"ry_cm\":5.1,\"j_cm4\":62.9,\"cw_cm6\":322785.16},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 49\",\"estado\":\"Revisado\",\"masa_kg_m\":72.9,\"d_ala_diametro_cm\":25.3,\"bf_b_cm\":25.4,\"tw_cm\":0.86,\"tf_t_cm\":1.42,\"area_cm2\":92.9,\"ix_cm4\":11321,\"iy_cm4\":3888,\"sx_cm3\":894.9407114624506,\"sy_cm3\":306.1417322834646,\"rx_cm\":11,\"ry_cm\":6.5,\"j_cm4\":57.9,\"cw_cm6\":554287.28},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 54\",\"estado\":\"Revisado\",\"masa_kg_m\":80,\"d_ala_diametro_cm\":25.6,\"bf_b_cm\":25.5,\"tw_cm\":0.9400000000000001,\"tf_t_cm\":1.56,\"area_cm2\":101.9,\"ix_cm4\":12612,\"iy_cm4\":4287,\"sx_cm3\":985.3125,\"sy_cm3\":336.2352941176471,\"rx_cm\":11.1,\"ry_cm\":6.5,\"j_cm4\":75.8,\"cw_cm6\":619387.47},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 60\",\"estado\":\"Revisado\",\"masa_kg_m\":89.1,\"d_ala_diametro_cm\":26,\"bf_b_cm\":25.6,\"tw_cm\":1.0699999999999998,\"tf_t_cm\":1.73,\"area_cm2\":113.6,\"ix_cm4\":14193,\"iy_cm4\":4828,\"sx_cm3\":1091.7692307692307,\"sy_cm3\":377.1875,\"rx_cm\":11.2,\"ry_cm\":6.5,\"j_cm4\":103,\"cw_cm6\":710962.71},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 68\",\"estado\":\"Revisado\",\"masa_kg_m\":101.3,\"d_ala_diametro_cm\":26.4,\"bf_b_cm\":25.7,\"tw_cm\":1.19,\"tf_t_cm\":1.9600000000000002,\"area_cm2\":129,\"ix_cm4\":16400,\"iy_cm4\":5577,\"sx_cm3\":1242.4242424242425,\"sy_cm3\":434.0077821011673,\"rx_cm\":11.3,\"ry_cm\":6.6,\"j_cm4\":148,\"cw_cm6\":832804.49},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 77\",\"estado\":\"Revisado\",\"masa_kg_m\":114.5,\"d_ala_diametro_cm\":26.9,\"bf_b_cm\":25.9,\"tw_cm\":1.35,\"tf_t_cm\":2.21,\"area_cm2\":145.8,\"ix_cm4\":18938,\"iy_cm4\":6410,\"sx_cm3\":1408.0297397769518,\"sy_cm3\":494.98069498069503,\"rx_cm\":11.4,\"ry_cm\":6.6,\"j_cm4\":213,\"cw_cm6\":976877.75},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 88\",\"estado\":\"Revisado\",\"masa_kg_m\":131.2,\"d_ala_diametro_cm\":27.5,\"bf_b_cm\":26.1,\"tw_cm\":1.54,\"tf_t_cm\":2.5100000000000002,\"area_cm2\":167.1,\"ix_cm4\":22227,\"iy_cm4\":7451,\"sx_cm3\":1616.509090909091,\"sy_cm3\":570.9578544061302,\"rx_cm\":11.5,\"ry_cm\":6.7,\"j_cm4\":313,\"cw_cm6\":1163287.56},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 100\",\"estado\":\"Revisado\",\"masa_kg_m\":148.9,\"d_ala_diametro_cm\":28.2,\"bf_b_cm\":26.3,\"tw_cm\":1.73,\"tf_t_cm\":2.84,\"area_cm2\":189.7,\"ix_cm4\":25931,\"iy_cm4\":8616,\"sx_cm3\":1839.0780141843973,\"sy_cm3\":655.2091254752852,\"rx_cm\":11.7,\"ry_cm\":6.7,\"j_cm4\":454,\"cw_cm6\":1385301.16},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 10 \u00d7 112\",\"estado\":\"Revisado\",\"masa_kg_m\":166.6,\"d_ala_diametro_cm\":28.9,\"bf_b_cm\":26.5,\"tw_cm\":1.92,\"tf_t_cm\":3.18,\"area_cm2\":212.3,\"ix_cm4\":29802,\"iy_cm4\":9823,\"sx_cm3\":2062.42214532872,\"sy_cm3\":741.3584905660377,\"rx_cm\":11.8,\"ry_cm\":6.8,\"j_cm4\":629,\"cw_cm6\":1624523.81},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 14\",\"estado\":\"Con restricciones\",\"masa_kg_m\":21.1,\"d_ala_diametro_cm\":30.3,\"bf_b_cm\":10.1,\"tw_cm\":0.5,\"tf_t_cm\":0.5700000000000001,\"area_cm2\":26.8,\"ix_cm4\":3688,\"iy_cm4\":98,\"sx_cm3\":243.43234323432344,\"sy_cm3\":19.405940594059405,\"rx_cm\":11.7,\"ry_cm\":1.9,\"j_cm4\":2.9,\"cw_cm6\":21654.89},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 16\",\"estado\":\"Revisado\",\"masa_kg_m\":23.9,\"d_ala_diametro_cm\":30.5,\"bf_b_cm\":10.1,\"tw_cm\":0.5599999999999999,\"tf_t_cm\":0.67,\"area_cm2\":30.4,\"ix_cm4\":4287,\"iy_cm4\":117,\"sx_cm3\":281.11475409836066,\"sy_cm3\":23.168316831683168,\"rx_cm\":11.9,\"ry_cm\":2,\"j_cm4\":4.2,\"cw_cm6\":26027.5},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 19\",\"estado\":\"Con restricciones\",\"masa_kg_m\":28.2,\"d_ala_diametro_cm\":30.9,\"bf_b_cm\":10.2,\"tw_cm\":0.6,\"tf_t_cm\":0.89,\"area_cm2\":35.9,\"ix_cm4\":5411,\"iy_cm4\":157,\"sx_cm3\":350.2265372168285,\"sy_cm3\":30.784313725490197,\"rx_cm\":12.2,\"ry_cm\":2.1,\"j_cm4\":7.5,\"cw_cm6\":35348.55},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 22\",\"estado\":\"Revisado\",\"masa_kg_m\":32.8,\"d_ala_diametro_cm\":31.3,\"bf_b_cm\":10.2,\"tw_cm\":0.6599999999999999,\"tf_t_cm\":1.08,\"area_cm2\":41.8,\"ix_cm4\":6493,\"iy_cm4\":194,\"sx_cm3\":414.888178913738,\"sy_cm3\":38.03921568627451,\"rx_cm\":12.5,\"ry_cm\":2.2,\"j_cm4\":12,\"cw_cm6\":44292.55},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 26\",\"estado\":\"Revisado\",\"masa_kg_m\":38.7,\"d_ala_diametro_cm\":31,\"bf_b_cm\":16.5,\"tw_cm\":0.58,\"tf_t_cm\":0.97,\"area_cm2\":49.4,\"ix_cm4\":8491,\"iy_cm4\":720,\"sx_cm3\":547.8064516129032,\"sy_cm3\":87.27272727272727,\"rx_cm\":13.1,\"ry_cm\":3.8,\"j_cm4\":12.5,\"cw_cm6\":162324.16},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 30\",\"estado\":\"Revisado\",\"masa_kg_m\":44.5,\"d_ala_diametro_cm\":31.3,\"bf_b_cm\":16.6,\"tw_cm\":0.6599999999999999,\"tf_t_cm\":1.1199999999999999,\"area_cm2\":56.7,\"ix_cm4\":9906,\"iy_cm4\":845,\"sx_cm3\":632.9712460063897,\"sy_cm3\":101.80722891566265,\"rx_cm\":13.2,\"ry_cm\":3.9,\"j_cm4\":19.1,\"cw_cm6\":192413.34},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 35\",\"estado\":\"Revisado\",\"masa_kg_m\":52.2,\"d_ala_diametro_cm\":31.8,\"bf_b_cm\":16.7,\"tw_cm\":0.76,\"tf_t_cm\":1.3199999999999998,\"area_cm2\":66.5,\"ix_cm4\":11863,\"iy_cm4\":1020,\"sx_cm3\":746.1006289308176,\"sy_cm3\":122.1556886227545,\"rx_cm\":13.3,\"ry_cm\":3.9,\"j_cm4\":30.8,\"cw_cm6\":236902.75},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 40\",\"estado\":\"Revisado\",\"masa_kg_m\":59.8,\"d_ala_diametro_cm\":30.3,\"bf_b_cm\":20.3,\"tw_cm\":0.75,\"tf_t_cm\":1.31,\"area_cm2\":76.1,\"ix_cm4\":12903,\"iy_cm4\":1836,\"sx_cm3\":851.6831683168317,\"sy_cm3\":180.88669950738915,\"rx_cm\":13,\"ry_cm\":4.9,\"j_cm4\":39.5,\"cw_cm6\":385752.83},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 45\",\"estado\":\"Revisado\",\"masa_kg_m\":66.9,\"d_ala_diametro_cm\":30.6,\"bf_b_cm\":20.4,\"tw_cm\":0.85,\"tf_t_cm\":1.46,\"area_cm2\":85.2,\"ix_cm4\":14568,\"iy_cm4\":2081,\"sx_cm3\":952.156862745098,\"sy_cm3\":204.01960784313727,\"rx_cm\":13.1,\"ry_cm\":4.9,\"j_cm4\":54.5,\"cw_cm6\":441764.88},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 50\",\"estado\":\"Revisado\",\"masa_kg_m\":74.4,\"d_ala_diametro_cm\":31,\"bf_b_cm\":20.5,\"tw_cm\":0.9400000000000001,\"tf_t_cm\":1.6300000000000001,\"area_cm2\":94.8,\"ix_cm4\":16400,\"iy_cm4\":2343,\"sx_cm3\":1058.0645161290322,\"sy_cm3\":228.58536585365854,\"rx_cm\":13.2,\"ry_cm\":5,\"j_cm4\":74.1,\"cw_cm6\":505266.13},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 53\",\"estado\":\"Revisado\",\"masa_kg_m\":79,\"d_ala_diametro_cm\":30.6,\"bf_b_cm\":25.4,\"tw_cm\":0.8800000000000001,\"tf_t_cm\":1.46,\"area_cm2\":100.7,\"ix_cm4\":17690,\"iy_cm4\":3987,\"sx_cm3\":1156.2091503267973,\"sy_cm3\":313.93700787401576,\"rx_cm\":13.3,\"ry_cm\":6.3,\"j_cm4\":65.8,\"cw_cm6\":846379.9},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 58\",\"estado\":\"Revisado\",\"masa_kg_m\":86.1,\"d_ala_diametro_cm\":31,\"bf_b_cm\":25.4,\"tw_cm\":0.9099999999999999,\"tf_t_cm\":1.6300000000000001,\"area_cm2\":109.7,\"ix_cm4\":19771,\"iy_cm4\":4454,\"sx_cm3\":1275.5483870967741,\"sy_cm3\":350.70866141732284,\"rx_cm\":13.4,\"ry_cm\":6.4,\"j_cm4\":87.4,\"cw_cm6\":960501.65},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 65\",\"estado\":\"Revisado\",\"masa_kg_m\":96.7,\"d_ala_diametro_cm\":30.8,\"bf_b_cm\":30.5,\"tw_cm\":0.99,\"tf_t_cm\":1.54,\"area_cm2\":123.2,\"ix_cm4\":22185,\"iy_cm4\":7242,\"sx_cm3\":1440.5844155844156,\"sy_cm3\":474.88524590163934,\"rx_cm\":13.4,\"ry_cm\":7.7,\"j_cm4\":90.7,\"cw_cm6\":1550055.23},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 72\",\"estado\":\"Revisado\",\"masa_kg_m\":106.9,\"d_ala_diametro_cm\":31.1,\"bf_b_cm\":30.6,\"tw_cm\":1.09,\"tf_t_cm\":1.7,\"area_cm2\":136.1,\"ix_cm4\":24850,\"iy_cm4\":8116,\"sx_cm3\":1598.0707395498391,\"sy_cm3\":530.4575163398692,\"rx_cm\":13.5,\"ry_cm\":7.7,\"j_cm4\":122,\"cw_cm6\":1753786.44},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 79\",\"estado\":\"Revisado\",\"masa_kg_m\":117.5,\"d_ala_diametro_cm\":31.4,\"bf_b_cm\":30.7,\"tw_cm\":1.19,\"tf_t_cm\":1.8699999999999999,\"area_cm2\":149.7,\"ix_cm4\":27554,\"iy_cm4\":8991,\"sx_cm3\":1755.031847133758,\"sy_cm3\":585.7328990228013,\"rx_cm\":13.6,\"ry_cm\":7.7,\"j_cm4\":160,\"cw_cm6\":1960084.98},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 87\",\"estado\":\"Revisado\",\"masa_kg_m\":129.7,\"d_ala_diametro_cm\":31.8,\"bf_b_cm\":30.8,\"tw_cm\":1.31,\"tf_t_cm\":2.06,\"area_cm2\":165.2,\"ix_cm4\":30801,\"iy_cm4\":10031,\"sx_cm3\":1937.1698113207547,\"sy_cm3\":651.3636363636364,\"rx_cm\":13.7,\"ry_cm\":7.8,\"j_cm4\":212,\"cw_cm6\":2218023.62},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 96\",\"estado\":\"Revisado\",\"masa_kg_m\":142.8,\"d_ala_diametro_cm\":32.3,\"bf_b_cm\":30.9,\"tw_cm\":1.4,\"tf_t_cm\":2.29,\"area_cm2\":181.9,\"ix_cm4\":34672,\"iy_cm4\":11238,\"sx_cm3\":2146.8730650154803,\"sy_cm3\":727.3786407766991,\"rx_cm\":13.8,\"ry_cm\":7.8,\"j_cm4\":286,\"cw_cm6\":2530235.98},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 106\",\"estado\":\"Revisado\",\"masa_kg_m\":158,\"d_ala_diametro_cm\":32.7,\"bf_b_cm\":31,\"tw_cm\":1.55,\"tf_t_cm\":2.5100000000000002,\"area_cm2\":201.3,\"ix_cm4\":38834,\"iy_cm4\":12529,\"sx_cm3\":2375.1681957186543,\"sy_cm3\":808.3225806451613,\"rx_cm\":13.9,\"ry_cm\":7.9,\"j_cm4\":380,\"cw_cm6\":2854845.72},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 12 \u00d7 120\",\"estado\":\"Revisado\",\"masa_kg_m\":178.8,\"d_ala_diametro_cm\":33.3,\"bf_b_cm\":31.2,\"tw_cm\":1.8,\"tf_t_cm\":2.81,\"area_cm2\":227.8,\"ix_cm4\":44537,\"iy_cm4\":14360,\"sx_cm3\":2674.894894894895,\"sy_cm3\":920.5128205128206,\"rx_cm\":14,\"ry_cm\":8,\"j_cm4\":537,\"cw_cm6\":3337407.96},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 22\",\"estado\":\"Revisado\",\"masa_kg_m\":32.9,\"d_ala_diametro_cm\":34.9,\"bf_b_cm\":12.7,\"tw_cm\":0.58,\"tf_t_cm\":0.85,\"area_cm2\":41.9,\"ix_cm4\":8283,\"iy_cm4\":291,\"sx_cm3\":474.6704871060172,\"sy_cm3\":45.82677165354331,\"rx_cm\":14.1,\"ry_cm\":2.6,\"j_cm4\":8.7,\"cw_cm6\":84346.53},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 26\",\"estado\":\"Revisado\",\"masa_kg_m\":38.9,\"d_ala_diametro_cm\":35.3,\"bf_b_cm\":12.8,\"tw_cm\":0.64,\"tf_t_cm\":1.0699999999999998,\"area_cm2\":49.6,\"ix_cm4\":10198,\"iy_cm4\":371,\"sx_cm3\":577.7903682719548,\"sy_cm3\":57.96875,\"rx_cm\":14.4,\"ry_cm\":2.7,\"j_cm4\":15,\"cw_cm6\":108674.52},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 30\",\"estado\":\"Revisado\",\"masa_kg_m\":44.8,\"d_ala_diametro_cm\":35.2,\"bf_b_cm\":17.1,\"tw_cm\":0.6900000000000001,\"tf_t_cm\":0.9800000000000001,\"area_cm2\":57.1,\"ix_cm4\":12112,\"iy_cm4\":816,\"sx_cm3\":688.1818181818181,\"sy_cm3\":95.43859649122807,\"rx_cm\":14.6,\"ry_cm\":3.8,\"j_cm4\":15.8,\"cw_cm6\":238885.71},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 34\",\"estado\":\"Revisado\",\"masa_kg_m\":50.6,\"d_ala_diametro_cm\":35.5,\"bf_b_cm\":17.1,\"tw_cm\":0.72,\"tf_t_cm\":1.16,\"area_cm2\":64.5,\"ix_cm4\":14152,\"iy_cm4\":970,\"sx_cm3\":797.2957746478874,\"sy_cm3\":113.4502923976608,\"rx_cm\":14.8,\"ry_cm\":3.9,\"j_cm4\":23.7,\"cw_cm6\":285964.63},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 38\",\"estado\":\"Revisado\",\"masa_kg_m\":56.7,\"d_ala_diametro_cm\":35.8,\"bf_b_cm\":17.2,\"tw_cm\":0.79,\"tf_t_cm\":1.31,\"area_cm2\":72.3,\"ix_cm4\":16025,\"iy_cm4\":1111,\"sx_cm3\":895.2513966480448,\"sy_cm3\":129.18604651162792,\"rx_cm\":14.9,\"ry_cm\":3.9,\"j_cm4\":33.3,\"cw_cm6\":330400.32},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 43\",\"estado\":\"Revisado\",\"masa_kg_m\":63.8,\"d_ala_diametro_cm\":34.7,\"bf_b_cm\":20.3,\"tw_cm\":0.78,\"tf_t_cm\":1.35,\"area_cm2\":81.3,\"ix_cm4\":17815,\"iy_cm4\":1881,\"sx_cm3\":1026.801152737752,\"sy_cm3\":185.32019704433498,\"rx_cm\":14.8,\"ry_cm\":4.8,\"j_cm4\":43.7,\"cw_cm6\":523022.63},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 48\",\"estado\":\"Revisado\",\"masa_kg_m\":71.4,\"d_ala_diametro_cm\":35,\"bf_b_cm\":20.4,\"tw_cm\":0.86,\"tf_t_cm\":1.51,\"area_cm2\":91,\"ix_cm4\":20187,\"iy_cm4\":2139,\"sx_cm3\":1153.5428571428572,\"sy_cm3\":209.7058823529412,\"rx_cm\":14.9,\"ry_cm\":4.9,\"j_cm4\":60.8,\"cw_cm6\":599764.96},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 53\",\"estado\":\"Revisado\",\"masa_kg_m\":79,\"d_ala_diametro_cm\":35.4,\"bf_b_cm\":20.5,\"tw_cm\":0.9400000000000001,\"tf_t_cm\":1.6800000000000002,\"area_cm2\":100.7,\"ix_cm4\":22518,\"iy_cm4\":2402,\"sx_cm3\":1272.2033898305085,\"sy_cm3\":234.34146341463415,\"rx_cm\":15,\"ry_cm\":4.9,\"j_cm4\":80.7,\"cw_cm6\":682791.56},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 61\",\"estado\":\"Recalculado\",\"masa_kg_m\":90.7,\"d_ala_diametro_cm\":35.3,\"bf_b_cm\":25.4,\"tw_cm\":0.95,\"tf_t_cm\":1.64,\"area_cm2\":115.5,\"ix_cm4\":26640,\"iy_cm4\":4454,\"sx_cm3\":1509.3484419263457,\"sy_cm3\":350.70866141732284,\"rx_cm\":15.2,\"ry_cm\":6.2,\"j_cm4\":91.6,\"cw_cm6\":1261590.6},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 68\",\"estado\":\"Revisado\",\"masa_kg_m\":101.3,\"d_ala_diametro_cm\":35.7,\"bf_b_cm\":25.5,\"tw_cm\":1.05,\"tf_t_cm\":1.83,\"area_cm2\":129,\"ix_cm4\":30093,\"iy_cm4\":5036,\"sx_cm3\":1685.8823529411764,\"sy_cm3\":394.98039215686276,\"rx_cm\":15.3,\"ry_cm\":6.2,\"j_cm4\":125.7,\"cw_cm6\":1444295.72},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 74\",\"estado\":\"Revisado\",\"masa_kg_m\":110.4,\"d_ala_diametro_cm\":36,\"bf_b_cm\":25.6,\"tw_cm\":1.1400000000000001,\"tf_t_cm\":1.9899999999999998,\"area_cm2\":140.7,\"ix_cm4\":33132,\"iy_cm4\":5577,\"sx_cm3\":1840.6666666666667,\"sy_cm3\":435.703125,\"rx_cm\":15.3,\"ry_cm\":6.3,\"j_cm4\":161.5,\"cw_cm6\":1612701.23},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 14 \u00d7 82\",\"estado\":\"Revisado\",\"masa_kg_m\":122.1,\"d_ala_diametro_cm\":36.3,\"bf_b_cm\":25.7,\"tw_cm\":1.3,\"tf_t_cm\":2.17,\"area_cm2\":155.5,\"ix_cm4\":36711,\"iy_cm4\":6160,\"sx_cm3\":2022.6446280991738,\"sy_cm3\":479.3774319066148,\"rx_cm\":15.4,\"ry_cm\":6.3,\"j_cm4\":211.5,\"cw_cm6\":1793879.63},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 26\",\"estado\":\"Revisado\",\"masa_kg_m\":38.9,\"d_ala_diametro_cm\":39.9,\"bf_b_cm\":14,\"tw_cm\":0.64,\"tf_t_cm\":0.8800000000000001,\"area_cm2\":49.6,\"ix_cm4\":12529,\"iy_cm4\":399,\"sx_cm3\":628.0200501253133,\"sy_cm3\":57,\"rx_cm\":15.9,\"ry_cm\":2.8,\"j_cm4\":10.8,\"cw_cm6\":151875.4},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 31\",\"estado\":\"Recalculado\",\"masa_kg_m\":46.2,\"d_ala_diametro_cm\":40.3,\"bf_b_cm\":14,\"tw_cm\":0.7,\"tf_t_cm\":1.1199999999999999,\"area_cm2\":58.8,\"ix_cm4\":15609,\"iy_cm4\":516,\"sx_cm3\":774.6401985111663,\"sy_cm3\":73.71428571428571,\"rx_cm\":16.3,\"ry_cm\":3,\"j_cm4\":19.1,\"cw_cm6\":198024.34},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 36\",\"estado\":\"Con restricciones\",\"masa_kg_m\":53.7,\"d_ala_diametro_cm\":40.3,\"bf_b_cm\":17.7,\"tw_cm\":0.75,\"tf_t_cm\":1.09,\"area_cm2\":68.4,\"ix_cm4\":18647,\"iy_cm4\":1020,\"sx_cm3\":925.409429280397,\"sy_cm3\":115.2542372881356,\"rx_cm\":16.5,\"ry_cm\":3.9,\"j_cm4\":22.5,\"cw_cm6\":392043.15},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 40\",\"estado\":\"Con restricciones\",\"masa_kg_m\":59.8,\"d_ala_diametro_cm\":40.7,\"bf_b_cm\":17.8,\"tw_cm\":0.78,\"tf_t_cm\":1.28,\"area_cm2\":76.1,\"ix_cm4\":21561,\"iy_cm4\":1203,\"sx_cm3\":1059.5085995085994,\"sy_cm3\":135.1685393258427,\"rx_cm\":16.8,\"ry_cm\":4,\"j_cm4\":32.9,\"cw_cm6\":467346.37},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 45\",\"estado\":\"Con restricciones\",\"masa_kg_m\":67.4,\"d_ala_diametro_cm\":41,\"bf_b_cm\":17.9,\"tw_cm\":0.8800000000000001,\"tf_t_cm\":1.44,\"area_cm2\":85.8,\"ix_cm4\":24391,\"iy_cm4\":1365,\"sx_cm3\":1189.8048780487804,\"sy_cm3\":152.51396648044695,\"rx_cm\":16.9,\"ry_cm\":4,\"j_cm4\":46.2,\"cw_cm6\":534054.07},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 50\",\"estado\":\"Con restricciones\",\"masa_kg_m\":74.4,\"d_ala_diametro_cm\":41.3,\"bf_b_cm\":18,\"tw_cm\":0.97,\"tf_t_cm\":1.6,\"area_cm2\":94.8,\"ix_cm4\":27430,\"iy_cm4\":1548,\"sx_cm3\":1328.3292978208233,\"sy_cm3\":172,\"rx_cm\":17,\"ry_cm\":4,\"j_cm4\":63.3,\"cw_cm6\":609946.83},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 57\",\"estado\":\"Con restricciones\",\"masa_kg_m\":83.1,\"d_ala_diametro_cm\":41.7,\"bf_b_cm\":18.1,\"tw_cm\":1.09,\"tf_t_cm\":1.8199999999999998,\"area_cm2\":108.4,\"ix_cm4\":31550,\"iy_cm4\":1794,\"sx_cm3\":1513.189448441247,\"sy_cm3\":198.232044198895,\"rx_cm\":17.1,\"ry_cm\":4.1,\"j_cm4\":92.4,\"cw_cm6\":713300.86},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 67\",\"estado\":\"Con restricciones\",\"masa_kg_m\":99.8,\"d_ala_diametro_cm\":41.5,\"bf_b_cm\":26,\"tw_cm\":1,\"tf_t_cm\":1.69,\"area_cm2\":127.1,\"ix_cm4\":39708,\"iy_cm4\":4953,\"sx_cm3\":1913.6385542168675,\"sy_cm3\":381,\"rx_cm\":17.7,\"ry_cm\":6.2,\"j_cm4\":99.5,\"cw_cm6\":1962423.3},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 77\",\"estado\":\"Con restricciones\",\"masa_kg_m\":114.5,\"d_ala_diametro_cm\":42,\"bf_b_cm\":26.1,\"tw_cm\":1.16,\"tf_t_cm\":1.9300000000000002,\"area_cm2\":145.8,\"ix_cm4\":46202,\"iy_cm4\":5744,\"sx_cm3\":2200.095238095238,\"sy_cm3\":440.15325670498083,\"rx_cm\":17.8,\"ry_cm\":6.3,\"j_cm4\":148.6,\"cw_cm6\":2305648.64},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 89\",\"estado\":\"Con restricciones\",\"masa_kg_m\":132.7,\"d_ala_diametro_cm\":42.5,\"bf_b_cm\":26.3,\"tw_cm\":1.3199999999999998,\"tf_t_cm\":2.2199999999999998,\"area_cm2\":169,\"ix_cm4\":54110,\"iy_cm4\":6785,\"sx_cm3\":2546.3529411764707,\"sy_cm3\":515.9695817490494,\"rx_cm\":17.9,\"ry_cm\":6.3,\"j_cm4\":226.9,\"cw_cm6\":2752128.99},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 16 \u00d7 100\",\"estado\":\"Con restricciones\",\"masa_kg_m\":148.9,\"d_ala_diametro_cm\":43.1,\"bf_b_cm\":26.5,\"tw_cm\":1.49,\"tf_t_cm\":2.5,\"area_cm2\":189.7,\"ix_cm4\":62018,\"iy_cm4\":7742,\"sx_cm3\":2877.8654292343385,\"sy_cm3\":584.3018867924528,\"rx_cm\":18,\"ry_cm\":6.4,\"j_cm4\":321.7,\"cw_cm6\":3190400.78},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 35\",\"estado\":\"Revisado\",\"masa_kg_m\":52.2,\"d_ala_diametro_cm\":45,\"bf_b_cm\":15.2,\"tw_cm\":0.76,\"tf_t_cm\":1.08,\"area_cm2\":66.5,\"ix_cm4\":21228,\"iy_cm4\":637,\"sx_cm3\":943.4666666666667,\"sy_cm3\":83.81578947368422,\"rx_cm\":17.9,\"ry_cm\":3.1,\"j_cm4\":21.2,\"cw_cm6\":307187.9},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 40\",\"estado\":\"Revisado\",\"masa_kg_m\":59.8,\"d_ala_diametro_cm\":45.3,\"bf_b_cm\":15.3,\"tw_cm\":0.8,\"tf_t_cm\":1.33,\"area_cm2\":76.1,\"ix_cm4\":25473,\"iy_cm4\":795,\"sx_cm3\":1124.635761589404,\"sy_cm3\":103.92156862745098,\"rx_cm\":18.3,\"ry_cm\":3.2,\"j_cm4\":33.7,\"cw_cm6\":384255.48},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 46\",\"estado\":\"Revisado\",\"masa_kg_m\":68.4,\"d_ala_diametro_cm\":45.9,\"bf_b_cm\":15.4,\"tw_cm\":0.9099999999999999,\"tf_t_cm\":1.54,\"area_cm2\":87.1,\"ix_cm4\":29636,\"iy_cm4\":937,\"sx_cm3\":1291.3289760348584,\"sy_cm3\":121.68831168831169,\"rx_cm\":18.4,\"ry_cm\":3.3,\"j_cm4\":50.8,\"cw_cm6\":460959.4},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 50\",\"estado\":\"Revisado\",\"masa_kg_m\":74.3,\"d_ala_diametro_cm\":45.7,\"bf_b_cm\":19,\"tw_cm\":0.9,\"tf_t_cm\":1.45,\"area_cm2\":94.8,\"ix_cm4\":33298,\"iy_cm4\":1669,\"sx_cm3\":1457.2428884026258,\"sy_cm3\":175.68421052631578,\"rx_cm\":18.7,\"ry_cm\":4.2,\"j_cm4\":51.6,\"cw_cm6\":817001.58},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 55\",\"estado\":\"Revisado\",\"masa_kg_m\":82,\"d_ala_diametro_cm\":46,\"bf_b_cm\":19.1,\"tw_cm\":0.99,\"tf_t_cm\":1.6,\"area_cm2\":104.5,\"ix_cm4\":37045,\"iy_cm4\":1869,\"sx_cm3\":1610.6521739130435,\"sy_cm3\":195.7068062827225,\"rx_cm\":18.8,\"ry_cm\":4.2,\"j_cm4\":69.1,\"cw_cm6\":921117.96},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 60\",\"estado\":\"Revisado\",\"masa_kg_m\":89.1,\"d_ala_diametro_cm\":46.3,\"bf_b_cm\":19.2,\"tw_cm\":1.05,\"tf_t_cm\":1.77,\"area_cm2\":113.6,\"ix_cm4\":40957,\"iy_cm4\":2085,\"sx_cm3\":1769.2008639308856,\"sy_cm3\":217.1875,\"rx_cm\":18.9,\"ry_cm\":4.3,\"j_cm4\":90.3,\"cw_cm6\":1033597.52},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 65\",\"estado\":\"Revisado\",\"masa_kg_m\":96.7,\"d_ala_diametro_cm\":46.6,\"bf_b_cm\":19.3,\"tw_cm\":1.1400000000000001,\"tf_t_cm\":1.9100000000000001,\"area_cm2\":123.2,\"ix_cm4\":44537,\"iy_cm4\":2281,\"sx_cm3\":1911.4592274678112,\"sy_cm3\":236.37305699481865,\"rx_cm\":19,\"ry_cm\":4.3,\"j_cm4\":114,\"cw_cm6\":1138901.08},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 71\",\"estado\":\"Revisado\",\"masa_kg_m\":105.3,\"d_ala_diametro_cm\":46.9,\"bf_b_cm\":19.4,\"tw_cm\":1.26,\"tf_t_cm\":2.06,\"area_cm2\":134.2,\"ix_cm4\":48699,\"iy_cm4\":2510,\"sx_cm3\":2076.716417910448,\"sy_cm3\":258.7628865979382,\"rx_cm\":19.1,\"ry_cm\":4.3,\"j_cm4\":145,\"cw_cm6\":1261667.56},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 76\",\"estado\":\"Revisado\",\"masa_kg_m\":112.9,\"d_ala_diametro_cm\":46.3,\"bf_b_cm\":28,\"tw_cm\":1.08,\"tf_t_cm\":1.73,\"area_cm2\":143.9,\"ix_cm4\":55359,\"iy_cm4\":6327,\"sx_cm3\":2391.317494600432,\"sy_cm3\":451.92857142857144,\"rx_cm\":19.6,\"ry_cm\":6.6,\"j_cm4\":118,\"cw_cm6\":3142122.49},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 86\",\"estado\":\"Con restricciones\",\"masa_kg_m\":128.1,\"d_ala_diametro_cm\":46.7,\"bf_b_cm\":28.2,\"tw_cm\":1.22,\"tf_t_cm\":1.9600000000000002,\"area_cm2\":163.2,\"ix_cm4\":63683,\"iy_cm4\":7284,\"sx_cm3\":2727.323340471092,\"sy_cm3\":516.5957446808511,\"rx_cm\":19.7,\"ry_cm\":6.7,\"j_cm4\":171,\"cw_cm6\":3645036.7},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 97\",\"estado\":\"Revisado\",\"masa_kg_m\":144.3,\"d_ala_diametro_cm\":47.2,\"bf_b_cm\":28.3,\"tw_cm\":1.3599999999999999,\"tf_t_cm\":2.21,\"area_cm2\":183.9,\"ix_cm4\":72840,\"iy_cm4\":8366,\"sx_cm3\":3086.4406779661017,\"sy_cm3\":591.2367491166077,\"rx_cm\":19.9,\"ry_cm\":6.7,\"j_cm4\":244,\"cw_cm6\":4233405.36},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 106\",\"estado\":\"Revisado\",\"masa_kg_m\":157.5,\"d_ala_diametro_cm\":47.6,\"bf_b_cm\":28.4,\"tw_cm\":1.5,\"tf_t_cm\":2.3899999999999997,\"area_cm2\":200.7,\"ix_cm4\":79500,\"iy_cm4\":9157,\"sx_cm3\":3340.3361344537816,\"sy_cm3\":644.8591549295775,\"rx_cm\":19.9,\"ry_cm\":6.8,\"j_cm4\":311,\"cw_cm6\":4679099.03},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 18 \u00d7 119\",\"estado\":\"Revisado\",\"masa_kg_m\":177.8,\"d_ala_diametro_cm\":48.2,\"bf_b_cm\":28.6,\"tw_cm\":1.6600000000000001,\"tf_t_cm\":2.69,\"area_cm2\":226.5,\"ix_cm4\":91154,\"iy_cm4\":10531,\"sx_cm3\":3782.323651452282,\"sy_cm3\":736.4335664335664,\"rx_cm\":20,\"ry_cm\":6.8,\"j_cm4\":441,\"cw_cm6\":5452846.75},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 44\",\"estado\":\"Revisado\",\"masa_kg_m\":65.8,\"d_ala_diametro_cm\":52.5,\"bf_b_cm\":16.5,\"tw_cm\":0.89,\"tf_t_cm\":1.1400000000000001,\"area_cm2\":83.9,\"ix_cm4\":35088,\"iy_cm4\":862,\"sx_cm3\":1336.6857142857143,\"sy_cm3\":104.48484848484848,\"rx_cm\":20.5,\"ry_cm\":3.2,\"j_cm4\":32,\"cw_cm6\":568456.59},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 50\",\"estado\":\"Revisado\",\"masa_kg_m\":74.4,\"d_ala_diametro_cm\":52.9,\"bf_b_cm\":16.6,\"tw_cm\":0.97,\"tf_t_cm\":1.3599999999999999,\"area_cm2\":94.8,\"ix_cm4\":40957,\"iy_cm4\":1036,\"sx_cm3\":1548.468809073724,\"sy_cm3\":124.81927710843372,\"rx_cm\":20.8,\"ry_cm\":3.3,\"j_cm4\":47.5,\"cw_cm6\":688000.24},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 57\",\"estado\":\"Revisado\",\"masa_kg_m\":84.6,\"d_ala_diametro_cm\":53.5,\"bf_b_cm\":16.6,\"tw_cm\":1.03,\"tf_t_cm\":1.65,\"area_cm2\":107.7,\"ix_cm4\":48699,\"iy_cm4\":1274,\"sx_cm3\":1820.5233644859813,\"sy_cm3\":153.49397590361446,\"rx_cm\":21.2,\"ry_cm\":3.4,\"j_cm4\":73.7,\"cw_cm6\":856262.57},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 48\",\"estado\":\"Con restricciones\",\"masa_kg_m\":71.5,\"d_ala_diametro_cm\":52.4,\"bf_b_cm\":20.7,\"tw_cm\":0.89,\"tf_t_cm\":1.1,\"area_cm2\":91.8,\"ix_cm4\":39917,\"iy_cm4\":1611,\"sx_cm3\":1523.5496183206108,\"sy_cm3\":155.65217391304347,\"rx_cm\":20.9,\"ry_cm\":4.2,\"j_cm4\":33,\"cw_cm6\":1059913.15},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 55\",\"estado\":\"Con restricciones\",\"masa_kg_m\":81.9,\"d_ala_diametro_cm\":52.8,\"bf_b_cm\":20.9,\"tw_cm\":0.97,\"tf_t_cm\":1.33,\"area_cm2\":105,\"ix_cm4\":47451,\"iy_cm4\":2015,\"sx_cm3\":1797.3863636363637,\"sy_cm3\":192.82296650717706,\"rx_cm\":21.3,\"ry_cm\":4.4,\"j_cm4\":52,\"cw_cm6\":1334514.8},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 62\",\"estado\":\"Revisado\",\"masa_kg_m\":92.7,\"d_ala_diametro_cm\":53.3,\"bf_b_cm\":20.9,\"tw_cm\":1.02,\"tf_t_cm\":1.56,\"area_cm2\":118.1,\"ix_cm4\":55359,\"iy_cm4\":2393,\"sx_cm3\":2077.2607879924954,\"sy_cm3\":228.9952153110048,\"rx_cm\":21.7,\"ry_cm\":4.5,\"j_cm4\":76.2,\"cw_cm6\":1601531.76},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 68\",\"estado\":\"Revisado\",\"masa_kg_m\":101.3,\"d_ala_diametro_cm\":53.7,\"bf_b_cm\":21,\"tw_cm\":1.09,\"tf_t_cm\":1.77,\"area_cm2\":129,\"ix_cm4\":61602,\"iy_cm4\":2693,\"sx_cm3\":2294.3016759776533,\"sy_cm3\":256.4761904761905,\"rx_cm\":21.8,\"ry_cm\":4.6,\"j_cm4\":102,\"cw_cm6\":1815570.04},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 73\",\"estado\":\"Revisado\",\"masa_kg_m\":108.9,\"d_ala_diametro_cm\":53.9,\"bf_b_cm\":21.1,\"tw_cm\":1.16,\"tf_t_cm\":1.8800000000000001,\"area_cm2\":138.7,\"ix_cm4\":66597,\"iy_cm4\":2939,\"sx_cm3\":2471.1317254174396,\"sy_cm3\":278.5781990521327,\"rx_cm\":21.9,\"ry_cm\":4.6,\"j_cm4\":126,\"cw_cm6\":1988292.57},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 83\",\"estado\":\"Revisado\",\"masa_kg_m\":123.1,\"d_ala_diametro_cm\":54.4,\"bf_b_cm\":21.2,\"tw_cm\":1.31,\"tf_t_cm\":2.12,\"area_cm2\":156.8,\"ix_cm4\":76170,\"iy_cm4\":3388,\"sx_cm3\":2800.3676470588234,\"sy_cm3\":319.62264150943395,\"rx_cm\":22,\"ry_cm\":4.6,\"j_cm4\":181,\"cw_cm6\":2315019.04},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 21 \u00d7 93\",\"estado\":\"Revisado\",\"masa_kg_m\":138.3,\"d_ala_diametro_cm\":54.9,\"bf_b_cm\":21.4,\"tw_cm\":1.47,\"tf_t_cm\":2.3600000000000003,\"area_cm2\":176.1,\"ix_cm4\":86160,\"iy_cm4\":3867,\"sx_cm3\":3138.7978142076504,\"sy_cm3\":361.4018691588785,\"rx_cm\":22.1,\"ry_cm\":4.7,\"j_cm4\":251,\"cw_cm6\":2668666.58},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 55\",\"estado\":\"Revisado\",\"masa_kg_m\":82,\"d_ala_diametro_cm\":59.9,\"bf_b_cm\":17.8,\"tw_cm\":1,\"tf_t_cm\":1.28,\"area_cm2\":104.5,\"ix_cm4\":56191,\"iy_cm4\":1211,\"sx_cm3\":1876.160267111853,\"sy_cm3\":136.06741573033707,\"rx_cm\":23.1,\"ry_cm\":3.4,\"j_cm4\":49,\"cw_cm6\":1040341.16},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 62\",\"estado\":\"Con restricciones\",\"masa_kg_m\":92.2,\"d_ala_diametro_cm\":60.3,\"bf_b_cm\":17.9,\"tw_cm\":1.09,\"tf_t_cm\":1.5,\"area_cm2\":117.4,\"ix_cm4\":64516,\"iy_cm4\":1436,\"sx_cm3\":2139.83416252073,\"sy_cm3\":160.44692737430168,\"rx_cm\":23.4,\"ry_cm\":3.5,\"j_cm4\":71,\"cw_cm6\":1241220.96},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 68\",\"estado\":\"Revisado\",\"masa_kg_m\":101.6,\"d_ala_diametro_cm\":60.3,\"bf_b_cm\":22.8,\"tw_cm\":1.05,\"tf_t_cm\":1.49,\"area_cm2\":129.7,\"ix_cm4\":76170,\"iy_cm4\":2930,\"sx_cm3\":2526.36815920398,\"sy_cm3\":257.0175438596491,\"rx_cm\":24.3,\"ry_cm\":4.7,\"j_cm4\":78,\"cw_cm6\":2533436.29},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 76\",\"estado\":\"Revisado\",\"masa_kg_m\":113.4,\"d_ala_diametro_cm\":60.8,\"bf_b_cm\":22.8,\"tw_cm\":1.1199999999999999,\"tf_t_cm\":1.73,\"area_cm2\":144.5,\"ix_cm4\":87408,\"iy_cm4\":3434,\"sx_cm3\":2875.263157894737,\"sy_cm3\":301.2280701754386,\"rx_cm\":24.6,\"ry_cm\":4.9,\"j_cm4\":112,\"cw_cm6\":2995533.92},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 84\",\"estado\":\"Revisado\",\"masa_kg_m\":125.1,\"d_ala_diametro_cm\":61.2,\"bf_b_cm\":22.9,\"tw_cm\":1.19,\"tf_t_cm\":1.9600000000000002,\"area_cm2\":159.4,\"ix_cm4\":98647,\"iy_cm4\":3929,\"sx_cm3\":3223.7581699346406,\"sy_cm3\":343.14410480349346,\"rx_cm\":24.9,\"ry_cm\":5,\"j_cm4\":154,\"cw_cm6\":3447086.15},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 94\",\"estado\":\"Revisado\",\"masa_kg_m\":140.3,\"d_ala_diametro_cm\":61.7,\"bf_b_cm\":23,\"tw_cm\":1.31,\"tf_t_cm\":2.2199999999999998,\"area_cm2\":178.7,\"ix_cm4\":112382,\"iy_cm4\":4537,\"sx_cm3\":3642.8525121555913,\"sy_cm3\":394.5217391304348,\"rx_cm\":25.1,\"ry_cm\":5,\"j_cm4\":219,\"cw_cm6\":4012829.5},{\"familia\":\"IR\",\"perfil_comercial\":\"IR 24 \u00d7 103\",\"estado\":\"Revisado\",\"masa_kg_m\":153.4,\"d_ala_diametro_cm\":62.3,\"bf_b_cm\":22.9,\"tw_cm\":1.4,\"tf_t_cm\":2.4899999999999998,\"area_cm2\":196,\"ix_cm4\":125000,\"iy_cm4\":5000,\"sx_cm3\":4012.841091492777,\"sy_cm3\":436.6812227074236,\"rx_cm\":25.3,\"ry_cm\":5.1,\"j_cm4\":295,\"cw_cm6\":4471545.13},{\"familia\":\"CE\",\"perfil_comercial\":\"CE\u1d2c 3 \u00d7 3.5\",\"estado\":\"Con restricciones\",\"masa_kg_m\":5.2,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":3.5799999999999996,\"tw_cm\":0.32,\"tf_t_cm\":0.645,\"area_cm2\":6.68,\"ix_cm4\":62.27,\"iy_cm4\":7.03,\"sx_cm3\":16.343832020997375,\"sy_cm3\":2.7677165354330713,\"rx_cm\":3.06,\"ry_cm\":1.03,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE\u1d2c 3 \u00d7 4.1\",\"estado\":\"Con restricciones\",\"masa_kg_m\":6.1,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":3.5799999999999996,\"tw_cm\":0.43,\"tf_t_cm\":0.693,\"area_cm2\":7.68,\"ix_cm4\":66.6,\"iy_cm4\":8.32,\"sx_cm3\":17.48031496062992,\"sy_cm3\":3.3643348160129403,\"rx_cm\":2.95,\"ry_cm\":1.04,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE 4 \u00d7 4.5\",\"estado\":\"Con restricciones\",\"masa_kg_m\":6.7,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":4,\"tw_cm\":0.32,\"tf_t_cm\":0.752,\"area_cm2\":null,\"ix_cm4\":152,\"iy_cm4\":null,\"sx_cm3\":29.921259842519685,\"sy_cm3\":null,\"rx_cm\":4.14,\"ry_cm\":null,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE 4 \u00d7 5.40\",\"estado\":\"Con restricciones\",\"masa_kg_m\":8.04,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":4.01,\"tw_cm\":0.45999999999999996,\"tf_t_cm\":0.752,\"area_cm2\":10.06,\"ix_cm4\":158.2,\"iy_cm4\":13.32,\"sx_cm3\":31.141732283464563,\"sy_cm3\":4.675324675324676,\"rx_cm\":3.98,\"ry_cm\":1.14,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE 6 \u00d7 8.20\",\"estado\":\"Con restricciones\",\"masa_kg_m\":12.2,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":4.88,\"tw_cm\":0.51,\"tf_t_cm\":0.8699999999999999,\"area_cm2\":15.4,\"ix_cm4\":535,\"iy_cm4\":27.6,\"sx_cm3\":70.20997375328083,\"sy_cm3\":7.645429362880887,\"rx_cm\":5.89,\"ry_cm\":1.34,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE 6 \u00d7 10.50\",\"estado\":\"Con restricciones\",\"masa_kg_m\":15.63,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":5.1,\"tw_cm\":0.8,\"tf_t_cm\":0.8699999999999999,\"area_cm2\":19.8,\"ix_cm4\":621,\"iy_cm4\":34.7,\"sx_cm3\":81.49606299212599,\"sy_cm3\":9.012987012987015,\"rx_cm\":5.6,\"ry_cm\":1.32,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"CE\",\"perfil_comercial\":\"CE 6 \u00d7 13.00\",\"estado\":\"Con restricciones\",\"masa_kg_m\":19.35,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":5.4,\"tw_cm\":1.1099999999999999,\"tf_t_cm\":0.8699999999999999,\"area_cm2\":24.5,\"ix_cm4\":711,\"iy_cm4\":42,\"sx_cm3\":93.30708661417323,\"sy_cm3\":10.21897810218978,\"rx_cm\":5.39,\"ry_cm\":1.31,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3/4 \u00d7 1/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":0.88,\"d_ala_diametro_cm\":1.905,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":1.11,\"ix_cm4\":0.37,\"iy_cm4\":0.37,\"sx_cm3\":0.27924528301886786,\"sy_cm3\":0.27924528301886786,\"rx_cm\":0.58,\"ry_cm\":0.58,\"j_cm4\":0.04,\"cw_cm6\":0.01},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 \u00d7 1/8\",\"estado\":\"Con restricciones\",\"masa_kg_m\":1.19,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":1.52,\"ix_cm4\":0.92,\"iy_cm4\":0.92,\"sx_cm3\":0.5168539325842697,\"sy_cm3\":0.5168539325842697,\"rx_cm\":0.79,\"ry_cm\":0.79,\"j_cm4\":0.05,\"cw_cm6\":0.02},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 \u00d7 3/16\",\"estado\":\"Con restricciones\",\"masa_kg_m\":1.73,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":2.21,\"ix_cm4\":1.25,\"iy_cm4\":1.25,\"sx_cm3\":0.7225433526011561,\"sy_cm3\":0.7225433526011561,\"rx_cm\":0.76,\"ry_cm\":0.76,\"j_cm4\":0.17,\"cw_cm6\":0.07},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 \u00d7 1/4\",\"estado\":\"Con restricciones\",\"masa_kg_m\":2.22,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":2.8,\"ix_cm4\":1.54,\"iy_cm4\":1.54,\"sx_cm3\":0.9166666666666666,\"sy_cm3\":0.9166666666666666,\"rx_cm\":0.74,\"ry_cm\":0.74,\"j_cm4\":0.41,\"cw_cm6\":0.16},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/4 \u00d7 1/8\",\"estado\":\"Con restricciones\",\"masa_kg_m\":1.5,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":1.93,\"ix_cm4\":1.83,\"iy_cm4\":1.83,\"sx_cm3\":0.800875273522976,\"sy_cm3\":0.800875273522976,\"rx_cm\":0.97,\"ry_cm\":0.97,\"j_cm4\":0.07,\"cw_cm6\":0.05},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/4 \u00d7 3/16\",\"estado\":\"Con restricciones\",\"masa_kg_m\":2.2,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":2.79,\"ix_cm4\":2.54,\"iy_cm4\":2.54,\"sx_cm3\":1.1519274376417232,\"sy_cm3\":1.1519274376417232,\"rx_cm\":0.97,\"ry_cm\":0.97,\"j_cm4\":0.22,\"cw_cm6\":0.15},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/4 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":2.86,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":3.72,\"ix_cm4\":3.21,\"iy_cm4\":3.21,\"sx_cm3\":1.4895591647331787,\"sy_cm3\":1.4895591647331787,\"rx_cm\":0.94,\"ry_cm\":0.94,\"j_cm4\":0.51,\"cw_cm6\":0.33},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 1/8\",\"estado\":\"Revisado\",\"masa_kg_m\":1.83,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":2.34,\"ix_cm4\":3.25,\"iy_cm4\":3.25,\"sx_cm3\":1.1861313868613137,\"sy_cm3\":1.1861313868613137,\"rx_cm\":1.17,\"ry_cm\":1.17,\"j_cm4\":0.08,\"cw_cm6\":0.09},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 5/32\",\"estado\":\"Revisado\",\"masa_kg_m\":2.25,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":0.396875,\"area_cm2\":2.86,\"ix_cm4\":3.92,\"iy_cm4\":3.92,\"sx_cm3\":1.4464944649446494,\"sy_cm3\":1.4464944649446494,\"rx_cm\":1.17,\"ry_cm\":1.17,\"j_cm4\":0.15,\"cw_cm6\":0.16},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 3/16\",\"estado\":\"Con restricciones\",\"masa_kg_m\":2.68,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":3.43,\"ix_cm4\":4.58,\"iy_cm4\":4.58,\"sx_cm3\":1.7026022304832715,\"sy_cm3\":1.7026022304832715,\"rx_cm\":1.17,\"ry_cm\":1.17,\"j_cm4\":0.27,\"cw_cm6\":0.27},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":3.48,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":4.4,\"ix_cm4\":5.83,\"iy_cm4\":5.83,\"sx_cm3\":2.2251908396946565,\"sy_cm3\":2.2251908396946565,\"rx_cm\":1.14,\"ry_cm\":1.14,\"j_cm4\":0.62,\"cw_cm6\":0.61},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 3/4 \u00d7 1/8\",\"estado\":\"Revisado\",\"masa_kg_m\":2.14,\"d_ala_diametro_cm\":4.445,\"bf_b_cm\":4.445,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":2.74,\"ix_cm4\":5.41,\"iy_cm4\":5.41,\"sx_cm3\":1.677519379844961,\"sy_cm3\":1.677519379844961,\"rx_cm\":1.4,\"ry_cm\":1.4,\"j_cm4\":0.09,\"cw_cm6\":0.14},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 3/4 \u00d7 3/16\",\"estado\":\"Revisado\",\"masa_kg_m\":3.15,\"d_ala_diametro_cm\":4.445,\"bf_b_cm\":4.445,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":4.03,\"ix_cm4\":7.49,\"iy_cm4\":7.49,\"sx_cm3\":2.381558028616852,\"sy_cm3\":2.381558028616852,\"rx_cm\":1.37,\"ry_cm\":1.37,\"j_cm4\":0.31,\"cw_cm6\":0.45},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 1/8\",\"estado\":\"Con restricciones\",\"masa_kg_m\":2.46,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.3175,\"area_cm2\":3.1,\"ix_cm4\":7.91,\"iy_cm4\":7.91,\"sx_cm3\":2.1494565217391304,\"sy_cm3\":2.1494565217391304,\"rx_cm\":1.6,\"ry_cm\":1.6,\"j_cm4\":0.11,\"cw_cm6\":0.21},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 3/16\",\"estado\":\"Revisado\",\"masa_kg_m\":3.63,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":4.61,\"ix_cm4\":11.45,\"iy_cm4\":11.45,\"sx_cm3\":3.1542699724517904,\"sy_cm3\":3.1542699724517904,\"rx_cm\":1.57,\"ry_cm\":1.57,\"j_cm4\":0.36,\"cw_cm6\":0.68},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":4.75,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":6.06,\"ix_cm4\":14.57,\"iy_cm4\":14.57,\"sx_cm3\":4.069832402234637,\"sy_cm3\":4.069832402234637,\"rx_cm\":1.55,\"ry_cm\":1.55,\"j_cm4\":0.84,\"cw_cm6\":1.54},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 5/16\",\"estado\":\"Revisado\",\"masa_kg_m\":5.83,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":7.42,\"ix_cm4\":17.46,\"iy_cm4\":17.46,\"sx_cm3\":4.946175637393767,\"sy_cm3\":4.946175637393767,\"rx_cm\":1.52,\"ry_cm\":1.52,\"j_cm4\":1.63,\"cw_cm6\":2.85},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":6.99,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":8.77,\"ix_cm4\":19.98,\"iy_cm4\":19.98,\"sx_cm3\":5.791304347826086,\"sy_cm3\":5.791304347826086,\"rx_cm\":1.5,\"ry_cm\":1.5,\"j_cm4\":2.79,\"cw_cm6\":4.68},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 1/2 \u00d7 5/32\",\"estado\":\"Revisado\",\"masa_kg_m\":3.83,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":6.35,\"tw_cm\":null,\"tf_t_cm\":0.396875,\"area_cm2\":4.88,\"ix_cm4\":19.44,\"iy_cm4\":19.44,\"sx_cm3\":4.207792207792209,\"sy_cm3\":4.207792207792209,\"rx_cm\":1.98,\"ry_cm\":1.98,\"j_cm4\":0.26,\"cw_cm6\":0.81},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 1/2 \u00d7 3/16\",\"estado\":\"Revisado\",\"masa_kg_m\":4.61,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":6.35,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":5.81,\"ix_cm4\":22.89,\"iy_cm4\":22.89,\"sx_cm3\":4.97608695652174,\"sy_cm3\":4.97608695652174,\"rx_cm\":1.98,\"ry_cm\":1.98,\"j_cm4\":0.45,\"cw_cm6\":1.37},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 1/2 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":6.1,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":6.35,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":7.68,\"ix_cm4\":29.14,\"iy_cm4\":29.14,\"sx_cm3\":6.446902654867257,\"sy_cm3\":6.446902654867257,\"rx_cm\":1.96,\"ry_cm\":1.96,\"j_cm4\":1.06,\"cw_cm6\":3.12},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 1/2 \u00d7 5/16\",\"estado\":\"Revisado\",\"masa_kg_m\":7.44,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":6.35,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":9.48,\"ix_cm4\":35.38,\"iy_cm4\":35.38,\"sx_cm3\":7.914988814317675,\"sy_cm3\":7.914988814317675,\"rx_cm\":1.93,\"ry_cm\":1.93,\"j_cm4\":2.05,\"cw_cm6\":5.86},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 1/2 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":8.78,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":6.35,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":11.16,\"ix_cm4\":40.79,\"iy_cm4\":40.79,\"sx_cm3\":9.228506787330316,\"sy_cm3\":9.228506787330316,\"rx_cm\":1.91,\"ry_cm\":1.91,\"j_cm4\":3.52,\"cw_cm6\":9.73},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 3/16\",\"estado\":\"Revisado\",\"masa_kg_m\":5.52,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":0.47625,\"area_cm2\":7.03,\"ix_cm4\":40.01,\"iy_cm4\":40.01,\"sx_cm3\":7.222021660649819,\"sy_cm3\":7.222021660649819,\"rx_cm\":2.39,\"ry_cm\":2.39,\"j_cm4\":0.54,\"cw_cm6\":2.41},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":7.29,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":9.29,\"ix_cm4\":51.6,\"iy_cm4\":51.6,\"sx_cm3\":9.398907103825136,\"sy_cm3\":9.398907103825136,\"rx_cm\":2.36,\"ry_cm\":2.36,\"j_cm4\":1.27,\"cw_cm6\":5.54},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 5/16\",\"estado\":\"Revisado\",\"masa_kg_m\":9.08,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":11.48,\"ix_cm4\":62.9,\"iy_cm4\":62.9,\"sx_cm3\":11.626617375231053,\"sy_cm3\":11.626617375231053,\"rx_cm\":2.34,\"ry_cm\":2.34,\"j_cm4\":2.47,\"cw_cm6\":10.47},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":10.72,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":13.61,\"ix_cm4\":73.3,\"iy_cm4\":73.3,\"sx_cm3\":13.675373134328357,\"sy_cm3\":13.675373134328357,\"rx_cm\":2.31,\"ry_cm\":2.31,\"j_cm4\":4.25,\"cw_cm6\":17.5},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 1/2\",\"estado\":\"Revisado\",\"masa_kg_m\":13.99,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":1.27,\"area_cm2\":17.74,\"ix_cm4\":92.4,\"iy_cm4\":92.4,\"sx_cm3\":17.566539923954373,\"sy_cm3\":17.566539923954373,\"rx_cm\":2.29,\"ry_cm\":2.29,\"j_cm4\":9.97,\"cw_cm6\":38.78},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 1/2 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":8.63,\"d_ala_diametro_cm\":8.89,\"bf_b_cm\":8.89,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":10.9,\"ix_cm4\":83.66,\"iy_cm4\":83.66,\"sx_cm3\":13.010886469673403,\"sy_cm3\":13.010886469673403,\"rx_cm\":2.77,\"ry_cm\":2.77,\"j_cm4\":1.49,\"cw_cm6\":8.96},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 1/2 \u00d7 5/16\",\"estado\":\"Revisado\",\"masa_kg_m\":10.71,\"d_ala_diametro_cm\":8.89,\"bf_b_cm\":8.89,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":13.48,\"ix_cm4\":101.95,\"iy_cm4\":101.95,\"sx_cm3\":15.979623824451409,\"sy_cm3\":15.979623824451409,\"rx_cm\":2.74,\"ry_cm\":2.74,\"j_cm4\":2.9,\"cw_cm6\":17.02},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 1/2 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":12.65,\"d_ala_diametro_cm\":8.89,\"bf_b_cm\":8.89,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":16,\"ix_cm4\":119.46,\"iy_cm4\":119.46,\"sx_cm3\":18.901898734177212,\"sy_cm3\":18.901898734177212,\"rx_cm\":2.72,\"ry_cm\":2.72,\"j_cm4\":4.98,\"cw_cm6\":28.6},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 1/2 \u00d7 1/2\",\"estado\":\"Revisado\",\"masa_kg_m\":16.52,\"d_ala_diametro_cm\":8.89,\"bf_b_cm\":8.89,\"tw_cm\":null,\"tf_t_cm\":1.27,\"area_cm2\":20.97,\"ix_cm4\":151.51,\"iy_cm4\":151.51,\"sx_cm3\":24.43709677419354,\"sy_cm3\":24.43709677419354,\"rx_cm\":2.69,\"ry_cm\":2.69,\"j_cm4\":11.71,\"cw_cm6\":64.02},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 1/4\",\"estado\":\"Revisado\",\"masa_kg_m\":9.82,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":0.635,\"area_cm2\":12.52,\"ix_cm4\":124.9,\"iy_cm4\":124.9,\"sx_cm3\":16.901217861975642,\"sy_cm3\":16.901217861975642,\"rx_cm\":3.18,\"ry_cm\":3.18,\"j_cm4\":1.71,\"cw_cm6\":13.56},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 5/16\",\"estado\":\"Revisado\",\"masa_kg_m\":12.2,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":15.48,\"ix_cm4\":154.4,\"iy_cm4\":154.4,\"sx_cm3\":21.092896174863387,\"sy_cm3\":21.092896174863387,\"rx_cm\":3.15,\"ry_cm\":3.15,\"j_cm4\":3.32,\"cw_cm6\":25.86},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":14.58,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":18.45,\"ix_cm4\":181.5,\"iy_cm4\":181.5,\"sx_cm3\":24.965612104539204,\"sy_cm3\":24.965612104539204,\"rx_cm\":3.12,\"ry_cm\":3.12,\"j_cm4\":5.72,\"cw_cm6\":43.6},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 7/16\",\"estado\":\"Revisado\",\"masa_kg_m\":16.82,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":1.11125,\"area_cm2\":21.35,\"ix_cm4\":206.9,\"iy_cm4\":206.9,\"sx_cm3\":28.656509695290858,\"sy_cm3\":28.656509695290858,\"rx_cm\":3.12,\"ry_cm\":3.12,\"j_cm4\":9.04,\"cw_cm6\":67.54},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 1/2\",\"estado\":\"Revisado\",\"masa_kg_m\":19.05,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":1.27,\"area_cm2\":24.19,\"ix_cm4\":231.4,\"iy_cm4\":231.4,\"sx_cm3\":32.273361227336125,\"sy_cm3\":32.273361227336125,\"rx_cm\":3.1,\"ry_cm\":3.1,\"j_cm4\":13.44,\"cw_cm6\":98.34},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 4 \u00d7 5/8\",\"estado\":\"Revisado\",\"masa_kg_m\":23.36,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":10.16,\"tw_cm\":null,\"tf_t_cm\":1.5875,\"area_cm2\":29.74,\"ix_cm4\":277.2,\"iy_cm4\":277.2,\"sx_cm3\":39.375,\"sy_cm3\":39.375,\"rx_cm\":3.05,\"ry_cm\":3.05,\"j_cm4\":26.04,\"cw_cm6\":182.63},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 5 \u00d7 5/16\",\"estado\":\"Con restricciones\",\"masa_kg_m\":15.33,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":12.7,\"tw_cm\":null,\"tf_t_cm\":0.79375,\"area_cm2\":19.5,\"ix_cm4\":310.7,\"iy_cm4\":310.7,\"sx_cm3\":33.51672060409925,\"sy_cm3\":33.51672060409925,\"rx_cm\":3.96,\"ry_cm\":3.96,\"j_cm4\":4.27,\"cw_cm6\":52.93},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 5 \u00d7 3/8\",\"estado\":\"Revisado\",\"masa_kg_m\":18.3,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":12.7,\"tw_cm\":null,\"tf_t_cm\":0.9525,\"area_cm2\":23.29,\"ix_cm4\":363.8,\"iy_cm4\":363.8,\"sx_cm3\":39.67284623773173,\"sy_cm3\":39.67284623773173,\"rx_cm\":3.94,\"ry_cm\":3.94,\"j_cm4\":7.18,\"cw_cm6\":87.69},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 5 \u00d7 1/2\",\"estado\":\"Revisado\",\"masa_kg_m\":24.11,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":12.7,\"tw_cm\":null,\"tf_t_cm\":1.27,\"area_cm2\":30.65,\"ix_cm4\":468.3,\"iy_cm4\":468.3,\"sx_cm3\":51.63175303197354,\"sy_cm3\":51.63175303197354,\"rx_cm\":3.91,\"ry_cm\":3.91,\"j_cm4\":16.91,\"cw_cm6\":199.86},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 \u00d7 0.109\",\"estado\":\"Revisado\",\"masa_kg_m\":1.04,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":0.27686,\"area_cm2\":1.33,\"ix_cm4\":0.81,\"iy_cm4\":0.81,\"sx_cm3\":0.45,\"sy_cm3\":0.45,\"rx_cm\":0.78,\"ry_cm\":0.78,\"j_cm4\":0.035,\"cw_cm6\":0.016},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/4 \u00d7 0.109\",\"estado\":\"Revisado\",\"masa_kg_m\":1.32,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":0.27686,\"area_cm2\":1.68,\"ix_cm4\":1.62,\"iy_cm4\":1.62,\"sx_cm3\":0.7120879120879121,\"sy_cm3\":0.7120879120879121,\"rx_cm\":0.98,\"ry_cm\":0.98,\"j_cm4\":0.044,\"cw_cm6\":0.033},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 0.109\",\"estado\":\"Pendiente de datos\",\"masa_kg_m\":null,\"d_ala_diametro_cm\":null,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":null,\"ix_cm4\":null,\"iy_cm4\":null,\"sx_cm3\":null,\"sy_cm3\":null,\"rx_cm\":null,\"ry_cm\":null,\"j_cm4\":null,\"cw_cm6\":null},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 1 1/2 \u00d7 0.137\",\"estado\":\"Revisado\",\"masa_kg_m\":1.99,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":0.34798,\"area_cm2\":2.53,\"ix_cm4\":3.51,\"iy_cm4\":3.51,\"sx_cm3\":1.2857142857142856,\"sy_cm3\":1.2857142857142856,\"rx_cm\":1.18,\"ry_cm\":1.18,\"j_cm4\":0.105,\"cw_cm6\":0.113},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 2 \u00d7 0.232\",\"estado\":\"Revisado\",\"masa_kg_m\":4.43,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":0.58928,\"area_cm2\":5.64,\"ix_cm4\":13.6,\"iy_cm4\":13.6,\"sx_cm3\":3.788300835654596,\"sy_cm3\":3.788300835654596,\"rx_cm\":1.55,\"ry_cm\":1.55,\"j_cm4\":0.673,\"cw_cm6\":1.246},{\"familia\":\"LI\",\"perfil_comercial\":\"LI 3 \u00d7 0.281\",\"estado\":\"Revisado\",\"masa_kg_m\":8.14,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":7.62,\"tw_cm\":null,\"tf_t_cm\":0.71374,\"area_cm2\":10.37,\"ix_cm4\":57.36,\"iy_cm4\":57.36,\"sx_cm3\":10.524770642201835,\"sy_cm3\":10.524770642201835,\"rx_cm\":2.35,\"ry_cm\":2.35,\"j_cm4\":1.804,\"cw_cm6\":7.74},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 \u00d7 3/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":0.95,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":0.47625,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.209675,\"ix_cm4\":0.6503616025000001,\"iy_cm4\":0.02286427508789063,\"sx_cm3\":0.51209575,\"sy_cm3\":0.096017953125,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.13748153285077966,\"j_cm4\":0.08065484308535399,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.27,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.6129,\"ix_cm4\":0.8671488033333334,\"iy_cm4\":0.05419680020833334,\"sx_cm3\":0.6827943333333333,\"sy_cm3\":0.17069858333333332,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.18265433128025108,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.58,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.0161249999999997,\"ix_cm4\":1.0839360041666668,\"iy_cm4\":0.10585312540690105,\"sx_cm3\":0.8534929166666666,\"sy_cm3\":0.2667165364583333,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.22913588808463273,\"j_cm4\":0.34011941341802115,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.9,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.41935,\"ix_cm4\":1.3007232050000002,\"iy_cm4\":0.18291420070312503,\"sx_cm3\":1.0241915,\"sy_cm3\":0.3840718125,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.2749630657015593,\"j_cm4\":0.5590877376299159,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.53,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.2258,\"ix_cm4\":1.7342976066666669,\"iy_cm4\":0.4335744016666667,\"sx_cm3\":1.3655886666666666,\"sy_cm3\":0.6827943333333333,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.3666174209354124,\"j_cm4\":1.1908391925776043,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 3/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.19,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":0.47625,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.51209375,\"ix_cm4\":1.2702375048828125,\"iy_cm4\":0.028580343859863282,\"sx_cm3\":0.800149609375,\"sy_cm3\":0.12002244140625,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.13748153285077966,\"j_cm4\":0.1035184612275958,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.58,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.0161249999999997,\"ix_cm4\":1.6936500065104168,\"iy_cm4\":0.06774600026041667,\"sx_cm3\":1.0668661458333333,\"sy_cm3\":0.21337322916666665,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.23684456944163418,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.97,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.52015625,\"ix_cm4\":2.1170625081380208,\"iy_cm4\":0.1323164067586263,\"sx_cm3\":1.3335826822916665,\"sy_cm3\":0.33339567057291664,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.22913588808463273,\"j_cm4\":0.44593342597717534,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.37,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.0241875,\"ix_cm4\":2.540475009765625,\"iy_cm4\":0.22864275087890626,\"sx_cm3\":1.60029921875,\"sy_cm3\":0.480089765625,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.2749630657015593,\"j_cm4\":0.7418337602469454,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.16,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":4.0322499999999994,\"ix_cm4\":3.3873000130208335,\"iy_cm4\":0.5419680020833334,\"sx_cm3\":2.1337322916666666,\"sy_cm3\":0.8534929166666666,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.3666174209354124,\"j_cm4\":1.6227337102250132,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.96,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":5.0403125,\"ix_cm4\":4.2341250162760415,\"iy_cm4\":1.0585312540690104,\"sx_cm3\":2.667165364583333,\"sy_cm3\":1.3335826822916665,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.45827177616926545,\"j_cm4\":2.907322247503916,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/4 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":4.75,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.048375,\"ix_cm4\":5.08095001953125,\"iy_cm4\":1.82914200703125,\"sx_cm3\":3.2005984375,\"sy_cm3\":1.9203590625,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.5499261314031186,\"j_cm4\":4.580774470811767,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 3/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.42,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":0.47625,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.8145125,\"ix_cm4\":2.1949704084375,\"iy_cm4\":0.034296412631835936,\"sx_cm3\":1.1522154375,\"sy_cm3\":0.1440269296875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.13748153285077966,\"j_cm4\":0.1263825003434403,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.9,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.41935,\"ix_cm4\":2.9266272112500005,\"iy_cm4\":0.08129520031250001,\"sx_cm3\":1.53628725,\"sy_cm3\":0.256047875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.2910390125909807,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.37,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.0241875,\"ix_cm4\":3.6582840140625,\"iy_cm4\":0.15877968811035156,\"sx_cm3\":1.9203590625,\"sy_cm3\":0.4000748046875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.22913588808463273,\"j_cm4\":0.5517725022177139,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.85,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.629025,\"ix_cm4\":4.389940816875,\"iy_cm4\":0.2743713010546875,\"sx_cm3\":2.304430875,\"sy_cm3\":0.57610771875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.2749630657015593,\"j_cm4\":0.924687552106271,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.8,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":4.8387,\"ix_cm4\":5.853254422500001,\"iy_cm4\":0.6503616025000001,\"sx_cm3\":3.0725745,\"sy_cm3\":1.0241915,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.3666174209354124,\"j_cm4\":2.0557047047910495,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":4.75,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.048375,\"ix_cm4\":7.316568028125,\"iy_cm4\":1.2702375048828125,\"sx_cm3\":3.840718125,\"sy_cm3\":1.60029921875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.45827177616926545,\"j_cm4\":3.750550664170321,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.7,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":7.25805,\"ix_cm4\":8.77988163375,\"iy_cm4\":2.1949704084375,\"sx_cm3\":4.60886175,\"sy_cm3\":2.304430875,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.5499261314031186,\"j_cm4\":6.028623412424121,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 1 1/2 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.6,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":9.6774,\"ix_cm4\":11.706508845000002,\"iy_cm4\":5.202892820000001,\"sx_cm3\":6.145149,\"sy_cm3\":4.096766,\"rx_cm\":1.0998522628062373,\"ry_cm\":0.7332348418708248,\"j_cm4\":12.214593810508642,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 3/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.9,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":0.47625,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.41935,\"ix_cm4\":5.202892820000001,\"iy_cm4\":0.04572855017578126,\"sx_cm3\":2.048383,\"sy_cm3\":0.19203590625,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.13748153285077966,\"j_cm4\":0.17211090026864795,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.53,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.2258,\"ix_cm4\":6.9371904266666675,\"iy_cm4\":0.10839360041666668,\"sx_cm3\":2.731177333333333,\"sy_cm3\":0.34139716666666664,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.3994311121965522,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.16,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":4.0322499999999994,\"ix_cm4\":8.671488033333334,\"iy_cm4\":0.2117062508138021,\"sx_cm3\":3.4139716666666664,\"sy_cm3\":0.5334330729166666,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.22913588808463273,\"j_cm4\":0.7634698075002958,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.8,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":4.8387,\"ix_cm4\":10.405785640000001,\"iy_cm4\":0.36582840140625006,\"sx_cm3\":4.096766,\"sy_cm3\":0.768143625,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.2749630657015593,\"j_cm4\":1.2904774893656639,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.06,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.4516,\"ix_cm4\":13.874380853333335,\"iy_cm4\":0.8671488033333334,\"sx_cm3\":5.462354666666666,\"sy_cm3\":1.3655886666666666,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.3666174209354124,\"j_cm4\":2.922469300484017,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":6.33,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":8.064499999999999,\"ix_cm4\":17.34297606666667,\"iy_cm4\":1.6936500065104168,\"sx_cm3\":6.827943333333333,\"sy_cm3\":2.1337322916666666,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.45827177616926545,\"j_cm4\":5.4419106146883385,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.6,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":9.6774,\"ix_cm4\":20.811571280000003,\"iy_cm4\":2.9266272112500005,\"sx_cm3\":8.193532,\"sy_cm3\":3.0725745,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.5499261314031186,\"j_cm4\":8.945403802078655,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":10.13,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.9032,\"ix_cm4\":27.74876170666667,\"iy_cm4\":6.9371904266666675,\"sx_cm3\":10.924709333333332,\"sy_cm3\":5.462354666666666,\"rx_cm\":1.4664696837416495,\"ry_cm\":0.7332348418708248,\"j_cm4\":19.05342708124167,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.95,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":5.0403125,\"ix_cm4\":16.936500065104166,\"iy_cm4\":0.2646328135172526,\"sx_cm3\":5.334330729166666,\"sy_cm3\":0.6667913411458333,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.22913588808463273,\"j_cm4\":0.9751736137611134,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":4.75,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.048375,\"ix_cm4\":20.323800078125,\"iy_cm4\":0.4572855017578125,\"sx_cm3\":6.401196875,\"sy_cm3\":0.96017953125,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.2749630657015593,\"j_cm4\":1.6562953796415327,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":6.33,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":8.064499999999999,\"ix_cm4\":27.098400104166668,\"iy_cm4\":1.0839360041666668,\"sx_cm3\":8.534929166666666,\"sy_cm3\":1.7069858333333332,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.3666174209354124,\"j_cm4\":3.789513111066147,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.91,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":10.080625,\"ix_cm4\":33.87300013020833,\"iy_cm4\":2.1170625081380208,\"sx_cm3\":10.668661458333332,\"sy_cm3\":2.667165364583333,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.45827177616926545,\"j_cm4\":7.134934815634805,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":9.49,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.09675,\"ix_cm4\":40.64760015625,\"iy_cm4\":3.6582840140625,\"sx_cm3\":12.80239375,\"sy_cm3\":3.840718125,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.5499261314031186,\"j_cm4\":11.869340163951126,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":12.66,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":16.128999999999998,\"ix_cm4\":54.196800208333336,\"iy_cm4\":8.671488033333334,\"sx_cm3\":17.069858333333332,\"sy_cm3\":6.827943333333333,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.7332348418708248,\"j_cm4\":25.96373936360021,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 2 1/2 \u00d7 1 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.82,\"d_ala_diametro_cm\":6.35,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":20.16125,\"ix_cm4\":67.74600026041666,\"iy_cm4\":16.936500065104166,\"sx_cm3\":21.337322916666665,\"sy_cm3\":10.668661458333332,\"rx_cm\":1.8330871046770618,\"ry_cm\":0.9165435523385309,\"j_cm4\":46.51715596006266,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.8,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":4.8387,\"ix_cm4\":23.413017690000004,\"iy_cm4\":0.16259040062500002,\"sx_cm3\":6.145149,\"sy_cm3\":0.51209575,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.6162177555857645,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":4.74,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.048375,\"ix_cm4\":29.2662721125,\"iy_cm4\":0.3175593762207031,\"sx_cm3\":7.68143625,\"sy_cm3\":0.800149609375,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.22913588808463273,\"j_cm4\":1.186878986502018,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.7,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":7.25805,\"ix_cm4\":35.119526535,\"iy_cm4\":0.548742602109375,\"sx_cm3\":9.2177235,\"sy_cm3\":1.1522154375,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.2749630657015593,\"j_cm4\":2.022120005495045,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.6,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":9.6774,\"ix_cm4\":46.82603538000001,\"iy_cm4\":1.3007232050000002,\"sx_cm3\":12.290298,\"sy_cm3\":2.048383,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.3666174209354124,\"j_cm4\":4.656624201455691,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":9.49,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.09675,\"ix_cm4\":58.532544225,\"iy_cm4\":2.540475009765625,\"sx_cm3\":15.3628725,\"sy_cm3\":3.2005984375,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.45827177616926545,\"j_cm4\":8.828360035483422,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":11.39,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":14.5161,\"ix_cm4\":70.23905307,\"iy_cm4\":4.389940816875,\"sx_cm3\":18.435447,\"sy_cm3\":4.60886175,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.5499261314031186,\"j_cm4\":14.795000833700335,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 3 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.19,\"d_ala_diametro_cm\":7.62,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":19.3548,\"ix_cm4\":93.65207076000002,\"iy_cm4\":10.405785640000001,\"sx_cm3\":24.580596,\"sy_cm3\":8.193532,\"rx_cm\":2.1997045256124745,\"ry_cm\":0.7332348418708248,\"j_cm4\":32.89127527665679,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.06,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.4516,\"ix_cm4\":55.49752341333334,\"iy_cm4\":0.21678720083333336,\"sx_cm3\":10.924709333333332,\"sy_cm3\":0.6827943333333333,\"rx_cm\":2.932939367483299,\"ry_cm\":0.1833087104677062,\"j_cm4\":0.8330048626184045,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":6.32,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":8.064499999999999,\"ix_cm4\":69.37190426666668,\"iy_cm4\":0.4234125016276042,\"sx_cm3\":13.655886666666666,\"sy_cm3\":1.0668661458333333,\"rx_cm\":2.932939367483299,\"ry_cm\":0.22913588808463273,\"j_cm4\":1.610290929033921,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.6,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":9.6774,\"ix_cm4\":83.24628512000001,\"iy_cm4\":0.7316568028125001,\"sx_cm3\":16.387064,\"sy_cm3\":1.53628725,\"rx_cm\":2.932939367483299,\"ry_cm\":0.2749630657015593,\"j_cm4\":2.753774404298367,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":10.13,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.9032,\"ix_cm4\":110.99504682666668,\"iy_cm4\":1.7342976066666669,\"sx_cm3\":21.849418666666665,\"sy_cm3\":2.731177333333333,\"rx_cm\":2.932939367483299,\"ry_cm\":0.3666174209354124,\"j_cm4\":6.390897795144835,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":12.66,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":16.128999999999998,\"ix_cm4\":138.74380853333335,\"iy_cm4\":3.3873000130208335,\"sx_cm3\":27.31177333333333,\"sy_cm3\":4.267464583333333,\"rx_cm\":2.932939367483299,\"ry_cm\":0.45827177616926545,\"j_cm4\":12.215516920004733,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.19,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":19.3548,\"ix_cm4\":166.49257024000002,\"iy_cm4\":5.853254422500001,\"sx_cm3\":32.774128,\"sy_cm3\":6.145149,\"rx_cm\":2.932939367483299,\"ry_cm\":0.5499261314031186,\"j_cm4\":20.647639829850622,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 4 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":20.26,\"d_ala_diametro_cm\":10.16,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":25.8064,\"ix_cm4\":221.99009365333336,\"iy_cm4\":13.874380853333335,\"sx_cm3\":43.69883733333333,\"sy_cm3\":10.924709333333332,\"rx_cm\":2.932939367483299,\"ry_cm\":0.7332348418708248,\"j_cm4\":46.759508807744275,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":6.33,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":8.064499999999999,\"ix_cm4\":108.39360041666667,\"iy_cm4\":0.2709840010416667,\"sx_cm3\":17.069858333333332,\"sy_cm3\":0.8534929166666666,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.1833087104677062,\"j_cm4\":1.0497920378187418,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.9,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":10.080625,\"ix_cm4\":135.49200052083333,\"iy_cm4\":0.5292656270345052,\"sx_cm3\":21.337322916666665,\"sy_cm3\":1.3335826822916665,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.22913588808463273,\"j_cm4\":2.0337032778769633,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":9.49,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.09675,\"ix_cm4\":162.590400625,\"iy_cm4\":0.914571003515625,\"sx_cm3\":25.6047875,\"sy_cm3\":1.9203590625,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.2749630657015593,\"j_cm4\":3.4854305501652174,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":12.66,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":16.128999999999998,\"ix_cm4\":216.78720083333334,\"iy_cm4\":2.1678720083333336,\"sx_cm3\":34.139716666666665,\"sy_cm3\":3.4139716666666664,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.3666174209354124,\"j_cm4\":8.12518883976455,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.82,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":20.16125,\"ix_cm4\":270.98400104166666,\"iy_cm4\":4.2341250162760415,\"sx_cm3\":42.67464583333333,\"sy_cm3\":5.334330729166666,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.45827177616926545,\"j_cm4\":15.602777820177815,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":18.99,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":24.1935,\"ix_cm4\":325.18080125,\"iy_cm4\":7.316568028125,\"sx_cm3\":51.209575,\"sy_cm3\":7.68143625,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.5499261314031186,\"j_cm4\":26.500726074264524,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 5 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":25.32,\"d_ala_diametro_cm\":12.7,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":32.257999999999996,\"ix_cm4\":433.5744016666667,\"iy_cm4\":17.34297606666667,\"sx_cm3\":68.27943333333333,\"sy_cm3\":13.655886666666666,\"rx_cm\":3.6661742093541236,\"ry_cm\":0.7332348418708248,\"j_cm4\":60.63220977705835,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.6,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":0.635,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":9.6774,\"ix_cm4\":187.30414152000003,\"iy_cm4\":0.32518080125000004,\"sx_cm3\":24.580596,\"sy_cm3\":1.0241915,\"rx_cm\":4.399409051224949,\"ry_cm\":0.1833087104677062,\"j_cm4\":1.2665792294448135,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 5/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":9.48,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":0.79375,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":12.09675,\"ix_cm4\":234.1301769,\"iy_cm4\":0.6351187524414063,\"sx_cm3\":30.725745,\"sy_cm3\":1.60029921875,\"rx_cm\":4.399409051224949,\"ry_cm\":0.22913588808463273,\"j_cm4\":2.4571157246250115,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 3/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":11.39,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":0.9525,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":14.5161,\"ix_cm4\":280.95621228,\"iy_cm4\":1.09748520421875,\"sx_cm3\":36.870894,\"sy_cm3\":2.304430875,\"rx_cm\":4.399409051224949,\"ry_cm\":0.2749630657015593,\"j_cm4\":4.217087117005672,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.19,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":19.3548,\"ix_cm4\":374.60828304000006,\"iy_cm4\":2.6014464100000003,\"sx_cm3\":49.161192,\"sy_cm3\":4.096766,\"rx_cm\":4.399409051224949,\"ry_cm\":0.3666174209354124,\"j_cm4\":9.859484089372232,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":18.99,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":24.1935,\"ix_cm4\":468.2603538,\"iy_cm4\":5.08095001953125,\"sx_cm3\":61.45149,\"sy_cm3\":6.401196875,\"rx_cm\":4.399409051224949,\"ry_cm\":0.45827177616926545,\"j_cm4\":18.990063784032287,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":22.79,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":29.0322,\"ix_cm4\":561.91242456,\"iy_cm4\":8.77988163375,\"sx_cm3\":73.741788,\"sy_cm3\":9.2177235,\"rx_cm\":4.399409051224949,\"ry_cm\":0.5499261314031186,\"j_cm4\":32.35392008792072,\"cw_cm6\":null},{\"familia\":\"SOL\",\"perfil_comercial\":\"SOL 6 \u00d7 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":30.38,\"d_ala_diametro_cm\":15.24,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":38.7096,\"ix_cm4\":749.2165660800001,\"iy_cm4\":20.811571280000003,\"sx_cm3\":98.322384,\"sy_cm3\":16.387064,\"rx_cm\":4.399409051224949,\"ry_cm\":0.7332348418708248,\"j_cm4\":74.50598722329106,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":0.994,\"d_ala_diametro_cm\":1.27,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.2667686977437442,\"ix_cm4\":0.12769820203693033,\"iy_cm4\":0.12769820203693033,\"sx_cm3\":0.20109953076681944,\"sy_cm3\":0.20109953076681944,\"rx_cm\":0.3175,\"ry_cm\":0.3175,\"j_cm4\":0.25539640407386066,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 9/16\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.257,\"d_ala_diametro_cm\":1.42875,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.6032541330819265,\"ix_cm4\":0.2045478280186279,\"iy_cm4\":0.2045478280186279,\"sx_cm3\":0.28633116783010026,\"sy_cm3\":0.28633116783010026,\"rx_cm\":0.3571875,\"ry_cm\":0.3571875,\"j_cm4\":0.4090956560372558,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.552,\"d_ala_diametro_cm\":1.5875,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.9793260902246004,\"ix_cm4\":0.31176318856672436,\"iy_cm4\":0.31176318856672436,\"sx_cm3\":0.3927725210289441,\"sy_cm3\":0.3927725210289441,\"rx_cm\":0.396875,\"ry_cm\":0.396875,\"j_cm4\":0.6235263771334487,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.235,\"d_ala_diametro_cm\":1.905,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.8502295699234246,\"ix_cm4\":0.6464721478119598,\"iy_cm4\":0.6464721478119598,\"sx_cm3\":0.6787109163380155,\"sy_cm3\":0.6787109163380155,\"rx_cm\":0.47625,\"ry_cm\":0.47625,\"j_cm4\":1.2929442956239197,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 7/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.045,\"d_ala_diametro_cm\":2.2225,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.8794791368402177,\"ix_cm4\":1.1976694651979287,\"iy_cm4\":1.1976694651979287,\"sx_cm3\":1.077767797703423,\"sy_cm3\":1.077767797703423,\"rx_cm\":0.555625,\"ry_cm\":0.555625,\"j_cm4\":2.3953389303958574,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":3.973,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":5.067074790974977,\"ix_cm4\":2.0431712325908853,\"iy_cm4\":2.0431712325908853,\"sx_cm3\":1.6087962461345555,\"sy_cm3\":1.6087962461345555,\"rx_cm\":0.635,\"ry_cm\":0.635,\"j_cm4\":4.086342465181771,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1 1/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.022,\"d_ala_diametro_cm\":2.8575,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.413016532327706,\"ix_cm4\":3.2727652482980463,\"iy_cm4\":3.2727652482980463,\"sx_cm3\":2.290649342640802,\"sy_cm3\":2.290649342640802,\"rx_cm\":0.714375,\"ry_cm\":0.714375,\"j_cm4\":6.5455304965960925,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":6.208,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":7.9173043608984015,\"ix_cm4\":4.98821101706759,\"iy_cm4\":4.98821101706759,\"sx_cm3\":3.142180168231553,\"sy_cm3\":3.142180168231553,\"rx_cm\":0.79375,\"ry_cm\":0.79375,\"j_cm4\":9.97642203413518,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":9,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":11.400918279693698,\"ix_cm4\":10.343554364991357,\"iy_cm4\":10.343554364991357,\"sx_cm3\":5.429687330704124,\"sy_cm3\":5.429687330704124,\"rx_cm\":0.9525,\"ry_cm\":0.9525,\"j_cm4\":20.687108729982715,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":10.49,\"d_ala_diametro_cm\":4.1275,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":13.380244369918303,\"ix_cm4\":14.246829485846753,\"iy_cm4\":14.246829485846753,\"sx_cm3\":6.903369829604725,\"sy_cm3\":6.903369829604725,\"rx_cm\":1.031875,\"ry_cm\":1.031875,\"j_cm4\":28.493658971693506,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 1 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":12.17,\"d_ala_diametro_cm\":4.445,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":15.517916547360871,\"ix_cm4\":19.16271144316686,\"iy_cm4\":19.16271144316686,\"sx_cm3\":8.622142381627384,\"sy_cm3\":8.622142381627384,\"rx_cm\":1.11125,\"ry_cm\":1.11125,\"j_cm4\":38.32542288633372,\"cw_cm6\":null},{\"familia\":\"OS\",\"perfil_comercial\":\"OS 2\",\"estado\":\"Recalculado\",\"masa_kg_m\":15.89,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":null,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":20.268299163899908,\"ix_cm4\":32.690739721454165,\"iy_cm4\":32.690739721454165,\"sx_cm3\":12.870369969076444,\"sy_cm3\":12.870369969076444,\"rx_cm\":1.27,\"ry_cm\":1.27,\"j_cm4\":65.38147944290833,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.266,\"d_ala_diametro_cm\":1.27,\"bf_b_cm\":1.27,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.6129,\"ix_cm4\":0.21678720083333336,\"iy_cm4\":0.21678720083333336,\"sx_cm3\":0.3413971666666667,\"sy_cm3\":0.3413971666666667,\"rx_cm\":0.3666174209354124,\"ry_cm\":0.3666174209354124,\"j_cm4\":0.3663703694083334,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 5/8\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.978,\"d_ala_diametro_cm\":1.5875,\"bf_b_cm\":1.5875,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":2.52015625,\"ix_cm4\":0.5292656270345051,\"iy_cm4\":0.5292656270345051,\"sx_cm3\":0.6667913411458333,\"sy_cm3\":0.6667913411458333,\"rx_cm\":0.45827177616926545,\"ry_cm\":0.45827177616926545,\"j_cm4\":0.8944589096883138,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 3/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":2.848,\"d_ala_diametro_cm\":1.905,\"bf_b_cm\":1.905,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":3.629025,\"ix_cm4\":1.09748520421875,\"iy_cm4\":1.09748520421875,\"sx_cm3\":1.1522154375,\"sy_cm3\":1.1522154375,\"rx_cm\":0.5499261314031186,\"ry_cm\":0.5499261314031186,\"j_cm4\":1.8547499951296875,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 1\",\"estado\":\"Recalculado\",\"masa_kg_m\":5.063,\"d_ala_diametro_cm\":2.54,\"bf_b_cm\":2.54,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":6.4516,\"ix_cm4\":3.4685952133333338,\"iy_cm4\":3.4685952133333338,\"sx_cm3\":2.7311773333333336,\"sy_cm3\":2.7311773333333336,\"rx_cm\":0.7332348418708248,\"ry_cm\":0.7332348418708248,\"j_cm4\":5.861925910533334,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 1 1/4\",\"estado\":\"Recalculado\",\"masa_kg_m\":7.91,\"d_ala_diametro_cm\":3.175,\"bf_b_cm\":3.175,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":10.080625,\"ix_cm4\":8.468250032552081,\"iy_cm4\":8.468250032552081,\"sx_cm3\":5.334330729166666,\"sy_cm3\":5.334330729166666,\"rx_cm\":0.9165435523385309,\"ry_cm\":0.9165435523385309,\"j_cm4\":14.31134255501302,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 1 1/2\",\"estado\":\"Recalculado\",\"masa_kg_m\":11.391,\"d_ala_diametro_cm\":3.81,\"bf_b_cm\":3.81,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":14.5161,\"ix_cm4\":17.5597632675,\"iy_cm4\":17.5597632675,\"sx_cm3\":9.2177235,\"sy_cm3\":9.2177235,\"rx_cm\":1.0998522628062373,\"ry_cm\":1.0998522628062373,\"j_cm4\":29.675999922075,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 2\",\"estado\":\"Recalculado\",\"masa_kg_m\":20.25,\"d_ala_diametro_cm\":5.08,\"bf_b_cm\":5.08,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":25.8064,\"ix_cm4\":55.49752341333334,\"iy_cm4\":55.49752341333334,\"sx_cm3\":21.84941866666667,\"sy_cm3\":21.84941866666667,\"rx_cm\":1.4664696837416495,\"ry_cm\":1.4664696837416495,\"j_cm4\":93.79081456853335,\"cw_cm6\":null},{\"familia\":\"CS\",\"perfil_comercial\":\"CS 0.472\",\"estado\":\"Recalculado\",\"masa_kg_m\":1.128,\"d_ala_diametro_cm\":1.19888,\"bf_b_cm\":1.19888,\"tw_cm\":null,\"tf_t_cm\":null,\"area_cm2\":1.4373132543999998,\"ix_cm4\":0.1721557826061599,\"iy_cm4\":0.1721557826061599,\"sx_cm3\":0.2871943524058453,\"sy_cm3\":0.2871943524058453,\"rx_cm\":0.34608684536302925,\"ry_cm\":0.34608684536302925,\"j_cm4\":0.2909432726044103,\"cw_cm6\":null}]"


## Bloque 3. Base de perfiles y lista ordenada (perfiles_gerdau.py)

In [ ]:
%%writefile perfiles_gerdau.py
"""Base de perfiles Gerdau rev04 (datos en gerdau_datos.py)."""
import json
from gerdau_datos import _DATOS_TEXTO

_DATOS = json.loads(_DATOS_TEXTO)
PERFILES = {p['perfil_comercial']: p for p in _DATOS}

FAMILIA_ORDEN = ['IR', 'CE', 'LI', 'SOL', 'OS', 'CS']


def obtener_perfil(nombre):
    """Dimensiones en cm: H = d (peralte o diametro), B = bf, e = tw, t = tf. None = dato pendiente."""
    p = PERFILES.get(nombre)
    if p is None:
        return None
    return {'familia': p['familia'], 'H': p['d_ala_diametro_cm'], 'B': p['bf_b_cm'],
            'e': p['tw_cm'], 't': p['tf_t_cm'], 'A': p['area_cm2'], 'Ix': p['ix_cm4'], 'Iy': p['iy_cm4'],
            'Sx': p['sx_cm3'], 'Sy': p['sy_cm3'], 'rx': p['rx_cm'], 'ry': p['ry_cm'], 'J': p['j_cm4'],
            'Cw': p['cw_cm6'], 'estado': p['estado'], 'masa': p['masa_kg_m']}


def _clave_seccion(p):
    """Familia, luego peralte (o diametro/lado), masa (mas liviano primero), ancho. Nulos al final."""
    def k(v):
        return (v is None, v or 0)
    return (FAMILIA_ORDEN.index(p['familia']),
            k(p['d_ala_diametro_cm']), k(p['masa_kg_m']), k(p['bf_b_cm']),
            p['perfil_comercial'])


def lista_perfiles_ordenada():
    """[(etiqueta, nombre)] para la lista desplegable. Muestra masa y estado si no es Revisado."""
    out = []
    for p in sorted(_DATOS, key=_clave_seccion):
        masa = '' if p['masa_kg_m'] is None else f"   {p['masa_kg_m']:g} kg/m"
        estado = '' if p['estado'] == 'Revisado' else f"   [{p['estado']}]"
        out.append((f"{p['perfil_comercial']}{masa}{estado}", p['perfil_comercial']))
    return out


## Bloque 4. Orientación y secciones (orientacion_gerdau.py)

In [ ]:
%%writefile orientacion_gerdau.py
"""
ORIENTACION DEL EJE FUERTE (peralte) EN PANTALLA, con la base Gerdau Mexico rev04.

Convencion del motor (copiada de motor_analisis.py del otro chat, sin depender de ese paquete):
  xAxis = (p2 - p1)/L ; yAxis = vecxz x xAxis ; zAxis = xAxis x yAxis.
  El eje fuerte (Iz) es el eje local y => su direccion es el PERALTE d del perfil.

Defaults del motor (sin marca):
  - vigas/diagonales: peralte VERTICAL (eje fuerte resiste gravedad)
  - columnas:         peralte en Y global (vecxz = (1,0,0))
"""
import math
import numpy as np
import perfiles_gerdau as G


def vector_vecxz(p1, p2, elemento):
    """vecxz por defecto segun el tipo de elemento (igual que el motor)."""
    v = np.array(p2, float) - np.array(p1, float)
    L = np.linalg.norm(v)
    x = v / L
    if abs(x[2]) > 0.9:                       # columna (casi vertical)
        return (1.0, 0.0, 0.0)
    vx = np.cross(x, np.array([0.0, 0.0, 1.0]))   # viga/diagonal: yAxis vertical
    n = np.linalg.norm(vx)
    if n < 1e-6:
        return (1.0, 0.0, 0.0)
    vx = vx / n
    return (float(vx[0]), float(vx[1]), float(vx[2]))


def vecxz_desde_direccion(p1, p2, s):
    """vecxz que hace que el eje local y (= peralte) apunte segun s (proyectado perpendicular a la barra)."""
    x = np.array(p2, float) - np.array(p1, float); x = x / np.linalg.norm(x)
    s = np.array(s, float)
    u = s - x * np.dot(s, x)
    nu = np.linalg.norm(u)
    if nu < 1e-6:
        return None
    u = u / nu
    v = np.cross(x, u)
    return (float(v[0]), float(v[1]), float(v[2]))


def vecxz_de_barra(nodos, b):
    """vecxz de una barra: usa 'eje_fuerte' si existe, si no el default por tipo de elemento."""
    p1 = nodos[b['n1']]; p2 = nodos[b['n2']]
    s = b.get('eje_fuerte')
    if s:
        v = vecxz_desde_direccion(p1, p2, s)
        if v is not None:
            return v
    return vector_vecxz(p1, p2, b.get('elemento'))


def ejes_locales(dib, i):
    """Ejes locales (x, y, z) de la barra i."""
    b = dib.barras[i]
    p1, p2 = dib.nodos[b['n1']], dib.nodos[b['n2']]
    vecxz = np.array(vecxz_de_barra(dib.nodos, b), float)
    x = np.array(p2, float) - np.array(p1, float); x = x / np.linalg.norm(x)
    y = np.cross(vecxz, x); y = y / np.linalg.norm(y)
    z = np.cross(x, y)
    return x, y, z


def dimensiones(perfil):
    """(d, bf) en metros desde Gerdau (H = d, B = bf en cm). None si el perfil no existe o no tiene dato."""
    p = G.obtener_perfil(perfil)
    if p is None or p['H'] is None or p['B'] is None:
        return None
    return float(p['H']) / 100.0, float(p['B']) / 100.0


def contorno(dib, i, escala=1.0):
    """4 vertices globales del rectangulo del perfil en el punto medio de la barra i.
    Alto = d a lo largo del eje local y (eje fuerte); ancho = bf a lo largo de z."""
    b = dib.barras[i]
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    pm = (p1 + p2) / 2
    dims = dimensiones(b['perfil'])
    if dims is None:
        return None
    d, bf = dims[0] * escala, dims[1] * escala
    x, y, z = ejes_locales(dib, i)
    c = [pm + sy * (d / 2) * y + sz * (bf / 2) * z for sy, sz in [(1, 1), (-1, 1), (-1, -1), (1, -1)]]
    return [tuple(float(v) for v in q) for q in c]


PRESETS = {'peralte vertical (Z)': (0.0, 0.0, 1.0),
           'peralte en X': (1.0, 0.0, 0.0),
           'peralte en Y': (0.0, 1.0, 0.0)}


def orientar(dib, i, preset):
    """Fija el peralte (eje fuerte) de la barra i con un preset o un vector. Rechaza direccion paralela a la barra."""
    b = dib.barras[i]
    s = np.array(PRESETS[preset] if isinstance(preset, str) else preset, float)
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    x = (p2 - p1) / np.linalg.norm(p2 - p1)
    if abs(np.dot(s, x)) > 0.999:
        raise ValueError(f'el peralte no puede ser paralelo a la barra {i}; elija otra direccion')
    dib.eje_fuerte(i, tuple(s))
    return b['eje_fuerte']


def ortogonalidad(dib, i):
    """Angulo (grados) entre el eje fuerte asignado y la barra; 90 = perpendicular (correcto)."""
    b = dib.barras[i]
    if b['eje_fuerte'] is None:
        return None
    p1, p2 = np.array(dib.nodos[b['n1']], float), np.array(dib.nodos[b['n2']], float)
    x = (p2 - p1) / np.linalg.norm(p2 - p1)
    s = np.array(b['eje_fuerte'], float)
    return math.degrees(math.acos(max(-1.0, min(1.0, abs(np.dot(s, x))))))


In [ ]:
%%writefile vista3d_plotly.py
"""
FIGURA 3D INTERACTIVA (plotly): gira, acerca y muestra datos al pasar el cursor. Funciona en Kaggle porque
plotly embebe su propio motor en la salida; no requiere widgets de matplotlib.

Contenido:
  - barras por tipo de elemento (color), con ID, perfil, conexiones y eje fuerte al pasar el cursor
  - seccion envolvente de cada perfil (caja de d x bf) en la barra, ampliada ESCALA_SECCION
  - flecha = eje fuerte (direccion del peralte) en el punto medio de cada barra
  - circulo abierto = extremo rotulado; cuadrado = apoyo empotrado; rombo = apoyo rotulado
"""
import numpy as np
import plotly.graph_objects as go
from orientacion_gerdau import contorno, ejes_locales, dimensiones

COLOR_ELEMENTO = {'COLUMNA': '#1f77b4', 'VIGA': '#2ca02c', 'ARRIOSTRE': '#d62728', 'DIAGONAL': '#ff7f0e'}


def figura_3d(dib, id_de_indice, escala_seccion=2.5, titulo=None):
    """Devuelve una go.Figure 3D del modelo. escala_seccion solo amplia la seccion dibujada, no el calculo."""
    fig = go.Figure()
    nodos = np.array(dib.nodos, float)
    id_barra = getattr(dib, 'id_barra_de_indice', {}) or {}

    def rotulada(b):
        return b['conexion_i'] == 'rotulado' or b['conexion_j'] == 'rotulado'

    def texto_barra(i, b):
        return (f"Barra ID {id_barra.get(i, i)} (nodos N{id_de_indice.get(b['n1'], '?')}-N{id_de_indice.get(b['n2'], '?')})"
                f"<br>{b['elemento']} {b['perfil']}"
                f"<br>conexion I: {b['conexion_i']}, J: {b['conexion_j']}"
                f"<br>eje fuerte: {'por defecto' if b['eje_fuerte'] is None else np.round(b['eje_fuerte'], 3).tolist()}")

    # --- barras con extremo rotulado: linea discontinua morada (se ven distintas de las continuas) ---
    xs, ys, zs, textos = [], [], [], []
    for i, b in enumerate(dib.barras):
        if not rotulada(b):
            continue
        p1, p2 = nodos[b['n1']], nodos[b['n2']]
        info = texto_barra(i, b)
        xs += [p1[0], p2[0], None]; ys += [p1[1], p2[1], None]; zs += [p1[2], p2[2], None]
        textos += [info, info, None]
    if xs:
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                                   line=dict(color='#6a1b9a', width=9, dash='dash'),
                                   name='barra con extremo rotulado (linea discontinua)',
                                   text=textos, hoverinfo='text'))

    # --- barras continuas por tipo de elemento (una traza por tipo, con huecos entre barras) ---
    for elem, color in COLOR_ELEMENTO.items():
        xs, ys, zs, textos = [], [], [], []
        for i, b in enumerate(dib.barras):
            if b['elemento'] != elem or rotulada(b):
                continue
            p1, p2 = nodos[b['n1']], nodos[b['n2']]
            info = texto_barra(i, b)
            xs += [p1[0], p2[0], None]; ys += [p1[1], p2[1], None]; zs += [p1[2], p2[2], None]
            textos += [info, info, None]
        if xs:
            fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', line=dict(color=color, width=7),
                                       name=elem, text=textos, hoverinfo='text'))

    # --- secciones envolventes (caja d x bf en ambos extremos) y eje fuerte ---
    for i, b in enumerate(dib.barras):
        caja = contorno(dib, i, escala_seccion)
        if not caja:
            continue
        p1, p2 = nodos[b['n1']], nodos[b['n2']]
        x_loc, y_loc, z_loc = ejes_locales(dib, i)
        d, bf = dimensiones(b['perfil'])
        d *= escala_seccion; bf *= escala_seccion
        vertices = []
        for p in (p1, p2):
            for sy, sz in [(1, 1), (-1, 1), (-1, -1), (1, -1)]:
                q = p + sy * (d / 2) * y_loc + sz * (bf / 2) * z_loc
                vertices.append(q)
        vertices = np.array(vertices)
        fig.add_trace(go.Mesh3d(x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2], alphahull=0,
                                color=COLOR_ELEMENTO[b['elemento']], opacity=0.12 if rotulada(b) else 0.35, flatshading=True,
                                name=f'seccion {b["perfil"]}', showlegend=False, hoverinfo='skip'))
        pm = (p1 + p2) / 2
        largo = 0.6 * d
        tip = pm + y_loc * largo
        fig.add_trace(go.Scatter3d(x=[pm[0], tip[0]], y=[pm[1], tip[1]], z=[pm[2], tip[2]], mode='lines',
                                   line=dict(color='black', width=4), showlegend=False, hoverinfo='skip'))
        fig.add_trace(go.Cone(x=[tip[0]], y=[tip[1]], z=[tip[2]], u=[y_loc[0]], v=[y_loc[1]], w=[y_loc[2]],
                              anchor='tip', sizemode='absolute', sizeref=0.12,
                              colorscale=[[0, 'black'], [1, 'black']], showscale=False, hoverinfo='skip',
                              showlegend=False))

    # --- extremos rotulados (circulo abierto) ---
    xr, yr, zr = [], [], []
    for b in dib.barras:
        for idx, c in [(b['n1'], b['conexion_i']), (b['n2'], b['conexion_j'])]:
            if c == 'rotulado':
                xr.append(nodos[idx][0]); yr.append(nodos[idx][1]); zr.append(nodos[idx][2])
    if xr:
        fig.add_trace(go.Scatter3d(x=xr, y=yr, z=zr, mode='markers',
                                   marker=dict(symbol='circle-open', size=9, color='black', line=dict(width=3)),
                                   name='extremo rotulado (circulo abierto)', hoverinfo='skip'))

    # --- nodos con IDs y apoyos ---
    nodos_id = [(i, nodos[i]) for i in range(len(nodos))]
    # IDs de barra (B) en el punto medio de cada barra; se pueden ocultar desde la leyenda
    mx = [(nodos[b['n1']][0] + nodos[b['n2']][0]) / 2 for b in dib.barras]
    my = [(nodos[b['n1']][1] + nodos[b['n2']][1]) / 2 for b in dib.barras]
    mz = [(nodos[b['n1']][2] + nodos[b['n2']][2]) / 2 for b in dib.barras]
    fig.add_trace(go.Scatter3d(x=mx, y=my, z=mz, mode='text',
                               text=[f"B{id_barra.get(i, i)}" for i in range(len(dib.barras))],
                               textfont=dict(size=12, color='#6a1b9a'), name='ID de barra (B)',
                               hoverinfo='skip'))
    fig.add_trace(go.Scatter3d(x=nodos[:, 0], y=nodos[:, 1], z=nodos[:, 2], mode='markers+text',
                               marker=dict(size=3, color='black'),
                               text=[f"N{id_de_indice.get(i, i)}" for i in range(len(nodos))],
                               textposition='top center', textfont=dict(size=12, color='black'),
                               name='ID de nodo (N)', hoverinfo='text',
                               hovertext=[f'nodo ID {id_de_indice.get(i, i)}<br>X {p[0]:g}  Y {p[1]:g}  Z {p[2]:g}'
                                          for i, p in nodos_id]))
    fig.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='black', width=4),
                               name='eje fuerte (flecha)', hoverinfo='skip'))
    for tipo, simbolo, color, nombre in [('empotrado', 'square', '#e67e22', 'apoyo empotrado (cuadrado)'),
                                         ('rotulado', 'diamond', '#e67e22', 'apoyo rotulado (rombo)')]:
        idx = [i for i, t in dib.apoyos.items() if t == tipo]
        if idx:
            fig.add_trace(go.Scatter3d(x=nodos[idx, 0], y=nodos[idx, 1], z=nodos[idx, 2], mode='markers',
                                       marker=dict(symbol=simbolo, size=7, color=color), name=nombre,
                                       hoverinfo='skip'))

    fig.update_layout(
        title=titulo or f'Modelo 3D: secciones x{escala_seccion:g}; flecha = eje fuerte (ver leyenda)',
        scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)', aspectmode='data',
                   camera=dict(eye=dict(x=1.5, y=-1.7, z=0.9))),
        margin=dict(l=0, r=0, t=50, b=0),
        legend=dict(itemsizing='constant', title=dict(text='Leyenda'), orientation='h',
                    yanchor='top', y=-0.02, xanchor='left', x=0, font=dict(size=11)),
        height=760,
    )
    return fig


## Bloque 5. Uso

Ejecutar los bloques 1 a 4 una sola vez. Luego trabajar con los bloques 6 a 8.

## Bloque 6. Cargar plantilla Excel

Carga `plantilla_soportes.xlsx` y muestra sus hojas. Llénela en Excel y súbala a Kaggle con Add data.

In [ ]:
# Plantilla Excel: escribe el libro completo (NODOS, BARRAS, CARGAS, ESTADOS, CONEXIONES, LISTAS, VISTA_VIVA).
# La misma funcion regenera el Excel con sus datos en el bloque 8, asi nada se pierde al guardar.
import os
import base64
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.chart import ScatterChart, Reference, Series
from openpyxl.drawing.image import Image as XLImage
from openpyxl.workbook.properties import CalcProperties
from dibujo_directo import DibujoModelo, ELEMENTOS, TIPOS_APOYO, TIPOS_CONEXION
import perfiles_gerdau as G
import pandas as pd

NODOS_COLS = ['ID', 'X (m)', 'Y (m)', 'Z (m)', 'APOYO', 'NOTA']
BARRAS_COLS = ['ID', 'NODO_I', 'NODO_J', 'ELEMENTO', 'PERFIL', 'EJE_FUERTE_X', 'EJE_FUERTE_Y', 'EJE_FUERTE_Z',
               'CONEXION_I', 'CONEXION_J', 'MATERIAL', 'NOTA']
MAT_MAX = 10
MAT_FIN = MAT_MAX + 1
MATERIALES_DEFAULT = [
    ['A-36', 'Acero', 2530, 4078, 'Manual Gerdau, pie de pagina 22 (fy). Fu: modelo SAP CMZ-5746 (A36).'],
    ['A-529 G 50', 'Acero', 3515, 4570, 'Manual Gerdau, pie de pagina 22 (fy). Fu 65 ksi: referencia Gerdau Corsa (confirmar en manual).'],
    ['A-572 G 50', 'Acero', 3515, None, 'Manual Gerdau, pie de pagina 22. Perfiles IR (pag. 6): A992 / A572 G50. Fu pendiente de dato.'],
    ['A-572 G 60', 'Acero', 4200, None, 'Manual Gerdau, pie de pagina 22. Fu pendiente de dato.'],
]
NODOS_MAX = 98          # filas 2..99 de NODOS
BARRAS_MAX = 106        # filas 2..107 de BARRAS
FIN_N = NODOS_MAX + 1
FIN_B = BARRAS_MAX + 1
NIV = 'VISTA_VIVA!$C$3'     # nivel Z para planta (celda editable)
COTA = 'VISTA_VIVA!$C$4'    # cota Y para alzado (celda editable)
PERFILES_ORDEN = G.lista_perfiles_ordenada()
ENCABEZADO = PatternFill('solid', fgColor='DDEBF7')
CABECERA_CALC = PatternFill('solid', fgColor='EDEDED')
ENTRADA = PatternFill('solid', fgColor='FFF2CC')
NEGRITA = Font(bold=True)


def _encabezado(ws, columnas, anchos, relleno=ENCABEZADO):
    for c, nombre in enumerate(columnas, start=1):
        celda = ws.cell(row=1, column=c, value=nombre)
        celda.font = NEGRITA; celda.fill = relleno
        celda.alignment = Alignment(horizontal='center', wrap_text=True)
        ws.column_dimensions[celda.column_letter].width = anchos[c - 1]
    ws.freeze_panes = 'A2'


def _filas(df, columnas):
    """Filas (listas) de un DataFrame leido de la planilla, en el orden de columnas; NaN -> None."""
    salida = []
    for _, r in df.iterrows():
        fila = []
        for c in columnas:
            v = r[c] if c in r.index else None
            if v is None or (isinstance(v, float) and pd.isna(v)):
                v = None
            fila.append(v)
        salida.append(fila)
    return salida


def ejemplo_filas():
    """Marco de ejemplo: 18 nodos, 26 barras, 6 apoyos empotrados."""
    ejemplo = DibujoModelo(ruta='ejemplo')
    ejemplo.marco_grilla([0, 5, 10], [0, 4], [0, 3.5, 7], 'IR 10 × 30', 'IR 12 × 16', 'IR 8 × 28')
    for i, q in enumerate(ejemplo.nodos):
        if abs(q[2]) < 1e-9:
            ejemplo.apoyo(i, 'empotrado')
    nodos = [[i + 1, x, y, z, ejemplo.apoyos.get(i), None] for i, (x, y, z) in enumerate(ejemplo.nodos)]
    barras = [[k + 1, b['n1'] + 1, b['n2'] + 1, b['elemento'], b['perfil'], None, None, None,
               b['conexion_i'], b['conexion_j'], (b.get('material') or 'A-572 G 50'), None] for k, b in enumerate(ejemplo.barras)]
    return nodos, barras


def _hoja_calculo(wb):
    """Columnas que alimentan los graficos. Regla: la coordenada X de cada serie es siempre numerica (asi el
    grafico no usa numeros de fila); solo la coordenada que se oculta (la de la vista que no corresponde) es #N/A."""
    ws = wb.create_sheet('CALCULO')
    _encabezado(ws, ['', 'nodo I (fila)', 'nodo J (fila)', 'X1', 'X2', 'Y1', 'Y2', 'Z1', 'Z2',
                     'planta X1', 'planta X2', 'planta Y1', 'planta Y2',
                     'alzado X1', 'alzado X2', 'alzado Z1', 'alzado Z2', '',
                     'nodo X (planta)', 'nodo Y (planta)', 'nodo X (alzado)', 'nodo Z (alzado)',
                     'apoyo X (planta)', 'apoyo Y (planta)', 'apoyo X (alzado)', 'apoyo Z (alzado)'],
                [4] + [11] * 16 + [3] + [11] * 8, relleno=CABECERA_CALC)
    for r in range(2, FIN_B + 1):
        ws[f'B{r}'] = f'=IFERROR(MATCH(BARRAS!B{r},NODOS!$A$2:$A${FIN_N},0),NA())'
        ws[f'C{r}'] = f'=IFERROR(MATCH(BARRAS!C{r},NODOS!$A$2:$A${FIN_N},0),NA())'
        for col, ref, idx in [('D', 'B', 'B'), ('E', 'B', 'C'), ('F', 'C', 'B'), ('G', 'C', 'C'),
                              ('H', 'D', 'B'), ('I', 'D', 'C')]:
            ws[f'{col}{r}'] = f'=IFERROR(INDEX(NODOS!${ref}$2:${ref}${FIN_N},{idx}{r}),0)'
        vis_p = f'AND(ISNUMBER(B{r}),H{r}={NIV},I{r}={NIV})'
        vis_a = f'AND(ISNUMBER(B{r}),F{r}={COTA},G{r}={COTA})'
        ws[f'J{r}'] = f'=D{r}'
        ws[f'K{r}'] = f'=E{r}'
        ws[f'L{r}'] = f'=IF({vis_p},F{r},NA())'
        ws[f'M{r}'] = f'=IF({vis_p},G{r},NA())'
        ws[f'N{r}'] = f'=D{r}'
        ws[f'O{r}'] = f'=E{r}'
        ws[f'P{r}'] = f'=IF({vis_a},H{r},NA())'
        ws[f'Q{r}'] = f'=IF({vis_a},I{r},NA())'
    for col_, titulo in [(28, 'rotula I planta (Y)'), (29, 'rotula J planta (Y)'),
                         (30, 'rotula I alzado (Z)'), (31, 'rotula J alzado (Z)')]:
        ws.cell(row=1, column=col_, value=titulo).font = NEGRITA
        ws.column_dimensions[_L(col_)].width = 14
    for r in range(2, FIN_B + 1):
        rot_i = f'LOWER(TRIM(BARRAS!I{r}))="rotulado"'
        rot_j = f'LOWER(TRIM(BARRAS!J{r}))="rotulado"'
        ws[f'AB{r}'] = f'=IF(AND({rot_i},ISNUMBER(B{r}),H{r}={NIV}),F{r},NA())'
        ws[f'AC{r}'] = f'=IF(AND({rot_j},ISNUMBER(C{r}),I{r}={NIV}),G{r},NA())'
        ws[f'AD{r}'] = f'=IF(AND({rot_i},ISNUMBER(B{r}),F{r}={COTA}),H{r},NA())'
        ws[f'AE{r}'] = f'=IF(AND({rot_j},ISNUMBER(C{r}),G{r}={COTA}),I{r},NA())'
    for r in range(2, FIN_N + 1):
        ok = f'NODOS!$A{r}<>""'
        ws[f'S{r}'] = f'=NODOS!$B{r}'
        ws[f'T{r}'] = f'=IF(AND({ok},NODOS!$D{r}={NIV}),NODOS!$C{r},NA())'
        ws[f'U{r}'] = f'=NODOS!$B{r}'
        ws[f'V{r}'] = f'=IF(AND({ok},NODOS!$C{r}={COTA}),NODOS!$D{r},NA())'
        ws[f'W{r}'] = f'=NODOS!$B{r}'
        ws[f'X{r}'] = f'=IF(AND(NODOS!$E{r}<>"",NODOS!$D{r}={NIV}),NODOS!$C{r},NA())'
        ws[f'Y{r}'] = f'=NODOS!$B{r}'
        ws[f'Z{r}'] = f'=IF(AND(NODOS!$E{r}<>"",NODOS!$C{r}={COTA}),NODOS!$D{r},NA())'
    return ws


from openpyxl.utils import get_column_letter as _L, column_index_from_string as _CI

CALC3D_BARRAS = ['bx1', 'bxm', 'bx2', 'bxm2', 'bxt',
                 'by1', 'bym', 'by2', 'bym2', 'byt',
                 'L', 'xu', 'yu', 'zu', 'nb', 'nc', 'hasU', 'dot',
                 'ux', 'uy', 'uz', 'un', 'ex', 'ey', 'ez', 'mx', 'my', 'mz', 'rotI', 'rotJ', 'lblB',
                 'rot', 'nA', 'nM', 'nB', 'rA', 'rM', 'rB']
CALC3D_NODOS = ['nsx', 'nsy', 'apE', 'apR', 'lblN']


def _hoja_3d(wb):
    """Vista 3D esquematica en vivo (proyeccion ortogonal). Azimut (C5), elevacion (C6) y largo de la flecha (C7)
    de VISTA_VIVA. Cada barra es una serie de 5 puntos: inicio, medio, fin, medio, punta de la flecha del eje
    fuerte (vector del usuario o por defecto del motor, igual que ejes_locales())."""
    ws = wb.create_sheet('CALCULO_3D')
    col = {}
    for k, nombre in enumerate(CALC3D_BARRAS, start=2):
        col[nombre] = _L(k)
    ini_n = len(CALC3D_BARRAS) + 4
    for k, nombre in enumerate(CALC3D_NODOS, start=ini_n):
        col[nombre] = _L(k)
    KC = _L(ini_n + len(CALC3D_NODOS) + 2)
    VC = _L(ini_n + len(CALC3D_NODOS) + 3)
    col['_VC'] = VC
    _encabezado(ws, [''] + CALC3D_BARRAS + [''] * 2 + CALC3D_NODOS,
                [4] + [9] * len(CALC3D_BARRAS) + [3, 3] + [9] * len(CALC3D_NODOS), relleno=CABECERA_CALC)
    cA, sA, cE, sE, largo = f'${VC}$2', f'${VC}$3', f'${VC}$4', f'${VC}$5', f'${VC}$6'
    ws[f'{KC}2'] = 'cos(azimut)'; ws[f'{VC}2'] = f'=COS(RADIANS(VISTA_VIVA!$C$5))'
    ws[f'{KC}3'] = 'sen(azimut)'; ws[f'{VC}3'] = f'=SIN(RADIANS(VISTA_VIVA!$C$5))'
    ws[f'{KC}4'] = 'cos(elevacion)'; ws[f'{VC}4'] = f'=COS(RADIANS(VISTA_VIVA!$C$6))'
    ws[f'{KC}5'] = 'sen(elevacion)'; ws[f'{VC}5'] = f'=SIN(RADIANS(VISTA_VIVA!$C$6))'
    ws[f'{KC}6'] = 'largo flecha (m)'; ws[f'{VC}6'] = '=VISTA_VIVA!$C$7'
    ws[f'{KC}7'] = 'x de leyenda (no se dibuja)'; ws[f'{VC}7'] = 0
    ws[f'{KC}8'] = 'y de leyenda (no se dibuja)'; ws[f'{VC}8'] = '=NA()'
    for r in range(2, FIN_B + 1):
        c = lambda n: f'{col[n]}{r}'
        vis = f'ISNUMBER(CALCULO!B{r})'
        mx, my, mz = c('mx'), c('my'), c('mz')
        f = {
            'bx1': f'=CALCULO!D{r}*{cA}-CALCULO!F{r}*{sA}',
            'bx2': f'=CALCULO!E{r}*{cA}-CALCULO!G{r}*{sA}',
            'bxm': f'={mx}*{cA}-{my}*{sA}',
            'bxm2': f'={c("bxm")}',
            'bxt': f'=({mx}+{largo}*{c("ex")})*{cA}-({my}+{largo}*{c("ey")})*{sA}',
            'by1': f'=IF({vis},CALCULO!H{r}*{cE}+(CALCULO!D{r}*{sA}+CALCULO!F{r}*{cA})*{sE},NA())',
            'by2': f'=IF(ISNUMBER(CALCULO!C{r}),CALCULO!I{r}*{cE}+(CALCULO!E{r}*{sA}+CALCULO!G{r}*{cA})*{sE},NA())',
            'bym': f'=IF({vis},{mz}*{cE}+({mx}*{sA}+{my}*{cA})*{sE},NA())',
            'bym2': f'={c("bym")}',
            'byt': f'=IF({vis},({mz}+{largo}*{c("ez")})*{cE}+(({mx}+{largo}*{c("ex")})*{sA}+({my}+{largo}*{c("ey")})*{cA})*{sE},NA())',
            'L': f'=IF({vis},SQRT((CALCULO!E{r}-CALCULO!D{r})^2+(CALCULO!G{r}-CALCULO!F{r})^2+(CALCULO!I{r}-CALCULO!H{r})^2),1)',
            'xu': f'=(CALCULO!E{r}-CALCULO!D{r})/{c("L")}',
            'yu': f'=(CALCULO!G{r}-CALCULO!F{r})/{c("L")}',
            'zu': f'=(CALCULO!I{r}-CALCULO!H{r})/{c("L")}',
            'nb': f'=MAX(SQRT({c("xu")}^2+{c("yu")}^2),0.000000001)',
            'nc': f'=MAX(SQRT({c("zu")}^2+{c("yu")}^2),0.000000001)',
            'hasU': f'=COUNT(BARRAS!F{r}:H{r})>0',
            'dot': f'=N(BARRAS!F{r})*{c("xu")}+N(BARRAS!G{r})*{c("yu")}+N(BARRAS!H{r})*{c("zu")}',
            'ux': f'=N(BARRAS!F{r})-{c("dot")}*{c("xu")}',
            'uy': f'=N(BARRAS!G{r})-{c("dot")}*{c("yu")}',
            'uz': f'=N(BARRAS!H{r})-{c("dot")}*{c("zu")}',
            'un': f'=MAX(SQRT({c("ux")}^2+{c("uy")}^2+{c("uz")}^2),0.000000001)',
            'ex': f'=IFERROR(IF({c("hasU")},{c("ux")}/{c("un")},IF(ABS({c("zu")})>0.9,0,-{c("xu")}*{c("zu")}/{c("nb")})),0)',
            'ey': f'=IFERROR(IF({c("hasU")},{c("uy")}/{c("un")},IF(ABS({c("zu")})>0.9,-{c("zu")}/{c("nc")},-{c("yu")}*{c("zu")}/{c("nb")})),0)',
            'ez': f'=IFERROR(IF({c("hasU")},{c("uz")}/{c("un")},IF(ABS({c("zu")})>0.9,{c("yu")}/{c("nc")},({c("xu")}^2+{c("yu")}^2)/{c("nb")})),0)',
            'mx': f'=(CALCULO!D{r}+CALCULO!E{r})/2',
            'my': f'=(CALCULO!F{r}+CALCULO!G{r})/2',
            'mz': f'=(CALCULO!H{r}+CALCULO!I{r})/2',
            'rotI': f'=IF(AND({vis},LOWER(TRIM(BARRAS!I{r}))="rotulado"),{c("by1")},NA())',
            'rotJ': f'=IF(AND(ISNUMBER(CALCULO!C{r}),LOWER(TRIM(BARRAS!J{r}))="rotulado"),{c("by2")},NA())',
            'lblB': f'=IF(BARRAS!A{r}="","","B"&BARRAS!A{r})',
            'rot': f'=IF(OR(LOWER(TRIM(BARRAS!I{r}))="rotulado",LOWER(TRIM(BARRAS!J{r}))="rotulado"),1,0)',
            'nA': f'=IF(AND({vis},{c("rot")}=0),{c("by1")},NA())',
            'nM': f'=IF(AND({vis},{c("rot")}=0),{c("bym")},NA())',
            'nB': f'=IF(AND(ISNUMBER(CALCULO!C{r}),{c("rot")}=0),{c("by2")},NA())',
            'rA': f'=IF(AND({vis},{c("rot")}=1),{c("by1")},NA())',
            'rM': f'=IF(AND({vis},{c("rot")}=1),{c("bym")},NA())',
            'rB': f'=IF(AND(ISNUMBER(CALCULO!C{r}),{c("rot")}=1),{c("by2")},NA())',
        }
        for nombre, formula in f.items():
            ws[c(nombre)] = formula
    for r in range(2, FIN_N + 1):
        ws[f"{col['nsx']}{r}"] = f'=NODOS!B{r}*{cA}-NODOS!C{r}*{sA}'
        ws[f"{col['nsy']}{r}"] = f'=IF(NODOS!A{r}<>"",NODOS!D{r}*{cE}+(NODOS!B{r}*{sA}+NODOS!C{r}*{cA})*{sE},NA())'
        for nombre, tipo in [('apE', 'empotrado'), ('apR', 'rotulado')]:
            ws[f"{col[nombre]}{r}"] = (f'=IF(AND(NODOS!A{r}<>"",LOWER(TRIM(NODOS!E{r}))="{tipo}"),'
                                       f'NODOS!D{r}*{cE}+(NODOS!B{r}*{sA}+NODOS!C{r}*{cA})*{sE},NA())')
        ws[f"{col['lblN']}{r}"] = f'=IF(NODOS!A{r}="","","N"&NODOS!A{r})'
    return ws, col


def _grafico_3d(ws_v, ws3, col):
    """Vista 3D esquematica en vivo. Cada barra y cada nodo es una serie cuyo nombre sale de una celda
    (B14, N3): asi las etiquetas funcionan en todas las versiones de Excel. Barra gris; flecha negra (ultimo
    tramo de la serie); extremo rotulado = circulo blanco; apoyo empotrado = cuadrado; rotulado = rombo.
    Limite de Excel: 255 series por grafico; aqui hay 214."""
    from openpyxl.chart.legend import Legend, LegendEntry
    from openpyxl.chart.series import SeriesLabel
    from openpyxl.chart.data_source import StrRef
    from openpyxl.chart.label import DataLabel, DataLabelList
    from openpyxl.chart.marker import DataPoint
    from openpyxl.chart.shapes import GraphicalProperties
    from openpyxl.drawing.line import LineProperties

    ch = ScatterChart()
    ch.title = '3D esquematico (C5 azimut, C6 elevacion)'
    ch.style = 13
    ch.height, ch.width = 13, 17
    ch.x_axis.title, ch.y_axis.title = ('horizontal', 'vertical')
    ch.x_axis.delete = False; ch.y_axis.delete = False

    def ci(nombre):
        return _CI(col[nombre])

    def ref_nombre(nombre, r):
        return f"'CALCULO_3D'!${col[nombre]}${r}"

    def etiqueta(idx):
        return DataLabelList(
            dLbl=[DataLabel(idx=idx, showSerName=True, showVal=False, showCatName=False,
                            showLegendKey=False, showPercent=False, showBubbleSize=False, dLblPos='t')],
            showSerName=False, showVal=False, showCatName=False, showLegendKey=False,
            showPercent=False, showBubbleSize=False)

    for r in range(2, FIN_B + 1):
        s = Series(Reference(ws3, min_col=ci('by1'), max_col=ci('byt'), min_row=r, max_row=r),
                   Reference(ws3, min_col=ci('bx1'), max_col=ci('bxt'), min_row=r, max_row=r))
        s.tx = SeriesLabel(strRef=StrRef(ref_nombre('lblB', r)))
        s.marker.symbol = 'none'
        s.graphicalProperties.line.solidFill = '7F7F7F'
        s.graphicalProperties.line.width = 28000
        s.smooth = False
        s.smooth = False
        s.dPt = [DataPoint(idx=4, spPr=GraphicalProperties(ln=LineProperties(solidFill='000000', w=15000)))]
        s.dLbls = etiqueta(1)
        ch.series.append(s)
    for nombre_x, nombre_y in [('bx1', 'rotI'), ('bx2', 'rotJ')]:
        s = Series(Reference(ws3, min_col=ci(nombre_y), min_row=2, max_row=FIN_B),
                   Reference(ws3, min_col=ci(nombre_x), min_row=2, max_row=FIN_B), title='Extremo rotulado')
        s.marker.symbol = 'circle'; s.marker.size = 8
        s.marker.graphicalProperties.solidFill = 'FFFFFF'
        s.marker.graphicalProperties.line.solidFill = '000000'
        s.graphicalProperties.line.noFill = True
        ch.series.append(s)
    for r in range(2, FIN_N + 1):
        n = Series(Reference(ws3, min_col=ci('nsy'), min_row=r, max_row=r),
                   Reference(ws3, min_col=ci('nsx'), min_row=r, max_row=r))
        n.tx = SeriesLabel(strRef=StrRef(ref_nombre('lblN', r)))
        n.marker.symbol = 'circle'; n.marker.size = 4
        n.marker.graphicalProperties.solidFill = '000000'
        n.graphicalProperties.line.noFill = True
        n.smooth = False
        n.dLbls = etiqueta(0)
        ch.series.append(n)
    for nombre_y, simbolo, etiq in [('apE', 'square', 'Apoyo empotrado'), ('apR', 'diamond', 'Apoyo rotulado')]:
        a = Series(Reference(ws3, min_col=ci(nombre_y), min_row=2, max_row=FIN_N),
                   Reference(ws3, min_col=ci('nsx'), min_row=2, max_row=FIN_N), title=etiq)
        a.marker.symbol = simbolo; a.marker.size = 9
        a.marker.graphicalProperties.solidFill = 'E67E22'
        a.graphicalProperties.line.noFill = True
        ch.series.append(a)
    # series de leyenda: no dibujan nada (y = #N/A); solo muestran el simbolo de cada tipo
    x_leyenda = f"{col['_VC']}7"
    y_leyenda = f"{col['_VC']}8"
    leyenda = [('Barra', 'linea', '7F7F7F'), ('Eje fuerte (flecha)', 'linea', '000000'),
               ('Extremo rotulado (circulo blanco)', 'circulo', 'FFFFFF'), ('Nodo (N)', 'circulo', '000000'),
               ('Apoyo empotrado (cuadrado)', 'square', 'E67E22'), ('Apoyo rotulado (rombo)', 'diamond', 'E67E22')]
    primera_leyenda = len(ch.series)
    for etiq, tipo, color in leyenda:
        s = Series(Reference(ws3, min_col=_CI(col['_VC']), min_row=8, max_row=8),
                   Reference(ws3, min_col=_CI(col['_VC']), min_row=7, max_row=7), title=etiq)
        if tipo == 'linea':
            s.marker.symbol = 'none'
            s.graphicalProperties.line.solidFill = color
            s.graphicalProperties.line.width = 28000 if color == '7F7F7F' else 15000
        else:
            s.marker.symbol = {'circulo': 'circle', 'square': 'square', 'diamond': 'diamond'}[tipo]
            s.marker.size = 8 if tipo != 'circulo' or color == 'FFFFFF' else 5
            s.marker.graphicalProperties.solidFill = color
            s.marker.graphicalProperties.line.solidFill = '000000'
            s.graphicalProperties.line.noFill = True
        ch.series.append(s)
    for serie in ch.series:
        serie.smooth = False          # lineas rectas: sin suavizado las flechas no se curvan
    assert len(ch.series) <= 255, f'grafico 3D con {len(ch.series)} series (limite de Excel: 255)'
    ch.legend = Legend()
    ch.legend.position = 'b'
    ch.legend.legendEntry = [LegendEntry(idx=i, delete=True) for i in range(primera_leyenda)]
    ws_v.add_chart(ch, 'A31')


def _grafico_3d_rotulas(ws_v, ws3, col):
    """3D con rotulas de los elementos (sin IDs): barras normales (gris), barras con extremo rotulado (morada
    discontinua) y circulo blanco en cada extremo rotulado. Usa la misma proyeccion que la 3D principal."""
    from openpyxl.chart.legend import Legend, LegendEntry
    ch = ScatterChart()
    ch.title = '3D con rotulas de los elementos'
    ch.style = 13
    ch.height, ch.width = 13, 17
    ch.x_axis.title, ch.y_axis.title = ('horizontal', 'vertical')
    ch.x_axis.delete = False; ch.y_axis.delete = False

    def ci(nombre):
        return _CI(col[nombre])

    for r in range(2, FIN_B + 1):
        s = Series(Reference(ws3, min_col=ci('nA'), max_col=ci('nB'), min_row=r, max_row=r),
                   Reference(ws3, min_col=ci('bx1'), max_col=ci('bx2'), min_row=r, max_row=r), title='Barra')
        s.marker.symbol = 'none'
        s.graphicalProperties.line.solidFill = '7F7F7F'
        s.graphicalProperties.line.width = 28000
        s.smooth = False
        ch.series.append(s)
    for r in range(2, FIN_B + 1):
        s = Series(Reference(ws3, min_col=ci('rA'), max_col=ci('rB'), min_row=r, max_row=r),
                   Reference(ws3, min_col=ci('bx1'), max_col=ci('bx2'), min_row=r, max_row=r),
                   title='Barra con extremo rotulado')
        s.marker.symbol = 'none'
        s.graphicalProperties.line.solidFill = '6A1B9A'
        s.graphicalProperties.line.width = 28000
        s.graphicalProperties.line.prstDash = 'dash'
        s.smooth = False
        ch.series.append(s)
    for nombre_x, nombre_y in [('bx1', 'rotI'), ('bx2', 'rotJ')]:
        s = Series(Reference(ws3, min_col=ci(nombre_y), min_row=2, max_row=FIN_B),
                   Reference(ws3, min_col=ci(nombre_x), min_row=2, max_row=FIN_B), title='Extremo rotulado')
        s.marker.symbol = 'circle'; s.marker.size = 8
        s.marker.graphicalProperties.solidFill = '6A1B9A'
        s.marker.graphicalProperties.line.solidFill = '6A1B9A'
        s.graphicalProperties.line.noFill = True
        s.smooth = False
        ch.series.append(s)
    n = Series(Reference(ws3, min_col=ci('nsy'), min_row=2, max_row=FIN_N),
               Reference(ws3, min_col=ci('nsx'), min_row=2, max_row=FIN_N), title='Nodo')
    n.marker.symbol = 'circle'; n.marker.size = 4
    n.marker.graphicalProperties.solidFill = '000000'
    n.graphicalProperties.line.noFill = True
    n.smooth = False
    ch.series.append(n)
    for nombre_y, simbolo, etiq in [('apE', 'square', 'Apoyo empotrado (cuadrado)'),
                                    ('apR', 'diamond', 'Apoyo rotulado (rombo)')]:
        a = Series(Reference(ws3, min_col=ci(nombre_y), min_row=2, max_row=FIN_N),
                   Reference(ws3, min_col=ci('nsx'), min_row=2, max_row=FIN_N), title=etiq)
        a.marker.symbol = simbolo; a.marker.size = 9
        a.marker.graphicalProperties.solidFill = 'E67E22'
        a.graphicalProperties.line.noFill = True
        a.smooth = False
        ch.series.append(a)
    primera_leyenda = len(ch.series)
    for etiq, tipo, color in [('Barra', 'linea', '7F7F7F'), ('Barra con extremo rotulado', 'linea', '6A1B9A'),
                              ('Extremo rotulado (circulo morado)', 'circulo', '6A1B9A'),
                              ('Nodo', 'circulo', '000000'), ('Apoyo empotrado (cuadrado)', 'square', 'E67E22'),
                              ('Apoyo rotulado (rombo)', 'diamond', 'E67E22')]:
        s = Series(Reference(ws3, min_col=_CI(col['_VC']), min_row=8, max_row=8),
                   Reference(ws3, min_col=_CI(col['_VC']), min_row=7, max_row=7), title=etiq)
        if tipo == 'linea':
            s.marker.symbol = 'none'
            s.graphicalProperties.line.solidFill = color
            s.graphicalProperties.line.width = 28000
        else:
            s.marker.symbol = {'circulo': 'circle', 'square': 'square', 'diamond': 'diamond'}[tipo]
            s.marker.size = 7
            s.marker.graphicalProperties.solidFill = color
            s.marker.graphicalProperties.line.solidFill = '000000'
            s.graphicalProperties.line.noFill = True
        ch.series.append(s)
    for serie in ch.series:
        serie.smooth = False
    assert len(ch.series) <= 255, f'grafico 3D con rotulas: {len(ch.series)} series'
    ch.legend = Legend()
    ch.legend.position = 'b'
    ch.legend.legendEntry = [LegendEntry(idx=i, delete=True) for i in range(primera_leyenda)]
    ws_v.add_chart(ch, 'A60')


def _graficos(ws_v, ws_c):
    """Planta (X-Y) y alzado (X-Z) en VISTA_VIVA: barras, nodos y apoyos vinculados a las celdas."""
    def grafico(titulo, ejes, x_nodo, y_nodo, x_apoyo, y_apoyo, rango_barras, etiquetas):
        ch = ScatterChart()
        ch.title = titulo
        ch.style = 13
        ch.legend = None
        ch.height, ch.width = 10, 16
        ch.x_axis.title, ch.y_axis.title = etiquetas
        ch.x_axis.delete = False; ch.y_axis.delete = False
        for r in range(2, FIN_B + 1):
            s = Series(Reference(ws_c, min_col=rango_barras[1], max_col=rango_barras[1] + 1, min_row=r, max_row=r),
                       Reference(ws_c, min_col=rango_barras[0], max_col=rango_barras[0] + 1, min_row=r, max_row=r))
            s.marker.symbol = 'none'
            s.graphicalProperties.line.solidFill = '555555'
            s.graphicalProperties.line.width = 22000
            s.smooth = False
            ch.series.append(s)
        n = Series(Reference(ws_c, min_col=y_nodo, min_row=2, max_row=FIN_N),
                   Reference(ws_c, min_col=x_nodo, min_row=2, max_row=FIN_N), title='Nodos')
        n.marker.symbol = 'circle'; n.marker.size = 5
        n.marker.graphicalProperties.solidFill = '000000'
        n.graphicalProperties.line.noFill = True
        ch.series.append(n)
        a = Series(Reference(ws_c, min_col=y_apoyo, min_row=2, max_row=FIN_N),
                   Reference(ws_c, min_col=x_apoyo, min_row=2, max_row=FIN_N), title='Apoyos')
        a.marker.symbol = 'square'; a.marker.size = 9
        a.marker.graphicalProperties.solidFill = 'E67E22'
        a.graphicalProperties.line.noFill = True
        ch.series.append(a)
        return ch

    # planta: barras X1:X2 (J:K) y Y1:Y2 (L:M); nodos S (x) y T (y); apoyos W, X
    p = grafico('Planta (nivel Z de C3)', None, 19, 20, 23, 24, (10, 12),
                ('X (m)', 'Y (m)'))
    # alzado: barras X1:X2 (N:O) y Z1:Z2 (P:Q); nodos U (x) y V (z); apoyos Y, Z
    z = grafico('Alzado X-Z (cota Y de C4)', None, 21, 22, 25, 26, (14, 16),
                ('X (m)', 'Z (m)'))
    for ch_obj, ycols in [(p, ('AB', 'AC')), (z, ('AD', 'AE'))]:
        for xcol, ycol in [('D', ycols[0]), ('E', ycols[1])]:
            s_ = Series(Reference(ws_c, min_col=_CI(ycol), min_row=2, max_row=FIN_B),
                        Reference(ws_c, min_col=_CI(xcol), min_row=2, max_row=FIN_B), title='Rotula')
            s_.marker.symbol = 'circle'; s_.marker.size = 8
            s_.marker.graphicalProperties.solidFill = '6A1B9A'
            s_.marker.graphicalProperties.line.solidFill = '6A1B9A'
            s_.graphicalProperties.line.noFill = True
            s_.smooth = False
            ch_obj.series.append(s_)
    ws_v.add_chart(p, 'A10')
    ws_v.add_chart(z, 'J10')


# ---------------- CARGAS EN NODOS ----------------
CARGAS_COLS = ['ID NODO', 'ESTADO', 'FX (kgf)', 'FY (kgf)', 'FZ (kgf)', 'MX (kgf·m)', 'MY (kgf·m)', 'MZ (kgf·m)', 'NOTA']
CARGA_FILA0 = 9                       # primera fila de datos de la hoja CARGAS
CARGA_MAX = 300
CARGA_FIN = CARGA_FILA0 + CARGA_MAX - 1
ESTADOS_MAX = 30                      # filas 2..31 de la hoja ESTADOS
ESTADOS_FIN = ESTADOS_MAX + 1


def ejemplo_estados():
    """Estados de carga de ejemplo (el usuario los nombra y agrega en la hoja ESTADOS)."""
    return [['Peso propio y muerta', 'D', 'Peso propio, tabiqueria y cubierta'],
            ['Sobrecarga', 'L', 'Sobrecarga de uso en el nivel superior'],
            ['Viento X', 'W', 'Viento en direccion X sobre fachada izquierda']]


def ejemplo_cargas():
    """Ejemplo: peso propio sobre los 6 nudos del nivel superior y viento X sobre 2 de ellos."""
    filas = [[id_, 'Peso propio y muerta', 0, 0, -1500, 0, 0, 0, None] for id_ in (3, 6, 9, 12, 15, 18)]
    filas += [[id_, 'Viento X', 800, 0, 0, 0, 0, 0, None] for id_ in (3, 6)]
    return filas


def _hoja_estados(wb, estados):
    """Hoja ESTADOS: lista de estados de carga que define el usuario (nombre, tipo y descripcion)."""
    ws = wb.create_sheet('ESTADOS')
    _encabezado(ws, ['NOMBRE DEL ESTADO', 'TIPO (D, L, W, E u otro)', 'DESCRIPCION'], [34, 22, 60])
    for f in estados:
        ws.append(list(f))
    for _ in range(ESTADOS_MAX - len(estados)):
        ws.append([None, None, None])
    return ws


def _hoja_calc_cargas(wb, col3d):
    """Calculo de la vista 3D de cargas: suma de cargas por nodo para el estado elegido en CARGAS!C3, y
    posicion en pantalla (misma proyeccion que CALCULO_3D) de cada nodo y de la punta de su flecha."""
    ws = wb.create_sheet('CALCULO_CARGAS')
    cols = ['id', 'fx', 'fy', 'fz', 'mx', 'my', 'mz', 'any', 'mag', 'tx3', 'ty3', 'tz3',
            'ax1', 'ax2', 'ay1', 'ay2', 'lbl']
    col = {nombre: _L(k) for k, nombre in enumerate(cols, start=1)}
    constantes = ['fmax', 'escala', 'dx', 'dy']
    KC = _L(len(cols) + 2)
    VC = _L(len(cols) + 3)
    for k, nombre in enumerate(constantes, start=2):
        col[nombre] = f'${VC}${k}'
    _encabezado(ws, [n for n in cols] + [''] + ['constante', 'valor'] + [''] * 0,
                [7] * 6 + [9] * 11 + [3, 14, 10], relleno=CABECERA_CALC)
    cA = f"CALCULO_3D!${col3d['_VC']}$2"; sA = f"CALCULO_3D!${col3d['_VC']}$3"
    cE = f"CALCULO_3D!${col3d['_VC']}$4"; sE = f"CALCULO_3D!${col3d['_VC']}$5"
    rango = lambda c: f"CARGAS!${c}${CARGA_FILA0}:${c}${CARGA_FIN}"
    ws[f'{KC}2'] = 'fuerza maxima (tf)'; ws[f'{VC}2'] = f'=MAX({col["mag"]}2:{col["mag"]}{FIN_N})'
    ws[f'{KC}3'] = 'escala flecha (m/tf)'; ws[f'{VC}3'] = f'=IF({col["fmax"]}>0,CARGAS!$C$4/{col["fmax"]},0)'
    ws[f'{KC}4'] = 'x de leyenda'; ws[f'{VC}4'] = 0
    ws[f'{KC}5'] = 'y de leyenda'; ws[f'{VC}5'] = '=NA()'
    for r in range(2, FIN_N + 1):
        c = lambda n: f'{col[n]}{r}'
        ws[c('id')] = f'=IF(NODOS!A{r}="","",NODOS!A{r})'
        for nombre, letra in [('fx', 'C'), ('fy', 'D'), ('fz', 'E'), ('mx', 'F'), ('my', 'G'), ('mz', 'H')]:
            ws[c(nombre)] = (f'=IF({c("id")}="",0,SUMIFS({rango(letra)},{rango("A")},{c("id")},'
                             f'{rango("B")},CARGAS!$C$3))')
        ws[c('any')] = (f'=IF(OR({c("fx")}<>0,{c("fy")}<>0,{c("fz")}<>0,{c("mx")}<>0,{c("my")}<>0,'
                        f'{c("mz")}<>0),1,0)')
        ws[c('mag')] = f'=SQRT({c("fx")}^2+{c("fy")}^2+{c("fz")}^2)'
        ws[c('tx3')] = f'=NODOS!B{r}+{col["escala"]}*{c("fx")}'
        ws[c('ty3')] = f'=NODOS!C{r}+{col["escala"]}*{c("fy")}'
        ws[c('tz3')] = f'=NODOS!D{r}+{col["escala"]}*{c("fz")}'
        ws[c('ax1')] = f'=CALCULO_3D!{col3d["nsx"]}{r}'
        ws[c('ax2')] = f'={c("tx3")}*{cA}-{c("ty3")}*{sA}'
        ws[c('ay1')] = f'=IF({c("any")}=1,CALCULO_3D!{col3d["nsy"]}{r},NA())'
        ws[c('ay2')] = (f'=IF(AND({c("any")}=1,{c("mag")}>0),{c("tz3")}*{cE}+({c("tx3")}*{sA}+{c("ty3")}*{cA})*{sE},'
                        f'NA())')
        texto = '&'.join(f'IF({c(n)}<>0,"{e} "&ROUND({c(n)},2)&" ","")'
                        for n, e in [('fx', 'Fx'), ('fy', 'Fy'), ('fz', 'Fz'), ('mx', 'Mx'), ('my', 'My'), ('mz', 'Mz')])
        ws[c('lbl')] = f'=IF({c("any")}=1,"N"&{c("id")}&": "&{texto},"")'
    return ws, col


def _hoja_cargas(wb, filas_cargas, estado_inicial=''):
    """Hoja CARGAS: estado de carga a mostrar (C3), largo de la flecha (C4) y la tabla de cargas por nodo."""
    ws = wb.create_sheet('CARGAS')
    ws['A1'] = 'Cargas aplicadas en los nodos (kgf, kgf·m)'
    ws['A1'].font = Font(bold=True, size=12)
    ws['A3'] = 'Estado de carga a mostrar en el 3D'
    ws['C3'] = estado_inicial
    ws['A4'] = 'Largo maximo de la flecha de carga (m)'
    ws['C4'] = 1.5
    ws['A5'] = 'Azimut y elevacion de la vista: se cambian en VISTA_VIVA (C5 y C6).'
    ws['A6'] = 'Unidades: fuerzas en kgf y momentos en kgf·m. Cada fila es un nodo con un estado de carga (hoja ESTADOS).'
    for celda in ('C3', 'C4'):
        ws[celda].fill = ENTRADA
    for k, nombre in enumerate(CARGAS_COLS, start=1):
        celda = ws.cell(row=CARGA_FILA0 - 1, column=k, value=nombre)
        celda.font = NEGRITA; celda.fill = ENCABEZADO
        celda.alignment = Alignment(horizontal='center', wrap_text=True)
    for k, ancho in enumerate([10, 28, 11, 11, 11, 12, 12, 12, 30], start=1):
        ws.column_dimensions[_L(k)].width = ancho
    for i, f in enumerate(filas_cargas):
        for k, v in enumerate(f, start=1):
            ws.cell(row=CARGA_FILA0 + i, column=k, value=v)
    dv_nodo = DataValidation(type='list', formula1=f'NODOS!$A$2:$A${FIN_N}', allow_blank=True)
    dv_estado = DataValidation(type='list', formula1=f'ESTADOS!$A$2:$A${ESTADOS_FIN}', allow_blank=True)
    dv_num = DataValidation(type='decimal', operator='between', formula1='-1000000', formula2='1000000',
                            allow_blank=True)
    for dv, rango_ in [(dv_nodo, f'A{CARGA_FILA0}:A{CARGA_FIN}'), (dv_estado, f'B{CARGA_FILA0}:B{CARGA_FIN}'),
                       (dv_num, f'C{CARGA_FILA0}:H{CARGA_FIN}')]:
        ws.add_data_validation(dv); dv.add(rango_)
    dv_caso = DataValidation(type='list', formula1=f'ESTADOS!$A$2:$A${ESTADOS_FIN}', allow_blank=True)
    ws.add_data_validation(dv_caso); dv_caso.add('C3')
    ws.freeze_panes = f'A{CARGA_FILA0}'
    return ws


def _grafico_cargas(ws_car, ws_cc, col_cc, ws3, col3d):
    """3D de cargas: barras (gris), nodos (negro), cada nodo con su etiqueta (N y carga) y su flecha de carga
    (roja, punta triangular). Apoyos: cuadrado (empotrado) y rombo (rotulado). Sin IDs de barras."""
    from openpyxl.chart.legend import Legend, LegendEntry
    from openpyxl.chart.series import SeriesLabel
    from openpyxl.chart.data_source import StrRef
    from openpyxl.chart.label import DataLabel, DataLabelList
    from openpyxl.chart.marker import DataPoint, Marker
    from openpyxl.chart.shapes import GraphicalProperties
    from openpyxl.drawing.line import LineProperties

    ch = ScatterChart()
    ch.title = '3D de cargas por nodo (estado en CARGAS!C3)'
    ch.style = 13
    ch.height, ch.width = 14, 18
    ch.x_axis.title, ch.y_axis.title = ('horizontal', 'vertical')
    ch.x_axis.delete = False; ch.y_axis.delete = False

    def ci3(n):
        return _CI(col3d[n])

    def cci(n):
        return _CI(col_cc[n])

    def etiqueta(idx):
        return DataLabelList(
            dLbl=[DataLabel(idx=idx, showSerName=True, showVal=False, showCatName=False, showLegendKey=False,
                            showPercent=False, showBubbleSize=False, dLblPos='t')],
            showSerName=False, showVal=False, showCatName=False, showLegendKey=False,
            showPercent=False, showBubbleSize=False)

    for r in range(2, FIN_B + 1):                     # barras: 3 puntos (inicio, medio, fin), sin etiqueta
        s = Series(Reference(ws3, min_col=ci3('by1'), max_col=ci3('by2'), min_row=r, max_row=r),
                   Reference(ws3, min_col=ci3('bx1'), max_col=ci3('bx2'), min_row=r, max_row=r))
        s.marker.symbol = 'none'
        s.graphicalProperties.line.solidFill = '7F7F7F'
        s.graphicalProperties.line.width = 28000
        s.smooth = False
        ch.series.append(s)
    for r in range(2, FIN_N + 1):                     # una serie por nodo: su nombre es la etiqueta N + carga
        s = Series(Reference(ws_cc, min_col=cci('ay1'), max_col=cci('ay2'), min_row=r, max_row=r),
                   Reference(ws_cc, min_col=cci('ax1'), max_col=cci('ax2'), min_row=r, max_row=r))
        s.tx = SeriesLabel(strRef=StrRef(f"'CALCULO_CARGAS'!${col_cc['lbl']}${r}"))
        s.marker.symbol = 'none'
        s.graphicalProperties.line.solidFill = 'C62828'
        s.graphicalProperties.line.width = 28000
        s.smooth = False
        s.dPt = [DataPoint(idx=1, marker=Marker(symbol='triangle', size=8),
                           spPr=GraphicalProperties(solidFill='C62828', ln=LineProperties(solidFill='C62828')))]
        s.dLbls = etiqueta(0)
        ch.series.append(s)
    n = Series(Reference(ws3, min_col=ci3('nsy'), min_row=2, max_row=FIN_N),
               Reference(ws3, min_col=ci3('nsx'), min_row=2, max_row=FIN_N), title='Nodo (N)')
    n.marker.symbol = 'circle'; n.marker.size = 4
    n.marker.graphicalProperties.solidFill = '000000'
    n.graphicalProperties.line.noFill = True
    n.smooth = False
    ch.series.append(n)
    for nombre_y, simbolo, etiq in [('apE', 'square', 'Apoyo empotrado (cuadrado)'),
                                    ('apR', 'diamond', 'Apoyo rotulado (rombo)')]:
        a = Series(Reference(ws3, min_col=ci3(nombre_y), min_row=2, max_row=FIN_N),
                   Reference(ws3, min_col=ci3('nsx'), min_row=2, max_row=FIN_N), title=etiq)
        a.marker.symbol = simbolo; a.marker.size = 9
        a.marker.graphicalProperties.solidFill = 'E67E22'
        a.graphicalProperties.line.noFill = True
        a.smooth = False
        ch.series.append(a)
    primera_leyenda = len(ch.series)
    x_leg = Reference(ws_cc, min_col=_CI(col_cc['dx'].split('$')[1]), min_row=4, max_row=4)
    y_leg = Reference(ws_cc, min_col=_CI(col_cc['dy'].split('$')[1]), min_row=5, max_row=5)
    for etiq, tipo, color in [('Barra', 'linea', '7F7F7F'), ('Carga aplicada (flecha roja)', 'linea', 'C62828'),
                              ('Nodo (N)', 'circulo', '000000'), ('Apoyo empotrado (cuadrado)', 'square', 'E67E22'),
                              ('Apoyo rotulado (rombo)', 'diamond', 'E67E22')]:
        s = Series(y_leg, x_leg, title=etiq)
        if tipo == 'linea':
            s.marker.symbol = 'none'
            s.graphicalProperties.line.solidFill = color
            s.graphicalProperties.line.width = 28000
        else:
            s.marker.symbol = {'circulo': 'circle', 'square': 'square', 'diamond': 'diamond'}[tipo]
            s.marker.size = 7
            s.marker.graphicalProperties.solidFill = color
            s.graphicalProperties.line.noFill = True
        ch.series.append(s)
    for serie in ch.series:
        serie.smooth = False
    assert len(ch.series) <= 255, f'grafico de cargas con {len(ch.series)} series'
    ch.legend = Legend()
    ch.legend.position = 'b'
    ch.legend.legendEntry = [LegendEntry(idx=i, delete=True) for i in range(primera_leyenda)]
    ws_car.add_chart(ch, 'K3')


def _hoja_materiales(wb, materiales):
    """Hoja MATERIALES: aceros del proyecto (nombre, tipo, Fy y Fu en kgf/cm2, fuente)."""
    ws = wb.create_sheet('MATERIALES')
    _encabezado(ws, ['NOMBRE', 'TIPO', 'FY (kgf/cm2)', 'FU (kgf/cm2)', 'FUENTE'], [14, 12, 14, 14, 80])
    for f in materiales:
        ws.append(list(f))
    for _ in range(MAT_MAX - len(materiales)):
        ws.append([None] * 5)
    return ws


# ---------------- COMBINACIONES DE CARGA ----------------
COMB_FILA0 = 3                       # primera fila de combinaciones
COMB_MAX = 40
COMB_FIN = COMB_FILA0 + COMB_MAX - 1
COMBINACIONES_DEFAULT = [
    [1] + [None] * (ESTADOS_MAX - 1),
    [1, 1] + [None] * (ESTADOS_MAX - 2),
    [1, 0.75, 0.75] + [None] * (ESTADOS_MAX - 3),
    [1, None, 1] + [None] * (ESTADOS_MAX - 3),
    [0.6, None, 1] + [None] * (ESTADOS_MAX - 3),
]


def _hoja_combinaciones(wb, combinaciones):
    """Hoja COMBINACIONES: columnas = estados (de la hoja ESTADOS), filas = combinaciones. Los factores son
    entradas del usuario (vacio = 0). La columna LC se arma sola: 'ASDn = f1·estado1 + f2·estado2 ...'."""
    ws = wb.create_sheet('COMBINACIONES')
    ws['A1'] = 'Combinaciones de carga: escriba los factores en las celdas amarillas (vacio = 0).'
    ws['A1'].font = NEGRITA
    cols = [_L(2 + j) for j in range(ESTADOS_MAX)]
    ws.cell(row=2, column=1, value='LC (calculada)').font = NEGRITA
    ws.cell(row=2, column=1).fill = ENCABEZADO
    ws.column_dimensions['A'].width = 70
    for j, col in enumerate(cols):
        ws[f'{col}2'] = f'=IF(ESTADOS!A{2 + j}="","",ESTADOS!A{2 + j})'
        ws[f'{col}2'].font = NEGRITA
        ws[f'{col}2'].fill = ENCABEZADO
        ws[f'{col}2'].alignment = Alignment(horizontal='center', wrap_text=True)
        ws.column_dimensions[col].width = 14
    ws.row_dimensions[2].height = 45
    for k in range(COMB_MAX):
        r = COMB_FILA0 + k
        terminos = '&'.join(
            f'IF(AND(ISNUMBER({c}{r}),{c}{r}<>0,{c}$2<>""),"+ "&ROUND({c}{r},2)&"·"&{c}$2&" ","")' for c in cols)
        ws[f'A{r}'] = f'=IF(LEN({terminos})=0,"","ASD{k + 1} = "&TRIM(MID({terminos},3,999)))'
    filas = list(combinaciones) if combinaciones else []
    for k in range(COMB_MAX):
        fila = filas[k] if k < len(filas) else [None] * ESTADOS_MAX
        for j, col in enumerate(cols):
            v = fila[j] if j < len(fila) else None
            celda = ws[f'{col}{COMB_FILA0 + k}']
            celda.value = None if v is None or (isinstance(v, str) and v.strip() == '') else v
            celda.fill = ENTRADA
            celda.number_format = '0.00'
    dv = DataValidation(type='decimal', operator='between', formula1='-100', formula2='100', allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(f'B{COMB_FILA0}:{cols[-1]}{COMB_FIN}')
    ws.freeze_panes = f'B{COMB_FILA0}'
    return ws


def escribir_planilla(ruta, filas_nodos, filas_barras, imagen=None, filas_cargas=None, estados=None,
                      materiales=None, combinaciones=None):
    """Escribe el libro completo con los datos dados. Cada fila de NODOS: NODOS_COLS; de BARRAS: BARRAS_COLS."""
    wb = openpyxl.Workbook()
    ws_i = wb.active; ws_i.title = 'INSTRUCCIONES'
    texto = [
        'PLANTILLA DE SOPORTES: como llenarla',
        '',
        'NODOS: una fila por nodo. X, Y, Z en metros. Capacidad: 98 nodos.',
        'APOYO: empotrado (3 traslaciones y 3 giros fijos), rotulado (3 traslaciones fijas, giros libres) o vacio (libre).',
        'BARRAS: una fila por barra. NODO_I y NODO_J son IDs de la hoja NODOS. Capacidad: 106 barras.',
        'ELEMENTO: COLUMNA, VIGA, ARRIOSTRE o DIAGONAL (lista).',
        'PERFIL: nombre Gerdau rev04 (lista ordenada por familia y seccion).',
        'MATERIAL: elija la calidad de la lista (base de datos de la hoja MATERIALES). Es obligatorio: vacio da error.',
        'EJE_FUERTE_X, _Y, _Z: vector libre del eje fuerte (direccion del peralte) en coordenadas globales.',
        '   Vacio = direccion por defecto del motor. Basta una componente (las vacias valen 0). No puede ser paralelo a la barra.',
        'CONEXION_I y CONEXION_J: empotrado o rotulado en cada extremo de la barra. Vacio = empotrado.',
        'Ver la hoja CONEXIONES para el significado de cada tipo.',
        'VISTA_VIVA: planta, alzado y 3D esquematico que se actualizan al cambiar NODOS y BARRAS.',
        '   Cambie el nivel Z (C3), la cota Y (C4), el azimut (C5), la elevacion (C6) y el largo de la flecha (C7).',
        '',
        'No cambiar los nombres de las hojas ni de las columnas. Las filas vacias se ignoran.',
    ]
    for linea in texto:
        ws_i.append([linea])
    ws_i['A1'].font = Font(bold=True, size=13)
    ws_i.column_dimensions['A'].width = 115

    ws_n = wb.create_sheet('NODOS')
    _encabezado(ws_n, NODOS_COLS, [8, 12, 12, 12, 14, 36])
    for f in filas_nodos:
        ws_n.append(f)
    for _ in range(FIN_N - 1 - len(filas_nodos)):
        ws_n.append([None] * len(NODOS_COLS))
    dv_apoyo = DataValidation(type='list', formula1='"' + ','.join(TIPOS_APOYO) + '"', allow_blank=True)
    ws_n.add_data_validation(dv_apoyo); dv_apoyo.add(f'E2:E{FIN_N}')

    ws_b = wb.create_sheet('BARRAS')
    _encabezado(ws_b, BARRAS_COLS, [8, 10, 10, 14, 18, 13, 13, 13, 14, 14, 12, 30])
    for f in filas_barras:
        ws_b.append(f)
    for _ in range(FIN_B - 1 - len(filas_barras)):
        ws_b.append([None] * len(BARRAS_COLS))
    dv_elem = DataValidation(type='list', formula1='"' + ','.join(ELEMENTOS) + '"', allow_blank=True)
    dv_perf = DataValidation(type='list', formula1=f'LISTAS!$A$2:$A${len(PERFILES_ORDEN) + 1}', allow_blank=True)
    dv_vec = DataValidation(type='decimal', operator='between', formula1='-1000000', formula2='1000000',
                            allow_blank=True)
    dv_ci = DataValidation(type='list', formula1='"' + ','.join(TIPOS_CONEXION) + '"', allow_blank=True)
    dv_cj = DataValidation(type='list', formula1='"' + ','.join(TIPOS_CONEXION) + '"', allow_blank=True)
    dv_mat = DataValidation(type='list', formula1=f'MATERIALES!$A$2:$A${MAT_FIN}', allow_blank=True)
    for dv, rango in [(dv_elem, f'D2:D{FIN_B}'), (dv_perf, f'E2:E{FIN_B}'), (dv_vec, f'F2:H{FIN_B}'),
                      (dv_ci, f'I2:I{FIN_B}'), (dv_cj, f'J2:J{FIN_B}'), (dv_mat, f'K2:K{FIN_B}')]:
        ws_b.add_data_validation(dv); dv.add(rango)

    ws_c = wb.create_sheet('CONEXIONES')
    _encabezado(ws_c, ['TIPO', 'EN NODOS (columna APOYO)', 'EN EXTREMOS DE BARRA (CONEXION_I / CONEXION_J)',
                       'USO TIPICO'], [14, 44, 50, 46])
    ws_c.append(['empotrado', 'Restringe 3 traslaciones y 3 giros (base de columna empotrada).',
                 'El extremo transmite momento: la barra sigue el giro del nodo (union rigida).',
                 'Base de columna, nudo rigido de marco.'])
    ws_c.append(['rotulado', 'Restringe 3 traslaciones; los giros quedan libres.',
                 'El momento en el extremo es cero (articulado).',
                 'Apoyo articulado, viga simple, extremo de arriostre.'])
    ws_c.append(['(vacio)', 'En NODOS: nodo libre, sin apoyo.', 'En BARRAS: se toma empotrado.', '-'])
    for fila in ws_c.iter_rows(min_row=2):
        for celda in fila:
            celda.alignment = Alignment(wrap_text=True, vertical='top')

    ws_l = wb.create_sheet('LISTAS')
    _encabezado(ws_l, ['PERFIL', 'ESTADO', 'MASA (kg/m)', 'ELEMENTO'], [18, 20, 14, 14])
    for k, (_, nombre) in enumerate(PERFILES_ORDEN):
        p = G.PERFILES[nombre]
        ws_l.cell(row=k + 2, column=1, value=nombre)
        ws_l.cell(row=k + 2, column=2, value=p['estado'])
        ws_l.cell(row=k + 2, column=3, value=p['masa_kg_m'])
    for k, e in enumerate(ELEMENTOS):
        ws_l.cell(row=k + 2, column=4, value=e)

    ws_calc = _hoja_calculo(wb)
    ws_3d, col_3d = _hoja_3d(wb)
    _hoja_estados(wb, estados or [])
    _hoja_materiales(wb, materiales if materiales is not None else MATERIALES_DEFAULT)
    _hoja_combinaciones(wb, combinaciones if combinaciones is not None else COMBINACIONES_DEFAULT)
    ws_cc, col_cc = _hoja_calc_cargas(wb, col_3d)
    primer_estado = estados[0][0] if estados else ''
    ws_car = _hoja_cargas(wb, filas_cargas or [], primer_estado)
    _grafico_cargas(ws_car, ws_cc, col_cc, ws_3d, col_3d)

    ws_v = wb.create_sheet('VISTA_VIVA')
    ws_v['A1'] = 'Vista en vivo de la planilla: se actualiza al cambiar NODOS o BARRAS.'
    ws_v['A1'].font = Font(bold=True, size=12)
    ws_v['A3'] = 'Nivel Z para planta (m)'
    ws_v['C3'] = 3.5
    ws_v['A4'] = 'Cota Y para alzado (m)'
    ws_v['C4'] = 0
    for celda in ('C3', 'C4'):
        ws_v[celda].fill = ENTRADA
    ws_v['A5'] = 'Azimut (grados)'
    ws_v['C5'] = 30
    ws_v['A6'] = 'Elevacion (grados)'
    ws_v['C6'] = 20
    ws_v['A7'] = 'Largo flecha eje fuerte (m)'
    ws_v['C7'] = 0.8
    for celda in ('C5', 'C6', 'C7'):
        ws_v[celda].fill = ENTRADA
    ws_v['A8'] = 'Cambie los angulos para girar la vista 3D esquematica (abajo).'
    ws_v.column_dimensions['A'].width = 26
    _graficos(ws_v, ws_calc)
    _grafico_3d(ws_v, ws_3d, col_3d)
    _grafico_3d_rotulas(ws_v, ws_3d, col_3d)

    wb.calculation = CalcProperties(fullCalcOnLoad=True)
    wb.save(ruta)
    return ruta


def _sin_suavizado(ruta_xlsx):
    """LibreOffice guarda las series con suavizado (smooth=1): las flechas del eje fuerte se verian curvas.
    Se reescriben los graficos con smooth=0 (lineas rectas)."""
    import zipfile
    tmp = ruta_xlsx + '.tmp'
    with zipfile.ZipFile(ruta_xlsx) as origen, zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as destino:
        for item in origen.infolist():
            datos = origen.read(item.filename)
            if item.filename.startswith('xl/charts/chart') and item.filename.endswith('.xml'):
                datos = datos.replace(b'<c:smooth val="1"/>', b'<c:smooth val="0"/>')
            destino.writestr(item, datos)
    os.replace(tmp, ruta_xlsx)


def guardar_con_valores(ruta):
    """Recalcula el libro con LibreOffice (si esta instalado) para que las formulas y graficos tengan valores
    guardados: asi se ven tambien en visores que no calculan. Sin LibreOffice, Excel calcula al abrir."""
    import shutil, subprocess, tempfile
    soffice = shutil.which('soffice') or shutil.which('libreoffice')
    if not soffice:
        print('LibreOffice no disponible: Excel calcula la hoja al abrirla.')
        return
    tmp = tempfile.mkdtemp()
    subprocess.run([soffice, '--headless', '--calc', '--convert-to', 'xlsx', '--outdir', tmp, ruta],
                   capture_output=True, timeout=300)
    salida = os.path.join(tmp, os.path.basename(ruta))
    if os.path.exists(salida):
        _sin_suavizado(salida)
        shutil.move(salida, ruta)
        print('Valores calculados guardados en', ruta)
    else:
        print('No se pudo recalcular con LibreOffice; Excel calcula la hoja al abrirla.')


def crear_plantilla(ruta=None, sobrescribir=False):
    ruta = ruta or os.path.join(os.getcwd(), 'plantilla_soportes.xlsx')
    if os.path.exists(ruta) and not sobrescribir:
        print('Ya existe', ruta, '- no se sobrescribe la planilla llenada.')
        return ruta
    nodos, barras = ejemplo_filas()
    escribir_planilla(ruta, nodos, barras, filas_cargas=ejemplo_cargas(), estados=ejemplo_estados(),
                      materiales=MATERIALES_DEFAULT, combinaciones=COMBINACIONES_DEFAULT)
    guardar_con_valores(ruta)
    print('Plantilla creada:', ruta)
    return ruta


def cargar_plantilla(ruta=None):
    ruta = ruta or os.path.join(os.getcwd(), 'plantilla_soportes.xlsx')
    if os.path.exists(ruta):
        print('Ya existe, se usa tal cual:', ruta)
    else:
        with open(ruta, 'wb') as f:
            f.write(base64.b64decode(PLANTILLA_B64))
        print('Plantilla escrita en:', ruta)
    return ruta


# Plantilla incrustada en el notebook (mismo archivo que plantilla_soportes.xlsx del repositorio)
PLANTILLA_B64 = '...'
RUTA_PLANTILLA = cargar_plantilla() if PLANTILLA_B64 else crear_plantilla()
_hojas = pd.read_excel(RUTA_PLANTILLA, sheet_name=None, dtype=object)
print('Hojas cargadas:', list(_hojas))
display(_hojas['NODOS'].head(8))
display(_hojas['BARRAS'].head(8))


## Bloque 7. Leer, validar y armar el modelo

Lee la planilla subida a Kaggle o, si no hay ninguna, la de ejemplo. Si hay ERROR, no se arma el modelo.

In [ ]:
# Leer la planilla llenada, validar y armar el modelo
import glob
import os
import numpy as np
import pandas as pd
from dibujo_directo import DibujoModelo, ELEMENTOS, TIPOS_APOYO, TIPOS_CONEXION

TOL = 1e-3
EJE_COLS = ['EJE_FUERTE_X', 'EJE_FUERTE_Y', 'EJE_FUERTE_Z']


def _vacio(v):
    return v is None or (isinstance(v, float) and np.isnan(v)) or str(v).strip() == ''


def _col(fila, nombre):
    """Valor de una columna; None si la planilla no tiene esa columna (planillas anteriores)."""
    return fila[nombre] if nombre in fila.index else None


def _tipo(v):
    return '' if _vacio(v) else str(v).strip().lower()


def buscar_planilla():
    """Busca la planilla llenada en los datos subidos a Kaggle (Add data); si no hay, usa la de ejemplo."""
    subidas = [f for f in glob.glob('/kaggle/input/**/*.xlsx', recursive=True)
               if 'soportes' in os.path.basename(f).lower()]
    return subidas[0] if subidas else RUTA_PLANTILLA


def leer_planilla(ruta):
    nodos = pd.read_excel(ruta, sheet_name='NODOS', dtype=object)
    barras = pd.read_excel(ruta, sheet_name='BARRAS', dtype=object)
    nodos = nodos[[not all(_vacio(v) for v in fila) for fila in nodos[['ID', 'X (m)', 'Y (m)', 'Z (m)']].values]]
    barras = barras[[not all(_vacio(v) for v in fila) for fila in barras[['ID', 'NODO_I', 'NODO_J']].values]]
    return nodos, barras


def leer_materiales(ruta):
    """Hoja MATERIALES de la planilla. None si la planilla no la tiene (se usa la lista por defecto)."""
    try:
        m = pd.read_excel(ruta, sheet_name='MATERIALES', dtype=object)
    except Exception:
        return None
    m = m.iloc[:, :5]
    return [[None if _vacio(v) else v for v in fila] for fila in m.values.tolist() if not _vacio(fila[0])]


def _letra(n):
    from openpyxl.utils import get_column_letter
    return get_column_letter(n)


def leer_combinaciones(ruta, estados):
    """Hoja COMBINACIONES (informacion para el notebook). Devuelve (filas_crudas, combinaciones, mensajes):
    filas_crudas = factores tal como estan en la hoja (40 x 30); combinaciones = {nombre: {estado: factor}}."""
    msgs = []
    try:
        import openpyxl
        ws = openpyxl.load_workbook(ruta, data_only=False)['COMBINACIONES']
    except Exception:
        return [], {}, msgs
    nombres_estado = [str(v).strip() if not _vacio(v) else None for v in estados['NOMBRE'].values]
    filas_crudas, combinaciones = [], {}
    for k in range(40):
        fila_excel = 3 + k
        crudo = [ws.cell(row=fila_excel, column=2 + j).value for j in range(30)]
        filas_crudas.append(crudo)
        factores = {}
        for j, v in enumerate(crudo):
            if _vacio(v):
                continue
            try:
                f = float(v)
            except (TypeError, ValueError):
                msgs.append(('COMBINACIONES', fila_excel, 'ERROR', f'factor no numerico en la columna {_letra(2 + j)}'))
                continue
            if f == 0:
                continue
            if j >= len(nombres_estado) or nombres_estado[j] is None:
                msgs.append(('COMBINACIONES', fila_excel, 'ERROR', f'factor {f} en la columna {_letra(2 + j)}, que no tiene estado en ESTADOS'))
                continue
            factores[nombres_estado[j]] = f
        if factores:
            combinaciones[f'ASD{k + 1}'] = factores
    return filas_crudas, combinaciones, msgs


def validar(nodos, barras):
    """Lista de (hoja, fila_excel, tipo, mensaje). ERROR = no se arma el modelo; AVISO = informativo."""
    msgs = []
    def avisar(hoja, fila, tipo, texto):
        msgs.append((hoja, fila, tipo, texto))

    ids, coords = {}, []
    for k, r in nodos.iterrows():
        fila = k + 2
        try:
            id_ = int(float(r['ID']))
        except (TypeError, ValueError):
            avisar('NODOS', fila, 'ERROR', 'ID no es un entero'); continue
        if id_ in ids:
            avisar('NODOS', fila, 'ERROR', f'ID {id_} repetido'); continue
        try:
            p = (float(r['X (m)']), float(r['Y (m)']), float(r['Z (m)']))
        except (TypeError, ValueError):
            avisar('NODOS', fila, 'ERROR', 'coordenada X, Y o Z no numerica'); continue
        for q_id, q in coords:
            if all(abs(p[c] - q[c]) < TOL for c in range(3)):
                avisar('NODOS', fila, 'ERROR', f'coincide con el nodo {q_id}')
        ids[id_] = p
        coords.append((id_, p))
        apoyo = _tipo(_col(r, 'APOYO'))
        if apoyo and apoyo not in TIPOS_APOYO:
            avisar('NODOS', fila, 'ERROR', f"APOYO '{_col(r, 'APOYO')}' no reconocido (use empotrado, rotulado o vacio)")

    vistos = set()
    for k, r in barras.iterrows():
        fila = k + 2
        try:
            n1, n2 = int(float(r['NODO_I'])), int(float(r['NODO_J']))
        except (TypeError, ValueError):
            avisar('BARRAS', fila, 'ERROR', 'NODO_I o NODO_J no es un entero'); continue
        if n1 not in ids or n2 not in ids:
            avisar('BARRAS', fila, 'ERROR', f'nodo inexistente en NODOS ({n1} o {n2})'); continue
        if n1 == n2:
            avisar('BARRAS', fila, 'ERROR', 'barra de un nodo consigo mismo'); continue
        par = frozenset((n1, n2))
        if par in vistos:
            avisar('BARRAS', fila, 'AVISO', f'barra {n1}-{n2} repetida: se ignora')
        vistos.add(par)
        elem = str(r['ELEMENTO']).strip().upper()
        if elem not in ELEMENTOS:
            avisar('BARRAS', fila, 'ERROR', f"ELEMENTO '{r['ELEMENTO']}' no reconocido")
        perfil = str(r['PERFIL']).strip()
        if perfil not in G.PERFILES:
            avisar('BARRAS', fila, 'ERROR', f"PERFIL '{perfil}' no existe en la base Gerdau")
        else:
            p = G.PERFILES[perfil]
            if p['d_ala_diametro_cm'] is None or p['bf_b_cm'] is None:
                avisar('BARRAS', fila, 'AVISO', f"perfil {perfil} sin dato geometrico (estado: {p['estado']})")
        # eje fuerte: vector libre. Una componente basta: las vacias valen 0. Todas vacias = por defecto.
        vec = [_col(r, c) for c in EJE_COLS]
        if any(not _vacio(v) for v in vec):
            try:
                v = np.array([0.0 if _vacio(x) else float(x) for x in vec])
            except (TypeError, ValueError):
                avisar('BARRAS', fila, 'ERROR', 'componente del eje fuerte no numerica'); v = None
            if v is not None:
                nv = np.linalg.norm(v)
                if nv < 1e-9:
                    avisar('BARRAS', fila, 'ERROR', 'eje fuerte nulo: indique una direccion o deje las tres vacias')
                elif n1 in ids and n2 in ids:
                    d = np.array(ids[n2]) - np.array(ids[n1])
                    if abs(np.dot(v / nv, d / np.linalg.norm(d))) > 0.999:
                        avisar('BARRAS', fila, 'ERROR', 'eje fuerte paralelo a la barra: elija otra direccion')
        material = '' if _vacio(_col(r, 'MATERIAL')) else str(_col(r, 'MATERIAL')).strip()
        if not material:
            avisar('BARRAS', fila, 'ERROR', 'falta MATERIAL: elija una calidad de la lista (hoja MATERIALES)')
        elif material not in MATERIALES_VALIDOS:
            avisar('BARRAS', fila, 'ERROR', f"MATERIAL '{material}' no existe en la hoja MATERIALES")
        for col in ('CONEXION_I', 'CONEXION_J'):
            c = _tipo(_col(r, col))
            if c and c not in TIPOS_CONEXION:
                avisar('BARRAS', fila, 'ERROR', f"{col} '{_col(r, col)}' no reconocida (use empotrado, rotulado o vacio)")
    return msgs


def construir_modelo(nodos, barras):
    """Arma el DibujoModelo con los IDs de la planilla. Devuelve (dib, id_de_indice, mensajes)."""
    dib = DibujoModelo(ruta=RUTA_PLANILLA_LEIDA)
    mapa, msgs = {}, []
    id_barra = {}                       # indice de barra en el modelo -> ID de la hoja BARRAS
    for _, r in nodos.iterrows():
        mapa[int(float(r['ID']))] = dib.nodo(float(r['X (m)']), float(r['Y (m)']), float(r['Z (m)']))
    for _, r in barras.iterrows():
        n1, n2 = mapa[int(float(r['NODO_I']))], mapa[int(float(r['NODO_J']))]
        ci = _tipo(_col(r, 'CONEXION_I')) or 'empotrado'
        cj = _tipo(_col(r, 'CONEXION_J')) or 'empotrado'
        material = '' if _vacio(_col(r, 'MATERIAL')) else str(_col(r, 'MATERIAL')).strip()
        try:
            idx = dib.barra(n1, n2, str(r['ELEMENTO']).strip().upper(), str(r['PERFIL']).strip(),
                            conexion_i=ci, conexion_j=cj, material=material)
        except ValueError as e:
            msgs.append(('BARRAS', int(float(r['ID'])), 'ERROR', str(e))); continue
        vec = [_col(r, c) for c in EJE_COLS]
        if idx is not None:
            id_barra[idx] = int(float(r['ID']))
        if idx is not None and any(not _vacio(v) for v in vec):
            try:
                dib.eje_fuerte(idx, tuple(0.0 if _vacio(v) else float(v) for v in vec))
            except ValueError as e:
                msgs.append(('BARRAS', int(float(r['ID'])), 'ERROR', str(e)))
    for _, r in nodos.iterrows():
        tipo = _tipo(_col(r, 'APOYO'))
        if tipo:
            dib.apoyo(mapa[int(float(r['ID']))], tipo)
    id_de_indice = {idx: id_ for id_, idx in mapa.items()}
    dib.id_barra_de_indice = id_barra
    return dib, id_de_indice, msgs


CARGA_FILA0 = 9
CARGA_COLS_LEER = ['ID', 'ESTADO', 'FX', 'FY', 'FZ', 'MX', 'MY', 'MZ', 'NOTA']


def leer_cargas(ruta):
    """Hoja CARGAS: una fila por nodo y estado. FX, FY, FZ en kgf; MX, MY, MZ en kgf·m (ejes globales X, Y, Z)."""
    try:
        c = pd.read_excel(ruta, sheet_name='CARGAS', header=CARGA_FILA0 - 2, dtype=object)
    except Exception:
        return pd.DataFrame(columns=CARGA_COLS_LEER)
    c = c.iloc[:, :9]
    c.columns = CARGA_COLS_LEER
    return c[[not all(_vacio(v) for v in fila) for fila in c[['ID', 'ESTADO']].values]]


def leer_estados(ruta):
    """Hoja ESTADOS: nombre del estado de carga (lo define el usuario), tipo y descripcion."""
    try:
        e = pd.read_excel(ruta, sheet_name='ESTADOS', dtype=object)
    except Exception:
        return pd.DataFrame(columns=['NOMBRE', 'TIPO', 'DESCRIPCION'])
    e = e.iloc[:, :3]
    e.columns = ['NOMBRE', 'TIPO', 'DESCRIPCION']
    return e[[not _vacio(v) for v in e['NOMBRE'].values]]


def validar_cargas(cargas, estados, ids_nodo):
    """Valida CARGAS contra NODOS y ESTADOS. Devuelve (mensajes, sumas) con sumas[(id_nodo, estado)] = [6 valores]."""
    msgs, sumas = [], {}
    nombres = {str(v).strip() for v in estados['NOMBRE'].values if not _vacio(v)}
    for k, r in cargas.iterrows():
        fila = k + CARGA_FILA0
        try:
            id_ = int(float(r['ID']))
        except (TypeError, ValueError):
            msgs.append(('CARGAS', fila, 'ERROR', 'ID NODO no es un entero'))
            continue
        if id_ not in ids_nodo:
            msgs.append(('CARGAS', fila, 'ERROR', f'el nodo {id_} no existe en NODOS'))
            continue
        est = str(r['ESTADO']).strip()
        if est not in nombres:
            msgs.append(('CARGAS', fila, 'ERROR', f"el estado '{est}' no existe en la hoja ESTADOS"))
            continue
        try:
            valores = [0.0 if _vacio(r[c]) else float(r[c]) for c in ['FX', 'FY', 'FZ', 'MX', 'MY', 'MZ']]
        except (TypeError, ValueError):
            msgs.append(('CARGAS', fila, 'ERROR', 'fuerza o momento no numerico'))
            continue
        clave = (id_, est)
        if clave in sumas:
            msgs.append(('CARGAS', fila, 'AVISO', f'el nodo {id_} ya tiene el estado {est}: se suman'))
            sumas[clave] = [a + b for a, b in zip(sumas[clave], valores)]
        else:
            sumas[clave] = valores
    return msgs, sumas


def adjuntar_cargas(dib, sumas, id_de_indice):
    """dib.cargas = {estado: {indice_nodo: [Fx, Fy, Fz (kgf), Mx, My, Mz (kgf·cm)]}}.
    Los momentos se pasan de kgf·m a kgf·cm porque el motor de OpenSees trabaja en kgf y kgf·cm."""
    idx_de_id = {v: k for k, v in id_de_indice.items()}
    dib.cargas = {}
    for (id_, est), v in sumas.items():
        fx, fy, fz, mx, my, mz = v
        dib.cargas.setdefault(est, {})[idx_de_id[id_]] = [fx, fy, fz, mx * 100.0, my * 100.0, mz * 100.0]


def filas_cargas_para_excel(cargas):
    """Filas de la hoja CARGAS para regenerar el Excel con sus datos."""
    filas = []
    for _, r in cargas.iterrows():
        try:
            id_ = int(float(r['ID']))
        except (TypeError, ValueError):
            id_ = None
        vals = [0.0 if _vacio(r[c]) else float(r[c]) for c in ['FX', 'FY', 'FZ', 'MX', 'MY', 'MZ']]
        filas.append([id_, None if _vacio(r['ESTADO']) else str(r['ESTADO']).strip(), *vals,
                      None if _vacio(r['NOTA']) else r['NOTA']])
    return filas


RUTA_PLANILLA_LEIDA = buscar_planilla()   # o escriba la ruta exacta, por ejemplo '/kaggle/input/mi-dataset/mis_soportes.xlsx'
print('Planilla que se lee:', RUTA_PLANILLA_LEIDA)
nodos, barras = leer_planilla(RUTA_PLANILLA_LEIDA)
materiales_filas = leer_materiales(RUTA_PLANILLA_LEIDA) or MATERIALES_DEFAULT
MATERIALES_VALIDOS = {str(f[0]).strip() for f in materiales_filas}
mensajes = validar(nodos, barras)
cargas = leer_cargas(RUTA_PLANILLA_LEIDA)
estados_df = leer_estados(RUTA_PLANILLA_LEIDA)
ids_nodo = set()
for _v in nodos['ID'].values:
    try:
        ids_nodo.add(int(float(_v)))
    except (TypeError, ValueError):
        pass
msgs_cargas, sumas_cargas = validar_cargas(cargas, estados_df, ids_nodo)
mensajes += msgs_cargas
comb_crudas, combinaciones_modelo, msgs_comb = leer_combinaciones(RUTA_PLANILLA_LEIDA, estados_df)
mensajes += msgs_comb
dib, id_de_indice, extra = (None, None, [])
if not any(t == 'ERROR' for _, _, t, _ in mensajes):
    dib, id_de_indice, extra = construir_modelo(nodos, barras)
    mensajes += extra
    if any(t == 'ERROR' for _, _, t, _ in extra):
        dib = None                  # un error al construir deja el modelo incompleto: no se entrega
if dib is not None:
    adjuntar_cargas(dib, sumas_cargas, id_de_indice)
    dib.combinaciones = combinaciones_modelo          # informacion para el notebook: {nombre: {estado: factor}}
    print(f'Combinaciones de carga: {len(combinaciones_modelo)}')
    print('Estados de carga:', [str(v) for v in estados_df['NOMBRE'].values])
    print(f'Asignaciones nodo-estado: {len(sumas_cargas)}')
resumen = pd.DataFrame(mensajes, columns=['hoja', 'fila_excel', 'tipo', 'mensaje'])
if resumen.empty:
    print('Planilla sin observaciones.')
else:
    display(resumen)
if dib is not None:
    print(f'Modelo armado: {len(dib.nodos)} nodos, {len(dib.barras)} barras, {len(dib.apoyos)} apoyos')
    modelo = dib.a_modelo()        # formato leer_dxf(): listo para el motor de analisis
else:
    print('Modelo NO armado: corrija los ERROR de la tabla y vuelva a ejecutar este bloque.')


## Bloque 8. Dibujo de revisión y planilla regenerada

Figura planta, alzado y 3D, y `planilla_con_vista.xlsx`. Ejecutar después de cada cambio.

In [ ]:
# Dibujo de revision (planta, alzado, 3D con secciones y ejes fuertes) y planilla regenerada con la vista
import math
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from orientacion_gerdau import contorno, ejes_locales, dimensiones
from vista3d_plotly import figura_3d

COLOR = {'COLUMNA': '#1f77b4', 'VIGA': '#2ca02c', 'ARRIOSTRE': '#d62728', 'DIAGONAL': '#ff7f0e'}
ESCALA_SECCION = 2.5    # las secciones se dibujan ampliadas en la figura (no cambia el calculo)


def plano_con_mas_barras(dib, planta):
    """Nivel Z (planta) o cota Y (alzado) con mas barras en el plano."""
    from collections import Counter
    cuenta = Counter()
    for b in dib.barras:
        p1, p2 = dib.nodos[b['n1']], dib.nodos[b['n2']]
        if planta and abs(p1[2] - p2[2]) < 1e-9:
            cuenta[round(p1[2], 6)] += 1
        if not planta and abs(p1[1] - p2[1]) < 1e-9:
            cuenta[round(p1[1], 6)] += 1
    return cuenta.most_common(1)[0][0] if cuenta else min(p[2] if planta else p[1] for p in dib.nodos)


def dibujar_vista(ax, dib, id_de_indice, vista='planta', plano=None):
    planta = vista == 'planta'
    if plano is None:
        plano = plano_con_mas_barras(dib, planta)

    def en_plano(p):
        return abs(p[2] - plano) < 1e-9 if planta else abs(p[1] - plano) < 1e-9

    def coord(p):
        return (p[0], p[1]) if planta else (p[0], p[2])

    for b in dib.barras:
        p1, p2 = dib.nodos[b['n1']], dib.nodos[b['n2']]
        if not (en_plano(p1) and en_plano(p2)):
            continue
        (x1, y1), (x2, y2) = coord(p1), coord(p2)
        ax.plot([x1, x2], [y1, y2], color=COLOR[b['elemento']], lw=3.5, zorder=2, solid_capstyle='round')
        for p, c in [(p1, b['conexion_i']), (p2, b['conexion_j'])]:
            if c == 'rotulado':
                ax.plot(*coord(p), 'o', mfc='white', mec='k', ms=7, zorder=4)
    for i, q in enumerate(dib.nodos):
        if en_plano(q):
            ax.plot(*coord(q), 'ko', ms=4, zorder=5)
            ax.text(coord(q)[0], coord(q)[1], f' {id_de_indice[i]}', fontsize=8, zorder=6)
            if i in dib.apoyos:
                marca = 's' if dib.apoyos[i] == 'empotrado' else '^'
                ax.plot(*coord(q), marca, color='#e67e22', ms=10, zorder=5)
    ax.set_aspect('equal'); ax.margins(0.12); ax.grid(True, color='#dddddd', lw=0.6)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)' if planta else 'Z (m)')
    ax.set_title(f"{'Planta' if planta else 'Alzado'} en {'Z' if planta else 'Y'} = {plano:g} m", fontsize=10)


def dibujar_3d(ax, dib, id_de_indice):
    for i, b in enumerate(dib.barras):
        p1, p2 = dib.nodos[b['n1']], dib.nodos[b['n2']]
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color=COLOR[b['elemento']], lw=2.5)
        for p, c in [(p1, b['conexion_i']), (p2, b['conexion_j'])]:
            if c == 'rotulado':
                ax.scatter(*p, s=40, facecolors='white', edgecolors='k', zorder=5)
        # seccion (envolvente del perfil) en el punto medio, y eje fuerte = eje local y (peralte)
        caja = contorno(dib, i, ESCALA_SECCION)
        if caja:
            ax.add_collection3d(Poly3DCollection([caja], facecolor=COLOR[b['elemento']], alpha=0.35,
                                                 edgecolor='k', linewidths=0.4))
            pm = [(p1[k] + p2[k]) / 2 for k in range(3)]
            d = dimensiones(b['perfil'])[0] * ESCALA_SECCION
            _, ey, _ = ejes_locales(dib, i)
            ax.quiver(*pm, *(ey * 0.6 * d), color='k', arrow_length_ratio=0.3, linewidth=1.2)
    for i, q in enumerate(dib.nodos):
        ax.text(q[0], q[1], q[2], f' {id_de_indice[i]}', fontsize=7)
        if i in dib.apoyos:
            ax.scatter(*q, s=60, marker='s' if dib.apoyos[i] == 'empotrado' else '^', color='#e67e22', zorder=6)
        else:
            ax.scatter(*q, s=8, color='k', zorder=6)
    rangos = [max(max(p[k] for p in dib.nodos) - min(p[k] for p in dib.nodos), 1.0) for k in range(3)]
    ax.set_box_aspect(rangos)
    from matplotlib.ticker import MaxNLocator
    for eje in (ax.xaxis, ax.yaxis, ax.zaxis):
        eje.set_major_locator(MaxNLocator(4))
    ax.view_init(elev=22, azim=-60)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_zlabel('Z (m)')
    ax.set_title(f'3D: secciones x{ESCALA_SECCION:g}, flecha = eje fuerte, circulo blanco = extremo rotulado, '
                 'cuadrado = empotrado, triangulo = apoyo rotulado', fontsize=8)


if dib is None:
    print('Sin modelo para dibujar: corrija la planilla (bloque 7).')
else:
    fig = plt.figure(figsize=(16, 5.8))
    dibujar_vista(fig.add_subplot(1, 3, 1), dib, id_de_indice, 'planta')
    dibujar_vista(fig.add_subplot(1, 3, 2), dib, id_de_indice, 'alzado')
    dibujar_3d(fig.add_subplot(1, 3, 3, projection='3d'), dib, id_de_indice)
    fig.tight_layout()
    plt.show()

    # 3D interactiva (gira con el mouse; al pasar el cursor muestra ID, perfil y eje fuerte de cada barra)
    figura_3d(dib, id_de_indice, escala_seccion=2.5).show(renderer='iframe')

    # Planilla regenerada con sus datos (las hojas de calculo y graficos se rehacen)
    escribir_planilla('planilla_con_vista.xlsx', _filas(nodos, NODOS_COLS), _filas(barras, BARRAS_COLS),
                      filas_cargas=filas_cargas_para_excel(cargas), materiales=materiales_filas,
                      combinaciones=comb_crudas,
                      estados=[[None if _vacio(v) else v for v in f] for f in estados_df.values.tolist()])
    guardar_con_valores('planilla_con_vista.xlsx')
    print('Planilla regenerada: planilla_con_vista.xlsx (hoja VISTA_VIVA)')
